# BetaTour v11 — POP6 aday kenar mekanizma pilotu

Bu deney **eski v8/v9 geliştirme örneklerinin dört 10k örneğini** kullanır; bağımsız doğrulama değildir. POPMUSIC-6/LKH'nin aynı ilk denemesinden sonra BetaTour, tur sıklığı veya rastgele kontrol, aynı kaynak düğümlere ve aynı sayıda yeni yönsüz aday kenara müdahale eder. Sham aynı model çıkarımını yapar ve aramayı değiştirmez. Native POP6 ve POP7 pratik kıyaslardır. Bütün kol bütçelerine kurulum, model çıkarımı, bekleme ve LKH araması dahildir.

**Amaç:** Modelin eklediği kenarların LKH aday taramasına girip girmediğini ve daha iyi incumbent turlarda görünme sıklığını ölçmek; modelin güçlü POP6/POP7 ve eşit kapasiteli kontrol seçicilerinden ayrışıp ayrışmadığını görmek. İlk tur ve aday durumları eşleşmeyen kolların eşli politika karşılaştırması yapılmaz. Tur maliyetinin yanında hedef kaliteye ulaşma süresi de kaydedilir. Bu pilot olumlu olsa bile makale, donmuş politika bağımsız örneklerde doğrulanana dek değiştirilmeyecek.

**Çalıştırma:** Colab'da tercihen A100 veya L4 GPU ve High RAM seçin. Altı eşzamanlı LKH için 12 vCPU tercih edilir. Hücreleri sırayla çalıştırın. Tamamlanan kollar `MyDrive/BetaTour/v11_pop6_candidate_mechanism` içinde doğrulanarak atlanır; kesilen kol yeniden çalışır. Sonuç ZIP'i sonunda tarayıcıya otomatik iner, tam denetim ZIP'i Drive'da kalır. Dört örnek × altı kol × 120 saniye = **48 dakika** salt çözücü sınırı; kurulum ve arşivleme eklenir.


In [ ]:
EXPERIMENT_NAME = "v11_pop6_candidate_mechanism"
SEEDS = list(range(301, 307))
ARMS = ["native_pop6", "native_pop7", "sham", "random_add", "frequency_add", "beta_add"]
INSTANCE_SPECS = [
    ["uni10k", 10000, 2, "uniform", 4107, 120.0],
    ["clu10k", 10000, 2, "clustered", 4208, 120.0],
]
EXPECTED_XY_SHA256 = {
    "augment_uni10k_000": "0d5d680b465faa68d33467c4f70bb5ad1e52b25a2d444066ab922130458efcd6",
    "augment_uni10k_001": "00747175507524f481bd6e34ea55f0a5474d3dcd642a67543d871d9a64b53796",
    "augment_clu10k_000": "072fc96ec668bb9c95b3a52562f23cedb0b2e89494946d159243b638b7b3dc8e",
    "augment_clu10k_001": "965fe4315060118d313a357404f750a23ea89cc11c174cb1138a9f32b312ca22",
    "augment_uni20k_000": "d0dbad4fdd4151029d3bf59cca81c19248129c87030003d7c455e37e32b7d674",
    "augment_uni20k_001": "c35e5c135b527d91c5c9b6258320f35526c2e735a50d581c48e96150e9cb3173",
    "augment_clu20k_000": "abdd6474a4cd0e25ee394c227ef8f5c4beb5c09c8a24abddcad273d0dc39181e",
    "augment_clu20k_001": "ee3edf42dbbb94cd344d1e3bef06d5674e47330e937a8acda4fd069f4faf79ff",
}
MAX_NEW_ARMS = None  # Örn. 3: bu oturumda en fazla üç yeni kol; sonra tekrar çalıştırın.
SPARSE_FRACTION = 0.25
GRAPH_K = 16
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
LKH_TAR_SHA256 = "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33"
LKH_FINDTOUR_SHA256 = "567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee"
EXPECTED_XY_SHA256 = {name: digest for name, digest in EXPECTED_XY_SHA256.items()
                      if "10k" in name}
TARGET_COSTS = {"augment_clu10k_000": 33760199,
                "augment_clu10k_001": 16201186,
                "augment_uni10k_000": 72087781,
                "augment_uni10k_001": 71648570}
assert len(SEEDS) == 6 and len(EXPECTED_XY_SHA256) == 4 and len(TARGET_COSTS) == 4


In [ ]:
import base64, hashlib, importlib.util, json, math, os, platform, shutil, subprocess, sys, tarfile, time, urllib.request, uuid, zipfile
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
for package in ("scipy", "pandas"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
import numpy as np
import scipy
from scipy.stats import binomtest
import pandas as pd
import torch
from IPython.display import display

assert torch.cuda.is_available(), "Colab GPU runtime seçin (tercihen A100 veya L4)."
torch.set_num_threads(1)
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
if not any(name in GPU_NAME for name in ("A100", "L4")):
    print("UYARI: GPU", GPU_NAME, "— sonuçları farklı profillerle eşlemeyin.")
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
if CPU_COUNT < 6:
    print("UYARI: Altı LKH süreci için en az 6 vCPU önerilir; görünen:", CPU_COUNT)
def cpu_model_name():
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.lower().startswith("model name"):
            return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"

WORK_ROOT = Path("/content/betatour_v11_candidate_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
assert hashlib.sha256((REPO / "checkpoints/betatour_v2.pt").read_bytes()).hexdigest() == CHECKPOINT_SHA256
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M
MODEL, _ = M.load_model(str(REPO / "checkpoints/betatour_v2.pt"), DEVICE)
assert MODEL.cfg.get("use_ensemble") is True

from google.colab import drive
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
for sub in ("instances", "code", "summary", "errors", "archives"):
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
HARDWARE = {"gpu": GPU_NAME, "vcpus": CPU_COUNT, "cpu_model": cpu_model_name(),
            "python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "scipy": scipy.__version__,
            "pandas": pd.__version__, "cuda": torch.version.cuda}
print("Hazır:", HARDWARE, "Drive:", OUTPUT_ROOT)


UYARI: GPU Tesla T4 — sonuçları farklı profillerle eşlemeyin.
UYARI: Altı LKH süreci için en az 6 vCPU önerilir; görünen: 2
Mounted at /content/drive
Hazır: {'gpu': 'Tesla T4', 'vcpus': 2, 'cpu_model': 'Intel(R) Xeon(R) CPU @ 2.00GHz', 'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'scipy': '1.16.3', 'pandas': '2.2.3', 'cuda': '13.0'} Drive: /content/drive/MyDrive/BetaTour/v11_pop6_candidate_mechanism


In [ ]:
BUNDLED_FILES = {'livehook/coordinator.py': 'IiIiV2FsbC1jbG9jayBjb29yZGluYXRvciBmb3Igc2l4IHN0YXRlLXByZXNlcnZpbmcgQmV0YVRvdXIvTEtIIHByb2Nlc3Nlcy4KCk9ubHkgb25lIHByb2JsZW0gaW5zdGFuY2Ugc2hvdWxkIHVzZSBhIGdpdmVuIHdvcmsgZGlyZWN0b3J5LiBUaGUgYWN0aW9uCmNhbGxiYWNrIHJlY2VpdmVzIHRoZSBzaXggZmlyc3QgdG91cnMgYW5kIG5hdGl2ZS9saXZlIGNhbmRpZGF0ZSBzbmFwc2hvdHMuCkl0IHJldHVybnMgb25lIHBlci1ub2RlIGFjdGlvbiBsaXN0IHBlciBzZWVkOyBlYWNoIGFjdGlvbiBpcwpgYChuZWlnaGJvcl9pZF8xX2Jhc2VkX29yXzAsIHNsb3RfMV9iYXNlZF9vcl8wLCBhbHBoYSlgYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUsIFNlcXVlbmNlCgoKQGRhdGFjbGFzcwpjbGFzcyBTbmFwc2hvdDoKICAgIHNlZWQ6IGludAogICAgZmlyc3RfdG91cjogbGlzdFtpbnRdCiAgICBuYXRpdmVfY2FuZGlkYXRlczogbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dCiAgICBsaXZlX2NhbmRpZGF0ZXM6IGxpc3RbbGlzdFt0dXBsZVtpbnQsIGludF1dXQoKCmRlZiByZWFkX3RvdXIocGF0aDogUGF0aCkgLT4gbGlzdFtpbnRdOgogICAgd29yZHMgPSBbaW50KHgpIGZvciB4IGluIHBhdGgucmVhZF90ZXh0KCkuc3BsaXQoKV0KICAgIGlmIG5vdCB3b3JkcyBvciBsZW4od29yZHMpICE9IHdvcmRzWzBdICsgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWFsZm9ybWVkIHRvdXI6IHtwYXRofSIpCiAgICBuID0gd29yZHNbMF0KICAgIHRvdXIgPSB3b3Jkc1sxOl0KICAgIGlmIHNvcnRlZCh0b3VyKSAhPSBsaXN0KHJhbmdlKDEsIG4gKyAxKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlRvdXIgaXMgbm90IGEgcGVybXV0YXRpb246IHtwYXRofSIpCiAgICByZXR1cm4gdG91cgoKCmRlZiByZWFkX2NhbmRpZGF0ZXMocGF0aDogUGF0aCkgLT4gbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dOgogICAgbGluZXMgPSBwYXRoLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKQogICAgaWYgbm90IGxpbmVzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFbXB0eSBjYW5kaWRhdGUgc25hcHNob3Q6IHtwYXRofSIpCiAgICBuID0gaW50KGxpbmVzWzBdKQogICAgaWYgbGVuKGxpbmVzKSAhPSBuICsgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ2FuZGlkYXRlIHNuYXBzaG90IGhhcyB7bGVuKGxpbmVzKS0xfSByb3dzLCBuZWVkIHtufSIpCiAgICByb3dzOiBsaXN0W2xpc3RbdHVwbGVbaW50LCBpbnRdXV0gPSBbXQogICAgZm9yIGV4cGVjdGVkX2lkLCBsaW5lIGluIGVudW1lcmF0ZShsaW5lc1sxOl0sIDEpOgogICAgICAgIG51bXMgPSBbaW50KHgpIGZvciB4IGluIGxpbmUuc3BsaXQoKV0KICAgICAgICBpZiBsZW4obnVtcykgPCAyIG9yIG51bXNbMF0gIT0gZXhwZWN0ZWRfaWQgb3IgbGVuKG51bXMpICE9IDIgKyAyICogbnVtc1sxXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1hbGZvcm1lZCBjYW5kaWRhdGUgcm93IHtleHBlY3RlZF9pZH06IHtwYXRofSIpCiAgICAgICAgcm93ID0gWyhudW1zW2ldLCBudW1zW2kgKyAxXSkgZm9yIGkgaW4gcmFuZ2UoMiwgbGVuKG51bXMpLCAyKV0KICAgICAgICBpZiBhbnkodiA8IDEgb3IgdiA+IG4gb3IgdiA9PSBleHBlY3RlZF9pZCBmb3IgdiwgXyBpbiByb3cpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBuZWlnaGJvciBpbiByb3cge2V4cGVjdGVkX2lkfToge3BhdGh9IikKICAgICAgICByb3dzLmFwcGVuZChyb3cpCiAgICByZXR1cm4gcm93cwoKCmRlZiBub29wX2FjdGlvbnMobjogaW50KSAtPiBsaXN0W3R1cGxlW2ludCwgaW50LCBpbnRdXToKICAgIHJldHVybiBbKDAsIDAsIDUwMCldICogbgoKCmRlZiB3cml0ZV9hY3Rpb25zKHBhdGg6IFBhdGgsIGFjdGlvbnM6IFNlcXVlbmNlW3R1cGxlW2ludCwgaW50LCBpbnRdXSkgLT4gTm9uZToKICAgIG4gPSBsZW4oYWN0aW9ucykKICAgIGxpbmVzID0gW3N0cihuKV0KICAgIGZvciBzb3VyY2VfaWQsIChuZWlnaGJvciwgc2xvdCwgYWxwaGEpIGluIGVudW1lcmF0ZShhY3Rpb25zLCAxKToKICAgICAgICBpZiBub3QgKDAgPD0gbmVpZ2hib3IgPD0gbiBhbmQgMCA8PSBzbG90IDw9IG4gKyAyIGFuZCAwIDw9IGFscGhhIDwgMioqMzEgLSAxKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgYWN0aW9uIGF0IHNvdXJjZSB7c291cmNlX2lkfSIpCiAgICAgICAgaWYgbmVpZ2hib3IgPT0gc291cmNlX2lkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU2VsZi1lZGdlIGF0IHNvdXJjZSB7c291cmNlX2lkfSIpCiAgICAgICAgbGluZXMuYXBwZW5kKGYie3NvdXJjZV9pZH0ge25laWdoYm9yfSB7c2xvdH0ge2FscGhhfSIpCiAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfc3VmZml4KHBhdGguc3VmZml4ICsgIi50bXAiKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoIlxuIi5qb2luKGxpbmVzKSArICJcbiIpCiAgICBvcy5yZXBsYWNlKHRlbXBvcmFyeSwgcGF0aCkKCgpkZWYgX3BhcmFtZXRlcl9maWxlKHBhdGg6IFBhdGgsIHByb2JsZW06IFBhdGgsIHNlZWQ6IGludCwgd2FsbF9idWRnZXRfczogZmxvYXQsCiAgICAgICAgICAgICAgICAgICAgbWF4X3RyaWFsczogaW50LCBleHRyYTogZGljdFtzdHIsIHN0ciB8IGludCB8IGZsb2F0XSkgLT4gTm9uZToKICAgIHBhcmFtczogZGljdFtzdHIsIHN0ciB8IGludCB8IGZsb2F0XSA9IHsKICAgICAgICAiUFJPQkxFTV9GSUxFIjogc3RyKHByb2JsZW0ucmVzb2x2ZSgpKSwKICAgICAgICAiQ0FORElEQVRFX1NFVF9UWVBFIjogIlBPUE1VU0lDIiwKICAgICAgICAiTUFYX0NBTkRJREFURVMiOiA1LAogICAgICAgICJTVUJHUkFESUVOVCI6ICJOTyIsCiAgICAgICAgIlNFRUQiOiBzZWVkLAogICAgICAgICJSVU5TIjogMSwKICAgICAgICAiTUFYX1RSSUFMUyI6IG1heF90cmlhbHMsCiAgICAgICAgIyBMS0gncyBUSU1FX0xJTUlUIG1lYXN1cmVzIHByb2Nlc3MgQ1BVIHRpbWUuIFRoZSBjb29yZGluYXRvciBiZWxvdwogICAgICAgICMgZW5mb3JjZXMgdGhlIGFjdHVhbCBlbmQtdG8tZW5kIHdhbGwgYnVkZ2V0LgogICAgICAgICJUSU1FX0xJTUlUIjogbWF4KDEwLjAsIDEwLjAgKiB3YWxsX2J1ZGdldF9zKSwKICAgICAgICAiVFJBQ0VfTEVWRUwiOiAwLAogICAgICAgICJTVE9QX0FUX09QVElNVU0iOiAiTk8iLAogICAgfQogICAgcGFyYW1zLnVwZGF0ZShleHRyYSkKICAgIHBhdGgud3JpdGVfdGV4dCgiXG4iLmpvaW4oZiJ7a2V5fSA9IHt2YWx1ZX0iIGZvciBrZXksIHZhbHVlIGluIHBhcmFtcy5pdGVtcygpKSArICJcbiIpCgoKZGVmIHJ1bl9zaXgoCiAgICBsa2hfYmluYXJ5OiBQYXRoLAogICAgcHJvYmxlbTogUGF0aCwKICAgIHdvcmtfZGlyOiBQYXRoLAogICAgc2VlZHM6IFNlcXVlbmNlW2ludF0sCiAgICB3YWxsX2J1ZGdldF9zOiBmbG9hdCwKICAgIGFjdGlvbl9mbjogQ2FsbGFibGVbW2xpc3RbU25hcHNob3RdXSwgU2VxdWVuY2VbU2VxdWVuY2VbdHVwbGVbaW50LCBpbnQsIGludF1dXV0sCiAgICAqLAogICAgbWF4X3RyaWFsczogaW50ID0gMTAwXzAwMF8wMDAsCiAgICBleHRyYV9wYXJhbXM6IGRpY3Rbc3RyLCBzdHIgfCBpbnQgfCBmbG9hdF0gfCBOb25lID0gTm9uZSwKICAgIGJhcnJpZXJfbmFtZTogc3RyID0gImZpcnN0LnRvdXIiLAogICAgaG9va19hZnRlcl9zOiBmbG9hdCA9IDAuMCwKKSAtPiBkaWN0OgogICAgIiIiUnVuIG9uZSBpbnN0YW5jZSB3aXRoIHNpeCBzZWVkczsgY2hhcmdlIHN0YXJ0dXAsIGJhcnJpZXIgYW5kIHBvbGljeSB0aW1lLgoKICAgIGB3b3JrX2RpcmAgbXVzdCBiZSBuZXcgdG8gYXZvaWQgc3RhbGUgYWN0aW9uIGZpbGVzLiBSdW5zIG1heSBlbmQgYmVmb3JlIHRoZQogICAgZGVhZGxpbmUgd2hlbiBgbWF4X3RyaWFsc2AgaXMgc21hbGwuIFJlc3VsdHMgaW5jbHVkZSBhbGwgaG9vayBmaWxlcyBzbyB0aGUKICAgIGNhbGxlciBjYW4gYXVkaXQgdGhlIGFjdHVhbCBjYW5kaWRhdGUgYXJyYXlzIGFuZCBmaXJzdCB0b3Vycy4KICAgICIiIgogICAgaWYgbGVuKHNlZWRzKSAhPSA2IG9yIGxlbihzZXQoc2VlZHMpKSAhPSA2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4YWN0bHkgc2l4IGRpc3RpbmN0IHNlZWRzIGFyZSByZXF1aXJlZCIpCiAgICBpZiB3YWxsX2J1ZGdldF9zIDw9IDAgb3IgbWF4X3RyaWFscyA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUG9zaXRpdmUgYnVkZ2V0IGFuZCB0cmlhbHMgYXJlIHJlcXVpcmVkIikKICAgIGlmIGJhcnJpZXJfbmFtZSBub3QgaW4gKCJmaXJzdC50b3VyIiwgImJhcnJpZXIudG91ciIpIG9yIGhvb2tfYWZ0ZXJfcyA8IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCBiYXJyaWVyIG1hcmtlciBvciBkZWxheSIpCiAgICBpZiB3b3JrX2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoZiJVc2UgYSBmcmVzaCBydW4gZGlyZWN0b3J5OiB7d29ya19kaXJ9IikKICAgIHdvcmtfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIHByb2Nlc3NlczogbGlzdFtzdWJwcm9jZXNzLlBvcGVuXSA9IFtdCiAgICBvcGVuX2xvZ3M6IGxpc3QgPSBbXQogICAgc3RhcnRfbW9ubyA9IHRpbWUubW9ub3RvbmljKCkKICAgIHN0YXJ0X2Vwb2NoID0gdGltZS50aW1lKCkKICAgIGRlYWRsaW5lID0gc3RhcnRfbW9ubyArIHdhbGxfYnVkZ2V0X3MKICAgIHRyeToKICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgc2VlZF9kaXIgPSB3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iCiAgICAgICAgICAgIHNlZWRfZGlyLm1rZGlyKCkKICAgICAgICAgICAgcGFyYW1zID0gc2VlZF9kaXIgLyAicnVuLnBhciIKICAgICAgICAgICAgX3BhcmFtZXRlcl9maWxlKHBhcmFtcywgcHJvYmxlbSwgc2VlZCwgd2FsbF9idWRnZXRfcywgbWF4X3RyaWFscywKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGV4dHJhX3BhcmFtcyBvciB7fSkKICAgICAgICAgICAgc3Rkb3V0ID0gKHNlZWRfZGlyIC8gInN0ZG91dC50eHQiKS5vcGVuKCJ3IikKICAgICAgICAgICAgc3RkZXJyID0gKHNlZWRfZGlyIC8gInN0ZGVyci50eHQiKS5vcGVuKCJ3IikKICAgICAgICAgICAgb3Blbl9sb2dzLmV4dGVuZCgoc3Rkb3V0LCBzdGRlcnIpKQogICAgICAgICAgICBlbnYgPSBkaWN0KG9zLmVudmlyb24pCiAgICAgICAgICAgIGVudlsiQkVUQVRPVVJfSE9PS19ESVIiXSA9IHN0cihzZWVkX2Rpci5yZXNvbHZlKCkpCiAgICAgICAgICAgIGVudlsiQkVUQVRPVVJfSE9PS19USU1FT1VUX1MiXSA9IHN0cihtYXgoNS4wLCB3YWxsX2J1ZGdldF9zICsgNS4wKSkKICAgICAgICAgICAgZW52WyJCRVRBVE9VUl9IT09LX0FGVEVSX1MiXSA9IHN0cihob29rX2FmdGVyX3MpCiAgICAgICAgICAgIHByb2Nlc3Nlcy5hcHBlbmQoc3VicHJvY2Vzcy5Qb3BlbigKICAgICAgICAgICAgICAgIFtzdHIobGtoX2JpbmFyeS5yZXNvbHZlKCkpLCBzdHIocGFyYW1zLnJlc29sdmUoKSldLAogICAgICAgICAgICAgICAgY3dkPXNlZWRfZGlyLCBzdGRvdXQ9c3Rkb3V0LCBzdGRlcnI9c3RkZXJyLCBlbnY9ZW52LAogICAgICAgICAgICApKQogICAgICAgIGZpcnN0X3BhdGhzID0gW3dvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIgLyBiYXJyaWVyX25hbWUgZm9yIHNlZWQgaW4gc2VlZHNdCiAgICAgICAgd2hpbGUgbm90IGFsbChwYXRoLmV4aXN0cygpIGZvciBwYXRoIGluIGZpcnN0X3BhdGhzKToKICAgICAgICAgICAgZmFpbHVyZXMgPSBbcC5yZXR1cm5jb2RlIGZvciBwIGluIHByb2Nlc3NlcyBpZiBwLnBvbGwoKSBub3QgaW4gKE5vbmUsIDApXQogICAgICAgICAgICBpZiBmYWlsdXJlczoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkxLSCBleGl0ZWQgYmVmb3JlIGZpcnN0LXRvdXIgYmFycmllcjoge2ZhaWx1cmVzfSIpCiAgICAgICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPj0gZGVhZGxpbmU6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICB0aW1lLnNsZWVwKDAuMDIpCiAgICAgICAgc25hcHNob3RzOiBsaXN0W1NuYXBzaG90XSA9IFtdCiAgICAgICAgaWYgYWxsKHBhdGguZXhpc3RzKCkgZm9yIHBhdGggaW4gZmlyc3RfcGF0aHMpIGFuZCB0aW1lLm1vbm90b25pYygpIDwgZGVhZGxpbmU6CiAgICAgICAgICAgIGZvciBzZWVkIGluIHNlZWRzOgogICAgICAgICAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICAgICAgICAgIHNuYXBzaG90cy5hcHBlbmQoU25hcHNob3QoCiAgICAgICAgICAgICAgICAgICAgc2VlZD1zZWVkLAogICAgICAgICAgICAgICAgICAgIGZpcnN0X3RvdXI9cmVhZF90b3VyKGJhc2UgLyBiYXJyaWVyX25hbWUpLAogICAgICAgICAgICAgICAgICAgIG5hdGl2ZV9jYW5kaWRhdGVzPXJlYWRfY2FuZGlkYXRlcyhiYXNlIC8gIm5hdGl2ZS5jYW5kaWRhdGVzIiksCiAgICAgICAgICAgICAgICAgICAgbGl2ZV9jYW5kaWRhdGVzPXJlYWRfY2FuZGlkYXRlcyhiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpLAogICAgICAgICAgICAgICAgKSkKICAgICAgICAgICAgYWN0aW9ucyA9IGFjdGlvbl9mbihzbmFwc2hvdHMpCiAgICAgICAgICAgIGlmIGxlbihhY3Rpb25zKSAhPSBsZW4oc2VlZHMpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUG9saWN5IG11c3QgcmV0dXJuIG9uZSBhY3Rpb24gbWF0cml4IHBlciBzZWVkIikKICAgICAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA8IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgZm9yIHNlZWQsIHNuYXBzaG90LCBhY3Rpb24gaW4gemlwKHNlZWRzLCBzbmFwc2hvdHMsIGFjdGlvbnMpOgogICAgICAgICAgICAgICAgICAgIGlmIGxlbihhY3Rpb24pICE9IGxlbihzbmFwc2hvdC5maXJzdF90b3VyKToKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlBvbGljeSBkaW1lbnNpb24gbWlzbWF0Y2ggZm9yIHNlZWQge3NlZWR9IikKICAgICAgICAgICAgICAgICAgICB3cml0ZV9hY3Rpb25zKHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIgLyAiYWN0aW9uLnR4dCIsIGFjdGlvbikKICAgICAgICB3aGlsZSBhbnkocC5wb2xsKCkgaXMgTm9uZSBmb3IgcCBpbiBwcm9jZXNzZXMpIGFuZCB0aW1lLm1vbm90b25pYygpIDwgZGVhZGxpbmU6CiAgICAgICAgICAgIHRpbWUuc2xlZXAoMC4wMikKICAgIGZpbmFsbHk6CiAgICAgICAgZm9yIHByb2Nlc3MgaW4gcHJvY2Vzc2VzOgogICAgICAgICAgICBpZiBwcm9jZXNzLnBvbGwoKSBpcyBOb25lOgogICAgICAgICAgICAgICAgcHJvY2Vzcy50ZXJtaW5hdGUoKQogICAgICAgIGZvciBwcm9jZXNzIGluIHByb2Nlc3NlczoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KHRpbWVvdXQ9MikKICAgICAgICAgICAgZXhjZXB0IHN1YnByb2Nlc3MuVGltZW91dEV4cGlyZWQ6CiAgICAgICAgICAgICAgICBwcm9jZXNzLmtpbGwoKQogICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KCkKICAgICAgICBmb3Igc3RyZWFtIGluIG9wZW5fbG9nczoKICAgICAgICAgICAgc3RyZWFtLmNsb3NlKCkKICAgIHJlc3VsdCA9IHsKICAgICAgICAic2VlZHMiOiBsaXN0KHNlZWRzKSwKICAgICAgICAid2FsbF9idWRnZXRfcyI6IHdhbGxfYnVkZ2V0X3MsCiAgICAgICAgIndhbGxfZWxhcHNlZF9zIjogdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0X21vbm8sCiAgICAgICAgInN0YXJ0X2Vwb2NoX3MiOiBzdGFydF9lcG9jaCwKICAgICAgICAiZGVhZGxpbmVfZXBvY2hfcyI6IHN0YXJ0X2Vwb2NoICsgd2FsbF9idWRnZXRfcywKICAgICAgICAiYmFycmllcl9jb21wbGV0ZWQiOiBsZW4oc25hcHNob3RzKSA9PSA2LAogICAgICAgICJydW5zIjogW10sCiAgICB9CiAgICBmb3Igc2VlZCwgcHJvY2VzcyBpbiB6aXAoc2VlZHMsIHByb2Nlc3Nlcyk6CiAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICBiZXN0ID0gYmFzZSAvICJiZXN0LnRvdXIiCiAgICAgICAgcmVzdWx0WyJydW5zIl0uYXBwZW5kKHsKICAgICAgICAgICAgInNlZWQiOiBzZWVkLAogICAgICAgICAgICAicmV0dXJuY29kZSI6IHByb2Nlc3MucmV0dXJuY29kZSwKICAgICAgICAgICAgImZpcnN0X3RvdXIiOiBzdHIoYmFzZSAvIGJhcnJpZXJfbmFtZSkgaWYgKGJhc2UgLyBiYXJyaWVyX25hbWUpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImJlc3RfdG91ciI6IHN0cihiZXN0KSBpZiBiZXN0LmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImJlc3RfZXZlbnRzIjogc3RyKGJhc2UgLyAiYmVzdC5ldmVudHMiKSBpZiAoYmFzZSAvICJiZXN0LmV2ZW50cyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgIm5hdGl2ZV9jYW5kaWRhdGVzIjogc3RyKGJhc2UgLyAibmF0aXZlLmNhbmRpZGF0ZXMiKSBpZiAoYmFzZSAvICJuYXRpdmUuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImxpdmVfY2FuZGlkYXRlcyI6IHN0cihiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpIGlmIChiYXNlIC8gImxpdmUuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgInBvc3RfY2FuZGlkYXRlcyI6IHN0cihiYXNlIC8gInBvc3QuY2FuZGlkYXRlcyIpIGlmIChiYXNlIC8gInBvc3QuY2FuZGlkYXRlcyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICAgICAgImFwcGxpZWRfc3RhdHMiOiBzdHIoYmFzZSAvICJhcHBsaWVkLnN0YXRzIikgaWYgKGJhc2UgLyAiYXBwbGllZC5zdGF0cyIpLmV4aXN0cygpIGVsc2UgTm9uZSwKICAgICAgICB9KQogICAgKHdvcmtfZGlyIC8gInJ1bl9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MikgKyAiXG4iKQogICAgcmV0dXJuIHJlc3VsdAo=', 'v11_candidate/BetaTourSearchHook.h': '...', 'v11_candidate/BetaTourVisit.h': 'LyogU2hhcmVkLCBsb3ctb3ZlcmhlYWQgY291bnRlcnMgZm9yIGFjdGlvbi1lZGdlIGVudW1lcmF0aW9uIGluIExLIGxvb3BzLiAqLwojaWZuZGVmIEJFVEFUT1VSX1ZJU0lUX0gKI2RlZmluZSBCRVRBVE9VUl9WSVNJVF9ICgpleHRlcm4gaW50ICpCVF9BY3Rpb25UbzsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyAqQlRfQWN0aW9uVmlzaXRzOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nICpCVF9BY3Rpb25GaXJzdEdhdGVQYXNzZXM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgKkJUX0FjdGlvblBvc2l0aXZlRzE7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgKkJUX0FjdGlvbkFjY2VwdGVkRW50cmllczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbENhbmRpZGF0ZUVudW1lcmF0aW9uczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbEFjdGlvbkVudW1lcmF0aW9uczsKZXh0ZXJuIHVuc2lnbmVkIGxvbmcgbG9uZyBCVF9Ub3RhbFRvcExldmVsRW51bWVyYXRpb25zOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nIEJUX1RvdGFsVG9wTGV2ZWxBY3Rpb25FbnVtZXJhdGlvbnM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgQlRfVG90YWxBY3Rpb25GaXJzdEdhdGVQYXNzZXM7CmV4dGVybiB1bnNpZ25lZCBsb25nIGxvbmcgQlRfVG90YWxBY3Rpb25Qb3NpdGl2ZUcxOwpleHRlcm4gdW5zaWduZWQgbG9uZyBsb25nIEJUX1RvdGFsQWN0aW9uQWNjZXB0ZWRFbnRyaWVzOwp2b2lkIEJUX0ZsdXNoVGVsZW1ldHJ5UGVyaW9kaWModm9pZCk7CgovKiBDb3VudHMgYSBkaXJlY3RlZCBjYW5kaWRhdGUgd2hlbiBhIHBhdGNoZWQgc2VhcmNoIGxvb3AgZW51bWVyYXRlcyBpdC4KICogRWxpZ2liaWxpdHkgYW5kIGdhaW4gZmlsdGVycyBtYXkgc3RpbGwgcmVqZWN0IGl0IGxhdGVyLiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZUNhbmRpZGF0ZShOb2RlICpGcm9tLCBOb2RlICpUbykKewogICAgaWYgKCFCVF9BY3Rpb25UbyB8fCAhVG8pCiAgICAgICAgcmV0dXJuOwogICAgQlRfVG90YWxDYW5kaWRhdGVFbnVtZXJhdGlvbnMrKzsKICAgIGlmIChCVF9BY3Rpb25Ub1tGcm9tLT5JZF0gPT0gVG8tPklkKSB7CiAgICAgICAgQlRfVG90YWxBY3Rpb25FbnVtZXJhdGlvbnMrKzsKICAgICAgICBCVF9BY3Rpb25WaXNpdHNbRnJvbS0+SWRdKys7CiAgICB9CiAgICBpZiAoKEJUX1RvdGFsQ2FuZGlkYXRlRW51bWVyYXRpb25zICYgNjU1MzVVTEwpID09IDApCiAgICAgICAgQlRfRmx1c2hUZWxlbWV0cnlQZXJpb2RpYygpOwp9CgovKiBUaGlzIGlzIHRoZSBmaXJzdCB0Mi0+dDMgY2hvaWNlIGluIEJlc3QyLzMvNC81L0svU3BlY2lhbE9wdE1vdmUgb25seS4gKi8Kc3RhdGljIGlubGluZSB2b2lkIEJUX05vdGVUb3BMZXZlbEVudW1lcmF0ZWQoTm9kZSAqRnJvbSwgTm9kZSAqVG8pCnsKICAgIGlmICghQlRfQWN0aW9uVG8pCiAgICAgICAgcmV0dXJuOwogICAgQlRfVG90YWxUb3BMZXZlbEVudW1lcmF0aW9ucysrOwogICAgaWYgKEJUX0FjdGlvblRvW0Zyb20tPklkXSA9PSBUby0+SWQpCiAgICAgICAgQlRfVG90YWxUb3BMZXZlbEFjdGlvbkVudW1lcmF0aW9ucysrOwp9CgovKiBDYWxsZWQgYWZ0ZXIgdGhhdCBsb29wJ3MgZmlyc3QgYWRqYWNlbmN5L0cxIGdhdGUuIFRoaXMgaXMgbm90IHRoZSBmdWxsCiAqIG11bHRpLXN0ZXAgZ2FpbiBmaWx0ZXIsIGFuZCBHMSA+IDAgaXMgdHJhY2tlZCBzZXBhcmF0ZWx5LiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZVRvcExldmVsUGFzcyhOb2RlICpGcm9tLCBOb2RlICpUbywgR2FpblR5cGUgRzEpCnsKICAgIGlmICghQlRfQWN0aW9uVG8gfHwgQlRfQWN0aW9uVG9bRnJvbS0+SWRdICE9IFRvLT5JZCkKICAgICAgICByZXR1cm47CiAgICBCVF9BY3Rpb25GaXJzdEdhdGVQYXNzZXNbRnJvbS0+SWRdKys7CiAgICBCVF9Ub3RhbEFjdGlvbkZpcnN0R2F0ZVBhc3NlcysrOwogICAgaWYgKEcxID4gMCkgewogICAgICAgIEJUX0FjdGlvblBvc2l0aXZlRzFbRnJvbS0+SWRdKys7CiAgICAgICAgQlRfVG90YWxBY3Rpb25Qb3NpdGl2ZUcxKys7CiAgICB9Cn0KCi8qIFN0b3JlVG91ciBjYWxscyB0aGlzIGJlZm9yZSByZXBsYWNpbmcgT2xkUHJlZC9PbGRTdWMgb24gYWZmZWN0ZWQgbm9kZXMuCiAqIEl0IGNvdW50cyBhbiBhY3Rpb24gZWRnZSBuZXdseSBwcmVzZW50IGluIGEgbG9jYWxseSBhY2NlcHRlZCBMSyB0b3VyLiAqLwpzdGF0aWMgaW5saW5lIHZvaWQgQlRfTm90ZUFjY2VwdGVkRWRnZShOb2RlICpGcm9tKQp7CiAgICBpbnQgVG87CiAgICBpZiAoIUJUX0FjdGlvblRvIHx8ICEoVG8gPSBCVF9BY3Rpb25Ub1tGcm9tLT5JZF0pKQogICAgICAgIHJldHVybjsKICAgIGlmICgoRnJvbS0+UHJlZC0+SWQgPT0gVG8gfHwgRnJvbS0+U3VjLT5JZCA9PSBUbykgJiYKICAgICAgICBGcm9tLT5PbGRQcmVkLT5JZCAhPSBUbyAmJiBGcm9tLT5PbGRTdWMtPklkICE9IFRvKSB7CiAgICAgICAgQlRfQWN0aW9uQWNjZXB0ZWRFbnRyaWVzW0Zyb20tPklkXSsrOwogICAgICAgIEJUX1RvdGFsQWN0aW9uQWNjZXB0ZWRFbnRyaWVzKys7CiAgICB9Cn0KCiNlbmRpZgo=', 'v11_candidate/build_search_hook.py': 'IiIiQnVpbGQgdGhlIHYxMSBpbnN0cnVtZW50ZWQgZmlyc3QtdHJpYWwgaG9vayBmcm9tIHByaXN0aW5lIExLSC0zLjAuMTQuCgpPbmx5IG91ciB0d28gaGVhZGVycyBhbmQgZXhhY3Qtc291cmNlIHBhdGNoZXMgYXJlIHNoaXBwZWQuIFVwc3RyZWFtIExLSCBpcwpkb3dubG9hZGVkIHNlcGFyYXRlbHkgYnkgdGhlIG5vdGVib29rIGFuZCB2ZXJpZmllZCBiZWZvcmUgbW9kaWZpY2F0aW9uLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgcmUKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2Vzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpVUFNUUkVBTV9TSEEyNTYgPSB7CiAgICAiRmluZFRvdXIuYyI6ICI1NjdlNzJhY2M5YjczOTI0ODhkZDE1OGJlNGRmMDI2NzU2MDQyNDIwNGEwZTg4ZDJkOWI2MmY5MjNhN2MwZWVlIiwKICAgICJCZXN0Mk9wdE1vdmUuYyI6ICIxNzVhZGUwMDhhYTgyNWRiODk2YTNmZDEyMWUxNDY5MWJlYzk4M2E0YjUzOWNiNjlhOWQ2MDgyZmZhMTE3ZWFkIiwKICAgICJCZXN0M09wdE1vdmUuYyI6ICIzNWFiZmU5YzJlNjZmYjA4ZTNjMTUyOTg1NTZjZjUzODAxMDU4YzUzZTY1NTFiMDVjMDViNDg0ODFmYzAzMzVmIiwKICAgICJCZXN0NE9wdE1vdmUuYyI6ICI3MDhhYzU3NTY2ZmJiZDIzNThlMjMwMzZiOTg3MGU1Yzc3ZjNlZmQ2MDBiMmYyMWI4MGVlZDNjYjk4MTExYmY5IiwKICAgICJCZXN0NU9wdE1vdmUuYyI6ICIwOWE3Yjg1ZDIwMGEzYTYxZjExMDZiNDM1MzFlYTY2MzZjNWFhZDM5N2JmOWVlODY0YmIzNjkyZmNlNjcwYmNiIiwKICAgICJCZXN0S09wdE1vdmUuYyI6ICI2MDg4ZGJkM2U3YTM2ZjQyYTI0ZGYxN2VkZDZhZDJlZTc3YjRiMjIwN2JlZjU0NTViOWI4ZmNkYzM4YjMyYzc2IiwKICAgICJCZXN0U3BlY2lhbE9wdE1vdmUuYyI6ICJlZmJkZmI1YWU5YTEyYjgxNmUzNzM0OTAxOTY5ZTU5ZjMxYmYwMTlkMDcxM2IyM2M4YmQxOWU0YWFmMDZkMzAxIiwKICAgICJHYWluMjMuYyI6ICI4NTNkODgxYjk5YzYwMzVlMGMwOGVkYjU4ZmQ1ZDRlNDQzN2Y4OGMxNTQxNWQ4ZDE4MmRiMTk0MjdlZDlkM2FiIiwKICAgICJCcmlkZ2VHYWluLmMiOiAiZDYyNWU0ZDYzYzE5MjAzN2ZjNTQwZjcyNDlkM2VkNjJhNWY4ZjdlMmU0NGYxMDhkNWFiNzBjOWIzMTRiYjk2MyIsCiAgICAiUGF0Y2hDeWNsZXMuYyI6ICIyNTNlNDdiMWJiNjRjMTFkNzZlMzcyNDEyYjgzMzk0MTFlNGQwNzkyNDExYmU5MzdlNDY2Njc4OTViZjYwYWJjIiwKICAgICJTdG9yZVRvdXIuYyI6ICI2ODMyNTUzNjBiOGQ5ZDcxZjM5MDQ4OTgwNTJjYTQ3YWUxYmVmNWZlYTc3YzIzMTFhMjBkMTY5NjcxMTY2Zjc0IiwKfQoKTU9WRV9MT09QX0NPVU5UUyA9IHsKICAgICJCZXN0Mk9wdE1vdmUuYyI6IDEsICJCZXN0M09wdE1vdmUuYyI6IDIsCiAgICAiQmVzdDRPcHRNb3ZlLmMiOiAzLCAiQmVzdDVPcHRNb3ZlLmMiOiA0LAogICAgIkJlc3RLT3B0TW92ZS5jIjogMSwgIkJlc3RTcGVjaWFsT3B0TW92ZS5jIjogMiwKICAgICJHYWluMjMuYyI6IDMsICJCcmlkZ2VHYWluLmMiOiAzLCAiUGF0Y2hDeWNsZXMuYyI6IDMsCn0KVE9QX0xFVkVMX0ZJTEVTID0gewogICAgIkJlc3QyT3B0TW92ZS5jIiwgIkJlc3QzT3B0TW92ZS5jIiwgIkJlc3Q0T3B0TW92ZS5jIiwKICAgICJCZXN0NU9wdE1vdmUuYyIsICJCZXN0S09wdE1vdmUuYyIsICJCZXN0U3BlY2lhbE9wdE1vdmUuYyIsCn0KCl9DQU5ESURBVEVfTE9PUCA9IHJlLmNvbXBpbGUoCiAgICByIig/bSleKFsgXHRdKilmb3JccypcKFxzKihOXHcrKVxzKj1ccyooXHcrKS0+Q2FuZGlkYXRlU2V0XHMqOyIKICAgIHIiXHMqXChccyooXHcrKVxzKj1ccypcMi0+VG9ccypcKVxzKjtccypcMlwrXCtccypcKVxzKlx7IgopCgoKZGVmIF9yZXBsYWNlX29uY2Uoc291cmNlOiBzdHIsIG9sZDogc3RyLCBuZXc6IHN0cikgLT4gc3RyOgogICAgY291bnQgPSBzb3VyY2UuY291bnQob2xkKQogICAgaWYgY291bnQgIT0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwZWN0ZWQgb25lIHNvdXJjZSBhbmNob3IsIGZvdW5kIHtjb3VudH06IHtvbGRbOjcwXSFyfSIpCiAgICByZXR1cm4gc291cmNlLnJlcGxhY2Uob2xkLCBuZXcsIDEpCgoKZGVmIHBhdGNoX2ZpbmR0b3VyKHNvdXJjZTogc3RyKSAtPiBzdHI6CiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwgJyNpbmNsdWRlICJMS0guaCJcbicsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICcjaW5jbHVkZSAiTEtILmgiXG4jaW5jbHVkZSAiQmV0YVRvdXJTZWFyY2hIb29rLmgiXG4nKQogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICIgICAgZG91YmxlIEVudHJ5VGltZSA9IEdldFRpbWUoKTtcblxuICAgIHQgPSBGaXJzdE5vZGU7IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICBkb3VibGUgRW50cnlUaW1lID0gR2V0VGltZSgpO1xuXG4iCiAgICAgICAgICAgICAgICAgICAgICAgICAgICIgICAgQlRfRXhwb3J0TmF0aXZlQ2FuZGlkYXRlcygpO1xuICAgIHQgPSBGaXJzdE5vZGU7IikKICAgIHNvdXJjZSA9IF9yZXBsYWNlX29uY2Uoc291cmNlLCAiICAgICAgICAgICAgUmVjb3JkQmV0dGVyVG91cigpO1xuIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgICAgIFJlY29yZEJldHRlclRvdXIoKTtcbiAgICAgICAgICAgIEJUX1JlY29yZEJlc3QoKTtcbiIpCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgfVxuICAgIH1cbiAgICBpZiAoQmFja2JvbmVUcmlhbHMgPiAwICYmIEJhY2tib25lVHJpYWxzIDwgTWF4VHJpYWxzKSB7IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgfVxuICAgICAgICBCVF9BZnRlckZpcnN0VHJpYWwoKTtcbiIKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICB9XG4gICAgaWYgKEJhY2tib25lVHJpYWxzID4gMCAmJiBCYWNrYm9uZVRyaWFscyA8IE1heFRyaWFscykgeyIpCiAgICByZXR1cm4gc291cmNlCgoKZGVmIHBhdGNoX21vdmVfZmlsZShzb3VyY2U6IHN0ciwgZmlsZW5hbWU6IHN0cikgLT4gc3RyOgogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsICcjaW5jbHVkZSAiTEtILmgiXG4nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAnI2luY2x1ZGUgIkxLSC5oIlxuI2luY2x1ZGUgIkJldGFUb3VyVmlzaXQuaCJcbicpCgogICAgZGVmIGluc2VydF92aXNpdChtYXRjaDogcmUuTWF0Y2hbc3RyXSkgLT4gc3RyOgogICAgICAgIGluZGVudCwgX2NhbmRpZGF0ZSwgZnJvbV9uYW1lLCB0b19uYW1lID0gbWF0Y2guZ3JvdXBzKCkKICAgICAgICByZXR1cm4gbWF0Y2guZ3JvdXAoMCkgKyBmIlxue2luZGVudH0gICAgQlRfTm90ZUNhbmRpZGF0ZSh7ZnJvbV9uYW1lfSwge3RvX25hbWV9KTsiCgogICAgc291cmNlLCBjb3VudCA9IF9DQU5ESURBVEVfTE9PUC5zdWJuKGluc2VydF92aXNpdCwgc291cmNlKQogICAgaWYgY291bnQgIT0gTU9WRV9MT09QX0NPVU5UU1tmaWxlbmFtZV06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuZXhwZWN0ZWQgY2FuZGlkYXRlIGxvb3BzIGluIHtmaWxlbmFtZX06IHtjb3VudH0iKQogICAgaWYgZmlsZW5hbWUgaW4gVE9QX0xFVkVMX0ZJTEVTOgogICAgICAgIG9sZCA9ICJCVF9Ob3RlQ2FuZGlkYXRlKHQyLCB0Myk7IgogICAgICAgIHNvdXJjZSA9IF9yZXBsYWNlX29uY2Uoc291cmNlLCBvbGQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvbGQgKyAiXG4gICAgICAgIEJUX05vdGVUb3BMZXZlbEVudW1lcmF0ZWQodDIsIHQzKTsiKQogICAgICAgIGZpcnN0ID0gc291cmNlLmluZGV4KCJCVF9Ob3RlVG9wTGV2ZWxFbnVtZXJhdGVkKHQyLCB0Myk7IikKICAgICAgICBnYXRlID0gIiAgICAgICAgICAgIGNvbnRpbnVlO1xuIgogICAgICAgIGVuZCA9IHNvdXJjZS5maW5kKGdhdGUsIGZpcnN0KQogICAgICAgIGlmIGVuZCA8IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDYW5ub3QgZmluZCBmaXJzdC1zdGFnZSBnYWluIGdhdGUgaW4ge2ZpbGVuYW1lfSIpCiAgICAgICAgZW5kICs9IGxlbihnYXRlKQogICAgICAgIHNvdXJjZSA9IHNvdXJjZVs6ZW5kXSArICIgICAgICAgIEJUX05vdGVUb3BMZXZlbFBhc3ModDIsIHQzLCBHMSk7XG4iICsgc291cmNlW2VuZDpdCiAgICByZXR1cm4gc291cmNlCgoKZGVmIHBhdGNoX3N0b3JldG91cihzb3VyY2U6IHN0cikgLT4gc3RyOgogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZShzb3VyY2UsICcjaW5jbHVkZSAiTEtILmgiXG4nLAogICAgICAgICAgICAgICAgICAgICAgICAgICAnI2luY2x1ZGUgIkxLSC5oIlxuI2luY2x1ZGUgIkJldGFUb3VyVmlzaXQuaCJcbicpCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKHNvdXJjZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgICAgIEFjdGl2YXRlKHQpO1xuICAgICAgICAgICAgdC0+T2xkUHJlZCA9IHQtPlByZWQ7IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgICAgIEFjdGl2YXRlKHQpO1xuIgogICAgICAgICAgICAgICAgICAgICAgICAgICAiICAgICAgICAgICAgQlRfTm90ZUFjY2VwdGVkRWRnZSh0KTtcbiIKICAgICAgICAgICAgICAgICAgICAgICAgICAgIiAgICAgICAgICAgIHQtPk9sZFByZWQgPSB0LT5QcmVkOyIpCiAgICByZXR1cm4gc291cmNlCgoKZGVmIGJ1aWxkKHNvdXJjZV9kaXI6IFBhdGgsIG91dHB1dF9kaXI6IFBhdGgsIGpvYnM6IGludCA9IDQpIC0+IFBhdGg6CiAgICBzb3VyY2VfZGlyLCBvdXRwdXRfZGlyID0gc291cmNlX2Rpci5yZXNvbHZlKCksIG91dHB1dF9kaXIucmVzb2x2ZSgpCiAgICBzcmMgPSBzb3VyY2VfZGlyIC8gIlNSQyIKICAgIGlmIG5vdCAoc291cmNlX2RpciAvICJNYWtlZmlsZSIpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTEtIIHNvdXJjZSB0cmVlIG1pc3Npbmc6IHtzb3VyY2VfZGlyfSIpCiAgICBvcmlnaW5hbHMgPSB7fQogICAgZm9yIGZpbGVuYW1lLCBleHBlY3RlZCBpbiBVUFNUUkVBTV9TSEEyNTYuaXRlbXMoKToKICAgICAgICBkYXRhID0gKHNyYyAvIGZpbGVuYW1lKS5yZWFkX2J5dGVzKCkKICAgICAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihkYXRhKS5oZXhkaWdlc3QoKQogICAgICAgIGlmIGRpZ2VzdCAhPSBleHBlY3RlZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuZXhwZWN0ZWQgdXBzdHJlYW0gU0hBMjU2IGZvciB7ZmlsZW5hbWV9OiB7ZGlnZXN0fSIpCiAgICAgICAgb3JpZ2luYWxzW2ZpbGVuYW1lXSA9IGRhdGEuZGVjb2RlKCJ1dGYtOCIpCiAgICBpZiBvdXRwdXRfZGlyLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihmIlJlZnVzaW5nIHRvIG92ZXJ3cml0ZSB7b3V0cHV0X2Rpcn0iKQogICAgc2h1dGlsLmNvcHl0cmVlKHNvdXJjZV9kaXIsIG91dHB1dF9kaXIsCiAgICAgICAgICAgICAgICAgICAgaWdub3JlPXNodXRpbC5pZ25vcmVfcGF0dGVybnMoIk9CSiIsICJMS0giLCAiKi5vIikpCiAgICAob3V0cHV0X2RpciAvICJTUkMiIC8gIk9CSiIpLm1rZGlyKGV4aXN0X29rPVRydWUpCiAgICAob3V0cHV0X2RpciAvICJTUkMiIC8gIkZpbmRUb3VyLmMiKS53cml0ZV90ZXh0KAogICAgICAgIHBhdGNoX2ZpbmR0b3VyKG9yaWdpbmFsc1siRmluZFRvdXIuYyJdKSkKICAgIGZvciBmaWxlbmFtZSBpbiBNT1ZFX0xPT1BfQ09VTlRTOgogICAgICAgIChvdXRwdXRfZGlyIC8gIlNSQyIgLyBmaWxlbmFtZSkud3JpdGVfdGV4dCgKICAgICAgICAgICAgcGF0Y2hfbW92ZV9maWxlKG9yaWdpbmFsc1tmaWxlbmFtZV0sIGZpbGVuYW1lKSkKICAgIChvdXRwdXRfZGlyIC8gIlNSQyIgLyAiU3RvcmVUb3VyLmMiKS53cml0ZV90ZXh0KAogICAgICAgIHBhdGNoX3N0b3JldG91cihvcmlnaW5hbHNbIlN0b3JlVG91ci5jIl0pKQogICAgZm9yIGhlYWRlciBpbiAoIkJldGFUb3VyU2VhcmNoSG9vay5oIiwgIkJldGFUb3VyVmlzaXQuaCIpOgogICAgICAgIHNodXRpbC5jb3B5MihQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoaGVhZGVyKSwgb3V0cHV0X2RpciAvICJTUkMiIC8gaGVhZGVyKQogICAgcmVzdWx0ID0gc3VicHJvY2Vzcy5ydW4oWyJtYWtlIiwgIi1qIiwgc3RyKGpvYnMpXSwgY3dkPW91dHB1dF9kaXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUsIGNoZWNrPUZhbHNlKQogICAgaWYgcmVzdWx0LnJldHVybmNvZGU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTEtIIGluc3RydW1lbnRlZCBidWlsZCBmYWlsZWQ6XG57cmVzdWx0LnN0ZG91dFstODAwMDpdfSIpCiAgICBiaW5hcnkgPSBvdXRwdXRfZGlyIC8gIkxLSCIKICAgIGlmIG5vdCBiaW5hcnkuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJNaXNzaW5nIGJ1aWx0IGJpbmFyeToge2JpbmFyeX0iKQogICAgbWFuaWZlc3QgPSB7InVwc3RyZWFtX3NoYTI1NiI6IFVQU1RSRUFNX1NIQTI1NiwKICAgICAgICAgICAgICAgICJzZWFyY2hfaG9va19zaGEyNTYiOiBoYXNobGliLnNoYTI1NigKICAgICAgICAgICAgICAgICAgICBQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoIkJldGFUb3VyU2VhcmNoSG9vay5oIikucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAgICAgICAgICJ2aXNpdF9oZWFkZXJfc2hhMjU2IjogaGFzaGxpYi5zaGEyNTYoCiAgICAgICAgICAgICAgICAgICAgUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCJCZXRhVG91clZpc2l0LmgiKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICAgICAgICAgImJpbmFyeV9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihiaW5hcnkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAgICAgICAgICJpbnN0cnVtZW50ZWRfbW92ZV9sb29wcyI6IHN1bShNT1ZFX0xPT1BfQ09VTlRTLnZhbHVlcygpKX0KICAgIChvdXRwdXRfZGlyIC8gImJldGF0b3VyX2J1aWxkLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVudD0yKSArICJcbiIpCiAgICByZXR1cm4gYmluYXJ5CgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc291cmNlLWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tam9icyIsIHR5cGU9aW50LCBkZWZhdWx0PTQpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoYnVpbGQoYXJncy5zb3VyY2VfZGlyLCBhcmdzLm91dHB1dF9kaXIsIGFyZ3Muam9icykpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=', 'v11_candidate/policy.py': 'IiIiTWF0Y2hlZCBzcGFyc2UgY2FuZGlkYXRlIGNvbnRyb2xzIGZvciB0aGUgQmV0YVRvdXIgUE9QNiBtZWNoYW5pc20gcGlsb3QuCgpUaGUgZm91ciBhcm1zIGFsbCBwYXkgZm9yIG9uZSBUb3VyRm9sZCBpbmZlcmVuY2UuIEJldGEsIGZyZXF1ZW5jeSwgYW5kIHJhbmRvbQp1c2UgdGhlICpzYW1lKiBtb2RlbC1kZWZpbmVkIHNvdXJjZS1ub2RlIGdhdGUgYW5kIGFkZCBvbmUgY2FuZGlkYXRlIGZyb20gdGhlCnNhbWUgbW9kZWwgZ3JhcGggYXQgZWFjaCBzZWxlY3RlZCBzb3VyY2UuIEEgY2FuZGlkYXRlIG11c3QgYmUgYWJzZW50IGZyb20gdGhlCnVuZGlyZWN0ZWQgUE9QNiBncmFwaCBhbmQgdGhhdCBzZWVkJ3MgZmlyc3QgdG91ci4gU2VsZWN0ZWQgYWRkaXRpb25zIG11c3QgYWxzbwpiZSBkaXN0aW5jdCBhcyB1bmRpcmVjdGVkIGVkZ2VzLiBQT1A2IGVkZ2VzIGFyZSBuZXZlciBkZWxldGVkIG9yIHJld2VpZ2h0ZWQuCgpUaGlzIGlzIGEgbWVjaGFuaXNtIGV4cGVyaW1lbnQuIFRvdXJGb2xkIHByb2JhYmlsaXRpZXMgcHJlZGljdCByZWZlcmVuY2UtdG91cgptZW1iZXJzaGlwLCBub3QgdGhlIG1hcmdpbmFsIHZhbHVlIG9mIGEgY2FuZGlkYXRlIHRvIExLSCBzZWFyY2guCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCgoKQVJNUyA9ICgic2hhbSIsICJiZXRhX2FkZCIsICJmcmVxdWVuY3lfYWRkIiwgInJhbmRvbV9hZGQiKQpHVUlERURfQVJNUyA9IEFSTVNbMTpdCkFDVElPTl9TTE9UID0gMwpBQ1RJT05fQUxQSEEgPSA1MDAKCgpkZWYgX3VuZGlyZWN0ZWRfa2V5KGE6IGludCwgYjogaW50LCBuOiBpbnQpIC0+IGludDoKICAgIHJldHVybiBtaW4oYSwgYikgKiBuICsgbWF4KGEsIGIpCgoKZGVmIF9jYW5vbmljYWxfZWRnZV9oYXNoKHRvdXI6IG5wLm5kYXJyYXkpIC0+IHN0cjoKICAgIHRvdXIgPSBucC5hc2FycmF5KHRvdXIsIGR0eXBlPW5wLmludDMyKQogICAgZm9sbG93aW5nID0gbnAucm9sbCh0b3VyLCAtMSkKICAgIGVkZ2VzID0gbnAuY29sdW1uX3N0YWNrKChucC5taW5pbXVtKHRvdXIsIGZvbGxvd2luZyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbnAubWF4aW11bSh0b3VyLCBmb2xsb3dpbmcpKSkKICAgIGVkZ2VzID0gbnAuYXNjb250aWd1b3VzYXJyYXkoZWRnZXNbbnAubGV4c29ydCgoZWRnZXNbOiwgMV0sIGVkZ2VzWzosIDBdKSldKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KGVkZ2VzLnRvYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgX2hhc2hfaW50cyh2YWx1ZXM6IG5wLm5kYXJyYXkpIC0+IHN0cjoKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihucC5hc2NvbnRpZ3VvdXNhcnJheSh2YWx1ZXMsIGR0eXBlPW5wLmludDMyKS50b2J5dGVzKCkpLmhleGRpZ2VzdCgpCgoKZGVmIF9zbmFwc2hvdF90b3VycyhzbmFwc2hvdHM6IGxpc3RbQW55XSwgeHk6IG5wLm5kYXJyYXksIFQ6IEFueSkgLT4gdHVwbGVbbGlzdFtucC5uZGFycmF5XSwgbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICBuID0gbGVuKHh5KQogICAgdG91cnMsIHN1Y2Nlc3NvcnMsIGNvc3RzID0gW10sIFtdLCBbXQogICAgZm9yIHNuYXBzaG90IGluIHNuYXBzaG90czoKICAgICAgICB0b3VyID0gbnAuYXNhcnJheShzbmFwc2hvdC5maXJzdF90b3VyLCBkdHlwZT1ucC5pbnQzMikgLSAxCiAgICAgICAgaWYgbm90IFQuaXNfdmFsaWRfdG91cih0b3VyLCBuKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImludmFsaWQgZmlyc3QgdG91ciBmb3Igc2VlZCB7c25hcHNob3Quc2VlZH0iKQogICAgICAgIHRvdXJzLmFwcGVuZCh0b3VyKQogICAgICAgIHN1Y2Nlc3NvcnMuYXBwZW5kKG5wLmFzYXJyYXkoVC50b3VyX3N1Y2NfcHJlZCh0b3VyKVswXSwgZHR5cGU9bnAuaW50MzIpKQogICAgICAgIGNvc3RzLmFwcGVuZChmbG9hdChULnRvdXJfbGVuZ3RoKHh5LCB0b3VyLCAiRVVDXzJEIikpKQogICAgcmV0dXJuIHRvdXJzLCBucC5zdGFjayhzdWNjZXNzb3JzLCBheGlzPTEpLCBucC5hc2FycmF5KGNvc3RzLCBkdHlwZT1ucC5mbG9hdDY0KQoKCmRlZiBfdmFsaWRhdGVfcHJlZGljdGlvbihwcmVkaWN0aW9uOiBkaWN0LCBuOiBpbnQpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgZ3JhcGggPSBucC5hc2FycmF5KHByZWRpY3Rpb25bIk5CIl0sIGR0eXBlPW5wLmludDMyKQogICAgcHJvYmFiaWxpdHkgPSBucC5hc2FycmF5KHByZWRpY3Rpb25bInByb2IiXSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGRpc3RhbmNlID0gbnAuYXNhcnJheShwcmVkaWN0aW9uWyJEIl0sIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBpZiBncmFwaC5zaGFwZSAhPSBwcm9iYWJpbGl0eS5zaGFwZSBvciBncmFwaC5zaGFwZSAhPSBkaXN0YW5jZS5zaGFwZSBvciBncmFwaC5uZGltICE9IDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibW9kZWwgZ3JhcGgsIHByb2JhYmlsaXR5IGFuZCBkaXN0YW5jZSBzaGFwZXMgZGlmZmVyIikKICAgIGlmIGdyYXBoLnNoYXBlWzBdICE9IG4gb3Igbm90IDEgPD0gZ3JhcGguc2hhcGVbMV0gPCBuOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1vZGVsIGdyYXBoIGhhcyBpbnZhbGlkIGRpbWVuc2lvbnMiKQogICAgaWYgbnAuYW55KChncmFwaCA8IDApIHwgKGdyYXBoID49IG4pKSBvciBucC5hbnkoZ3JhcGggPT0gbnAuYXJhbmdlKG4pWzosIE5vbmVdKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtb2RlbCBncmFwaCBjb250YWlucyBpbnZhbGlkL3NlbGYgZWRnZXMiKQogICAgaWYgbnAuYW55KG5wLnNvcnQoZ3JhcGgsIGF4aXM9MSlbOiwgMTpdID09IG5wLnNvcnQoZ3JhcGgsIGF4aXM9MSlbOiwgOi0xXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibW9kZWwgZ3JhcGggY29udGFpbnMgZHVwbGljYXRlIG5laWdoYm91cnMiKQogICAgaWYgbm90IG5wLmlzZmluaXRlKHByb2JhYmlsaXR5KS5hbGwoKSBvciBub3QgbnAuaXNmaW5pdGUoZGlzdGFuY2UpLmFsbCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1vZGVsIHByZWRpY3Rpb24gaXMgbm90IGZpbml0ZSIpCiAgICBpZiBucC5hbnkoKHByb2JhYmlsaXR5IDwgMCkgfCAocHJvYmFiaWxpdHkgPiAxKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibW9kZWwgcHJvYmFiaWxpdHkgaXMgb3V0c2lkZSBbMCwxXSIpCiAgICByZXR1cm4gZ3JhcGgsIHByb2JhYmlsaXR5LCBkaXN0YW5jZQoKCmRlZiBfZXhpc3RpbmdfZ3JhcGgoc25hcHNob3Q6IEFueSwgbjogaW50KSAtPiBzZXRbaW50XToKICAgIHJvd3MgPSBzbmFwc2hvdC5saXZlX2NhbmRpZGF0ZXMKICAgIGlmIGxlbihyb3dzKSAhPSBuOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNhbmRpZGF0ZSBzbmFwc2hvdCBoYXMgd3JvbmcgZGltZW5zaW9uIikKICAgIGVkZ2VzOiBzZXRbaW50XSA9IHNldCgpCiAgICBmb3Igc291cmNlLCByb3cgaW4gZW51bWVyYXRlKHJvd3MpOgogICAgICAgIGZvciBkZXN0aW5hdGlvbiwgX2FscGhhIGluIHJvdzoKICAgICAgICAgICAgZGVzdGluYXRpb24gPSBpbnQoZGVzdGluYXRpb24pIC0gMQogICAgICAgICAgICBpZiBub3QgMCA8PSBkZXN0aW5hdGlvbiA8IG4gb3IgZGVzdGluYXRpb24gPT0gc291cmNlOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2FuZGlkYXRlIHNuYXBzaG90IGhhcyBpbnZhbGlkIG5laWdoYm91ciIpCiAgICAgICAgICAgIGVkZ2VzLmFkZChfdW5kaXJlY3RlZF9rZXkoc291cmNlLCBkZXN0aW5hdGlvbiwgbikpCiAgICByZXR1cm4gZWRnZXMKCgpkZWYgX2Vuc2VtYmxlX2NvdW50cyh0b3VyczogbGlzdFtucC5uZGFycmF5XSwgbjogaW50KSAtPiBkaWN0W2ludCwgaW50XToKICAgIGNvdW50czogZGljdFtpbnQsIGludF0gPSB7fQogICAgZm9yIHRvdXIgaW4gdG91cnM6CiAgICAgICAgZm9yIGEsIGIgaW4gemlwKHRvdXIsIG5wLnJvbGwodG91ciwgLTEpKToKICAgICAgICAgICAga2V5ID0gX3VuZGlyZWN0ZWRfa2V5KGludChhKSwgaW50KGIpLCBuKQogICAgICAgICAgICBjb3VudHNba2V5XSA9IGNvdW50cy5nZXQoa2V5LCAwKSArIDEKICAgIHJldHVybiBjb3VudHMKCgpkZWYgX2NhbmRpZGF0ZV9vcHRpb25zKAogICAgc291cmNlOiBpbnQsCiAgICBncmFwaDogbnAubmRhcnJheSwKICAgIHByb2JhYmlsaXR5OiBucC5uZGFycmF5LAogICAgZGlzdGFuY2U6IG5wLm5kYXJyYXksCiAgICBleGlzdGluZzogc2V0W2ludF0sCiAgICBjb3VudHM6IGRpY3RbaW50LCBpbnRdLAogICAgbjogaW50LAogICAgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLAopIC0+IGRpY3Rbc3RyLCBsaXN0W2ludF1dOgogICAgbm92ZWxfY29scyA9IFtjb2x1bW4gZm9yIGNvbHVtbiwgZGVzdGluYXRpb24gaW4gZW51bWVyYXRlKGdyYXBoW3NvdXJjZV0pCiAgICAgICAgICAgICAgICAgIGlmIF91bmRpcmVjdGVkX2tleShzb3VyY2UsIGludChkZXN0aW5hdGlvbiksIG4pIG5vdCBpbiBleGlzdGluZ10KICAgIGlmIG5vdCBub3ZlbF9jb2xzOgogICAgICAgIHJldHVybiB7YXJtOiBbXSBmb3IgYXJtIGluIEdVSURFRF9BUk1TfQogICAgYmV0YV9vcmRlciA9IHNvcnRlZChub3ZlbF9jb2xzLCBrZXk9bGFtYmRhIGo6ICgtZmxvYXQocHJvYmFiaWxpdHlbc291cmNlLCBqXSksIGopKQogICAgZnJlcXVlbmN5X29yZGVyID0gc29ydGVkKAogICAgICAgIG5vdmVsX2NvbHMsCiAgICAgICAga2V5PWxhbWJkYSBqOiAoLWNvdW50cy5nZXQoX3VuZGlyZWN0ZWRfa2V5KHNvdXJjZSwgaW50KGdyYXBoW3NvdXJjZSwgal0pLCBuKSwgMCksCiAgICAgICAgICAgICAgICAgICAgICAgZmxvYXQoZGlzdGFuY2Vbc291cmNlLCBqXSksIGludChncmFwaFtzb3VyY2UsIGpdKSksCiAgICApCiAgICByYW5kb21fb3JkZXIgPSBsaXN0KG5wLmFzYXJyYXkobm92ZWxfY29scylbcm5nLnBlcm11dGF0aW9uKGxlbihub3ZlbF9jb2xzKSldKQogICAgcmV0dXJuIHsKICAgICAgICAiYmV0YV9hZGQiOiBbaW50KGdyYXBoW3NvdXJjZSwgal0pIGZvciBqIGluIGJldGFfb3JkZXJdLAogICAgICAgICJmcmVxdWVuY3lfYWRkIjogW2ludChncmFwaFtzb3VyY2UsIGpdKSBmb3IgaiBpbiBmcmVxdWVuY3lfb3JkZXJdLAogICAgICAgICJyYW5kb21fYWRkIjogW2ludChncmFwaFtzb3VyY2UsIGpdKSBmb3IgaiBpbiByYW5kb21fb3JkZXJdLAogICAgfQoKCmRlZiBjaG9vc2VfYWN0aW9ucygKICAgIGtpbmQ6IHN0ciwKICAgIHNuYXBzaG90czogbGlzdFtBbnldLAogICAgeHk6IG5wLm5kYXJyYXksCiAgICBtb2RlbDogQW55LAogICAgZGV2aWNlOiBBbnksCiAgICBUOiBBbnksCiAgICBNOiBBbnksCiAgICAqLAogICAgZnJhY3Rpb246IGZsb2F0ID0gMC4yNSwKICAgIGdyYXBoX2s6IGludCA9IDE2LAopIC0+IHR1cGxlW2xpc3RbbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV1dLCBkaWN0XToKICAgICIiIlJldHVybiBMS0ggYWN0aW9uIG1hdHJpY2VzIGFuZCBhIEpTT04tc2VyaWFsaXphYmxlIHBhcml0eSBhdWRpdC4KCiAgICBUaGUgc291cmNlIGdhdGUgaXMgVG91ckZvbGQncyB2OS1zdHlsZSBwb3NpdGl2ZSBwcm9iYWJpbGl0eSBtYXJnaW4gb3ZlcgogICAgdGhlIHdlYWtlc3QgbW9kZWwtdmlzaWJsZSBQT1A2IGVkZ2UuIEZyZXF1ZW5jeSBhbmQgcmFuZG9tIGNvbnRyb2xzIGtlZXAKICAgIHRoYXQgZXhhY3QgZ2F0ZS4gVGhlIG91dHB1dCBtYXkgY29udGFpbiBmZXdlciB0aGFuIGZsb29yKGZyYWN0aW9uKm4pCiAgICBhY3Rpb25zIGlmIG5vIGNvbW1vbiBmZWFzaWJsZSBub3ZlbCBjaG9pY2UgcmVtYWluczsgYWxsIGd1aWRlZCBhcm1zIHN0aWxsCiAgICBnZXQgdGhlIHNhbWUgbnVtYmVyIGFuZCBub2RlIElEcyBmb3IgZWFjaCBzZWVkLgogICAgIiIiCiAgICBpZiBraW5kIG5vdCBpbiBBUk1TOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmtub3duIGNhbmRpZGF0ZSBhcm06IHtraW5kfSIpCiAgICBpZiBsZW4oc25hcHNob3RzKSAhPSA2IG9yIGxlbih7aW50KHMuc2VlZCkgZm9yIHMgaW4gc25hcHNob3RzfSkgIT0gNjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJleGFjdGx5IHNpeCBkaXN0aW5jdCBzZWVkIHNuYXBzaG90cyByZXF1aXJlZCIpCiAgICBpZiBub3QgMC4wIDwgZnJhY3Rpb24gPD0gMS4wIG9yIGdyYXBoX2sgPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgZ2F0ZSBmcmFjdGlvbiBvciBncmFwaCB3aWR0aCIpCiAgICB4eSA9IG5wLmFzYXJyYXkoeHkpCiAgICBpZiB4eS5uZGltICE9IDIgb3IgeHkuc2hhcGVbMV0gIT0gMiBvciBsZW4oeHkpIDwgNDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ4eSBtdXN0IGhhdmUgc2hhcGUgKG4sMikiKQogICAgbiA9IGxlbih4eSkKICAgIHRvdXJzLCBzdWNjZXNzb3JzLCBjb3N0cyA9IF9zbmFwc2hvdF90b3VycyhzbmFwc2hvdHMsIHh5LCBUKQogICAgcHJlZGljdGlvbiA9IE0ucHJlZGljdF90b3VyZm9sZChtb2RlbCwgeHksIGdyYXBoX2ssIGRldmljZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZW5zX3N1Y2M9c3VjY2Vzc29ycywgZW5zX2Nvc3Q9Y29zdHMpCiAgICBncmFwaCwgcHJvYmFiaWxpdHksIGRpc3RhbmNlID0gX3ZhbGlkYXRlX3ByZWRpY3Rpb24ocHJlZGljdGlvbiwgbikKICAgIGVuc2VtYmxlX2NvdW50cyA9IF9lbnNlbWJsZV9jb3VudHModG91cnMsIG4pCiAgICBkZXN0aW5hdGlvbnMgPSB7YXJtOiBucC5mdWxsKCg2LCBuKSwgLTEsIGR0eXBlPW5wLmludDMyKSBmb3IgYXJtIGluIEdVSURFRF9BUk1TfQogICAgZ2F0ZXM6IGxpc3RbbGlzdFtpbnRdXSA9IFtdCiAgICBlbGlnaWJsZV9jb3VudHM6IGxpc3RbaW50XSA9IFtdCiAgICBwb3BfZWRnZV9jb3VudHM6IGxpc3RbaW50XSA9IFtdCiAgICBsaW1pdCA9IGludChucC5mbG9vcihmcmFjdGlvbiAqIG4pKQoKICAgIGZvciBwYW5lbF9pbmRleCwgc25hcHNob3QgaW4gZW51bWVyYXRlKHNuYXBzaG90cyk6CiAgICAgICAgZXhpc3RpbmcgPSBfZXhpc3RpbmdfZ3JhcGgoc25hcHNob3QsIG4pCiAgICAgICAgcG9wX2VkZ2VfY291bnRzLmFwcGVuZChsZW4oZXhpc3RpbmcpKQogICAgICAgIGZpcnN0X3RvdXIgPSB0b3Vyc1twYW5lbF9pbmRleF0KICAgICAgICBmaXJzdF9lZGdlcyA9IHtfdW5kaXJlY3RlZF9rZXkoaW50KGEpLCBpbnQoYiksIG4pCiAgICAgICAgICAgICAgICAgICAgICAgZm9yIGEsIGIgaW4gemlwKGZpcnN0X3RvdXIsIG5wLnJvbGwoZmlyc3RfdG91ciwgLTEpKX0KICAgICAgICBibG9ja2VkID0gZXhpc3RpbmcgfCBmaXJzdF9lZGdlcwogICAgICAgIG1hcmdpbnMgPSBucC5mdWxsKG4sIC1ucC5pbmYsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICAgICAgZm9yIHNvdXJjZSBpbiByYW5nZShuKToKICAgICAgICAgICAgdmlzaWJsZSA9IHtpbnQodik6IGZsb2F0KHApIGZvciB2LCBwIGluIHppcChncmFwaFtzb3VyY2VdLCBwcm9iYWJpbGl0eVtzb3VyY2VdKX0KICAgICAgICAgICAgc2NvcmVkX3BvcCA9IFt2aXNpYmxlW2ludCh2KSAtIDFdIGZvciB2LCBfYWxwaGEgaW4gc25hcHNob3QubGl2ZV9jYW5kaWRhdGVzW3NvdXJjZV0KICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBpbnQodikgLSAxIGluIHZpc2libGVdCiAgICAgICAgICAgIGlmIGxlbihzY29yZWRfcG9wKSA8IDI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBub3ZlbF9wcm9iID0gW2Zsb2F0KHApIGZvciB2LCBwIGluIHppcChncmFwaFtzb3VyY2VdLCBwcm9iYWJpbGl0eVtzb3VyY2VdKQogICAgICAgICAgICAgICAgICAgICAgICAgIGlmIF91bmRpcmVjdGVkX2tleShzb3VyY2UsIGludCh2KSwgbikgbm90IGluIGJsb2NrZWRdCiAgICAgICAgICAgIGlmIG5vdmVsX3Byb2I6CiAgICAgICAgICAgICAgICBtYXJnaW4gPSBtYXgobm92ZWxfcHJvYikgLSBtaW4oc2NvcmVkX3BvcCkKICAgICAgICAgICAgICAgIGlmIG1hcmdpbiA+IDA6CiAgICAgICAgICAgICAgICAgICAgbWFyZ2luc1tzb3VyY2VdID0gbWFyZ2luCiAgICAgICAgZWxpZ2libGUgPSBucC5mbGF0bm9uemVybyhucC5pc2Zpbml0ZShtYXJnaW5zKSkKICAgICAgICBlbGlnaWJsZV9jb3VudHMuYXBwZW5kKGxlbihlbGlnaWJsZSkpCiAgICAgICAgb3JkZXJlZCA9IGVsaWdpYmxlW25wLmFyZ3NvcnQoLW1hcmdpbnNbZWxpZ2libGVdLCBraW5kPSJzdGFibGUiKV0KICAgICAgICBzZWVkX21hdGVyaWFsID0gKG5wLmFzY29udGlndW91c2FycmF5KHh5KS50b2J5dGVzKCkgKwogICAgICAgICAgICAgICAgICAgICAgICAgbnAuYXNjb250aWd1b3VzYXJyYXkodG91cnNbcGFuZWxfaW5kZXhdKS50b2J5dGVzKCkgKwogICAgICAgICAgICAgICAgICAgICAgICAgaW50KHNuYXBzaG90LnNlZWQpLnRvX2J5dGVzKDgsICJsaXR0bGUiLCBzaWduZWQ9VHJ1ZSkpCiAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKGludC5mcm9tX2J5dGVzKGhhc2hsaWIuc2hhMjU2KHNlZWRfbWF0ZXJpYWwpLmRpZ2VzdCgpWzo4XSwgImxpdHRsZSIpKQogICAgICAgIHJlc2VydmVkID0ge2FybTogc2V0KCkgZm9yIGFybSBpbiBHVUlERURfQVJNU30KICAgICAgICBnYXRlOiBsaXN0W2ludF0gPSBbXQogICAgICAgIGZvciBzb3VyY2UgaW4gb3JkZXJlZDoKICAgICAgICAgICAgaWYgbGVuKGdhdGUpID49IGxpbWl0OgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgc291cmNlID0gaW50KHNvdXJjZSkKICAgICAgICAgICAgb3B0aW9ucyA9IF9jYW5kaWRhdGVfb3B0aW9ucyhzb3VyY2UsIGdyYXBoLCBwcm9iYWJpbGl0eSwgZGlzdGFuY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYmxvY2tlZCwgZW5zZW1ibGVfY291bnRzLCBuLCBybmcpCiAgICAgICAgICAgIHByb3Bvc2FsOiBkaWN0W3N0ciwgdHVwbGVbaW50LCBpbnRdXSA9IHt9CiAgICAgICAgICAgIGZvciBhcm0gaW4gR1VJREVEX0FSTVM6CiAgICAgICAgICAgICAgICBhdmFpbGFibGUgPSAoKGludChkZXN0aW5hdGlvbiksIF91bmRpcmVjdGVkX2tleShzb3VyY2UsIGludChkZXN0aW5hdGlvbiksIG4pKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBkZXN0aW5hdGlvbiBpbiBvcHRpb25zW2FybV0pCiAgICAgICAgICAgICAgICBjaG9pY2UgPSBuZXh0KCgoZGVzdGluYXRpb24sIGtleSkgZm9yIGRlc3RpbmF0aW9uLCBrZXkgaW4gYXZhaWxhYmxlCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBrZXkgbm90IGluIHJlc2VydmVkW2FybV0pLCBOb25lKQogICAgICAgICAgICAgICAgaWYgY2hvaWNlIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIHByb3Bvc2FsW2FybV0gPSBjaG9pY2UKICAgICAgICAgICAgaWYgbGVuKHByb3Bvc2FsKSAhPSBsZW4oR1VJREVEX0FSTVMpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZ2F0ZS5hcHBlbmQoc291cmNlKQogICAgICAgICAgICBmb3IgYXJtLCAoZGVzdGluYXRpb24sIGtleSkgaW4gcHJvcG9zYWwuaXRlbXMoKToKICAgICAgICAgICAgICAgIGRlc3RpbmF0aW9uc1thcm1dW3BhbmVsX2luZGV4LCBzb3VyY2VdID0gZGVzdGluYXRpb24KICAgICAgICAgICAgICAgIHJlc2VydmVkW2FybV0uYWRkKGtleSkKICAgICAgICBnYXRlcy5hcHBlbmQoZ2F0ZSkKICAgICAgICBpZiBsZW4oe2xlbihyZXNlcnZlZFthcm1dKSBmb3IgYXJtIGluIEdVSURFRF9BUk1TfSkgIT0gMToKICAgICAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoImd1aWRlZCBhcm0gY2FwYWNpdGllcyBkaWZmZXIiKQoKICAgIGNob3NlbiA9IChucC5mdWxsKCg2LCBuKSwgLTEsIGR0eXBlPW5wLmludDMyKSBpZiBraW5kID09ICJzaGFtIgogICAgICAgICAgICAgIGVsc2UgZGVzdGluYXRpb25zW2tpbmRdKQogICAgYWN0aW9ucyA9IFtbKGludCh2KSArIDEsIEFDVElPTl9TTE9ULCBBQ1RJT05fQUxQSEEpIGlmIHYgPj0gMCBlbHNlICgwLCAwLCBBQ1RJT05fQUxQSEEpCiAgICAgICAgICAgICAgICBmb3IgdiBpbiByb3ddIGZvciByb3cgaW4gY2hvc2VuXQogICAgZ2F0ZV9oYXNoZXMgPSBbX2hhc2hfaW50cyhucC5hc2FycmF5KGdhdGUsIGR0eXBlPW5wLmludDMyKSkgZm9yIGdhdGUgaW4gZ2F0ZXNdCiAgICBhY3Rpb25faGFzaGVzID0ge2FybTogW19oYXNoX2ludHMoZGVzdGluYXRpb25zW2FybV1baV0pIGZvciBpIGluIHJhbmdlKDYpXQogICAgICAgICAgICAgICAgICAgICBmb3IgYXJtIGluIEdVSURFRF9BUk1TfQogICAgc2VsZWN0ZWRfZnJlcXVlbmN5ID0ge30KICAgIHNlbGVjdGVkX3Byb2JhYmlsaXR5ID0ge30KICAgIHVuaXF1ZV91bmRpcmVjdGVkID0ge30KICAgIGZvciBhcm0gaW4gR1VJREVEX0FSTVM6CiAgICAgICAgZnJlcXVlbmNpZXNfcGVyX3NlZWQsIHByb2JhYmlsaXRpZXNfcGVyX3NlZWQsIHVuaXF1ZV9wZXJfc2VlZCA9IFtdLCBbXSwgW10KICAgICAgICBmb3IgaSwgZ2F0ZSBpbiBlbnVtZXJhdGUoZ2F0ZXMpOgogICAgICAgICAgICBmcmVxdWVuY2llcywgcHJvYmFiaWxpdGllcywga2V5cyA9IFtdLCBbXSwgc2V0KCkKICAgICAgICAgICAgZm9yIHNvdXJjZSBpbiBnYXRlOgogICAgICAgICAgICAgICAgZGVzdGluYXRpb24gPSBpbnQoZGVzdGluYXRpb25zW2FybV1baSwgc291cmNlXSkKICAgICAgICAgICAgICAgIGtleXMuYWRkKF91bmRpcmVjdGVkX2tleShzb3VyY2UsIGRlc3RpbmF0aW9uLCBuKSkKICAgICAgICAgICAgICAgIGZyZXF1ZW5jaWVzLmFwcGVuZChpbnQoZW5zZW1ibGVfY291bnRzLmdldChfdW5kaXJlY3RlZF9rZXkoc291cmNlLCBkZXN0aW5hdGlvbiwgbiksIDApKSkKICAgICAgICAgICAgICAgIGNvbHVtbiA9IG5wLmZsYXRub256ZXJvKGdyYXBoW3NvdXJjZV0gPT0gZGVzdGluYXRpb24pCiAgICAgICAgICAgICAgICBpZiBsZW4oY29sdW1uKSAhPSAxOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJzZWxlY3RlZCBlZGdlIG1pc3NpbmcgZnJvbSBtb2RlbCBncmFwaCIpCiAgICAgICAgICAgICAgICBwcm9iYWJpbGl0aWVzLmFwcGVuZChmbG9hdChwcm9iYWJpbGl0eVtzb3VyY2UsIGludChjb2x1bW5bMF0pXSkpCiAgICAgICAgICAgIGZyZXF1ZW5jaWVzX3Blcl9zZWVkLmFwcGVuZChmcmVxdWVuY2llcykKICAgICAgICAgICAgcHJvYmFiaWxpdGllc19wZXJfc2VlZC5hcHBlbmQocHJvYmFiaWxpdGllcykKICAgICAgICAgICAgdW5pcXVlX3Blcl9zZWVkLmFwcGVuZChsZW4oa2V5cykpCiAgICAgICAgICAgIGlmIGxlbihrZXlzKSAhPSBsZW4oZ2F0ZSk6CiAgICAgICAgICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigic2VsZWN0ZWQgYWN0aW9ucyBhcmUgbm90IHVuaXF1ZSB1bmRpcmVjdGVkIGVkZ2VzIikKICAgICAgICBzZWxlY3RlZF9mcmVxdWVuY3lbYXJtXSA9IGZyZXF1ZW5jaWVzX3Blcl9zZWVkCiAgICAgICAgc2VsZWN0ZWRfcHJvYmFiaWxpdHlbYXJtXSA9IHByb2JhYmlsaXRpZXNfcGVyX3NlZWQKICAgICAgICB1bmlxdWVfdW5kaXJlY3RlZFthcm1dID0gdW5pcXVlX3Blcl9zZWVkCiAgICBhdWRpdCA9IHsKICAgICAgICAiYXJtIjoga2luZCwKICAgICAgICAiZ2F0ZV9mcmFjdGlvbiI6IGZsb2F0KGZyYWN0aW9uKSwKICAgICAgICAiZ3JhcGhfayI6IGludChncmFwaF9rKSwKICAgICAgICAibW9kZWxfaW5mZXJlbmNlX3MiOiBmbG9hdChwcmVkaWN0aW9uWyJ0X3RvdGFsIl0pLAogICAgICAgICJzZWVkX2lkcyI6IFtpbnQocy5zZWVkKSBmb3IgcyBpbiBzbmFwc2hvdHNdLAogICAgICAgICJmaXJzdF90b3VyX3Jhd19zaGEyNTYiOiBbX2hhc2hfaW50cyh0b3VyKSBmb3IgdG91ciBpbiB0b3Vyc10sCiAgICAgICAgImZpcnN0X3RvdXJfZWRnZV9zaGEyNTYiOiBbX2Nhbm9uaWNhbF9lZGdlX2hhc2godG91cikgZm9yIHRvdXIgaW4gdG91cnNdLAogICAgICAgICJtb2RlbF9wcm9iYWJpbGl0eV9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihucC5hc2NvbnRpZ3VvdXNhcnJheShwcm9iYWJpbGl0eSkudG9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAibmF0aXZlX3VuZGlyZWN0ZWRfZWRnZXNfcGVyX3NlZWQiOiBwb3BfZWRnZV9jb3VudHMsCiAgICAgICAgImVsaWdpYmxlX25vZGVzX3Blcl9zZWVkIjogZWxpZ2libGVfY291bnRzLAogICAgICAgICJnYXRlX25vZGVzX3Blcl9zZWVkIjogZ2F0ZXMsCiAgICAgICAgImdhdGVfc2hhMjU2X3Blcl9zZWVkIjogZ2F0ZV9oYXNoZXMsCiAgICAgICAgImd1aWRlZF9hY3Rpb25zX3Blcl9zZWVkIjogW2xlbihnYXRlKSBmb3IgZ2F0ZSBpbiBnYXRlc10sCiAgICAgICAgInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiOiBbaW50KChyb3cgPj0gMCkuc3VtKCkpIGZvciByb3cgaW4gY2hvc2VuXSwKICAgICAgICAic2VsZWN0ZWRfdW5pcXVlX3VuZGlyZWN0ZWRfZWRnZXNfcGVyX3NlZWQiOiAoCiAgICAgICAgICAgIFswXSAqIDYgaWYga2luZCA9PSAic2hhbSIgZWxzZSB1bmlxdWVfdW5kaXJlY3RlZFtraW5kXSksCiAgICAgICAgInNlbGVjdGVkX3BhaXJzX3Blcl9zZWVkIjogW1tbaW50KHNvdXJjZSksIGludChjaG9zZW5baSwgc291cmNlXSldCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3Igc291cmNlIGluIGdhdGVzW2ldIGlmIGNob3NlbltpLCBzb3VyY2VdID49IDBdCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKDYpXSwKICAgICAgICAiYWxsX2d1aWRlZF9hY3Rpb25fc2hhMjU2X3Blcl9zZWVkIjogYWN0aW9uX2hhc2hlcywKICAgICAgICAiYWxsX2d1aWRlZF91bmlxdWVfdW5kaXJlY3RlZF9lZGdlc19wZXJfc2VlZCI6IHVuaXF1ZV91bmRpcmVjdGVkLAogICAgICAgICJhbGxfZ3VpZGVkX3NlbGVjdGVkX2Vuc2VtYmxlX2ZyZXF1ZW5jeV9wZXJfc2VlZCI6IHNlbGVjdGVkX2ZyZXF1ZW5jeSwKICAgICAgICAiYWxsX2d1aWRlZF9zZWxlY3RlZF9tb2RlbF9wcm9iYWJpbGl0eV9wZXJfc2VlZCI6IHNlbGVjdGVkX3Byb2JhYmlsaXR5LAogICAgICAgICJhY3Rpb25fc2xvdCI6IEFDVElPTl9TTE9ULAogICAgICAgICJhY3Rpb25fYWxwaGEiOiBBQ1RJT05fQUxQSEEsCiAgICB9CiAgICByZXR1cm4gYWN0aW9ucywgYXVkaXQK', 'v11_candidate/pilot.py': 'IiIiV2FsbC1jbG9jayB2MTEgUE9QNi1yZXNpZHVhbCBjYW5kaWRhdGUgcGlsb3QgaGVscGVycy4KCkFsbCBndWlkZWQgYXJtcyBzdGFydCBmcm9tIHNpeCBmcmVzaCBQT1BNVVNJQy02IExLSCBwcm9jZXNzZXMgYW5kIHBhdXNlIGF0IHRoZQpzYW1lIGNvbXBsZXRlZCBmaXJzdCB0cmlhbC4gVGhlIHVuY2hhbmdlZCBwcm9jZXNzZXMgcmVzdW1lIGFmdGVyIHRoZWlyIG9uZQpjYW5kaWRhdGUgYWN0aW9uIHBlciBzZWxlY3RlZCBzb3VyY2UuIFNvdXJjZS9jYW5kaWRhdGUgc25hcHNob3RzIGFyZSBoYXNoZWQgc28KdGhlIG5vdGVib29rIGNhbiByZWplY3QgYSBjb21wYXJpc29uIHdoZW4gdGhlIHN0YXJ0aW5nIHNlYXJjaCBzdGF0ZXMgZGlmZmVyLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gY29vcmRpbmF0b3IgaW1wb3J0IHJlYWRfY2FuZGlkYXRlcywgcmVhZF90b3VyLCBydW5fc2l4CmZyb20gcG9saWN5IGltcG9ydCBjaG9vc2VfYWN0aW9ucwoKCkdVSURFRF9BUk1TID0gKCJzaGFtIiwgImJldGFfYWRkIiwgImZyZXF1ZW5jeV9hZGQiLCAicmFuZG9tX2FkZCIpCk5BVElWRV9BUk1TID0gKCJuYXRpdmVfcG9wNiIsICJuYXRpdmVfcG9wNyIpCgoKZGVmIHNoYTI1Nl9maWxlKHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCJyYiIpIGFzIHNvdXJjZToKICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6IHNvdXJjZS5yZWFkKDQgKiAxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKZGVmIGNhbm9uaWNhbF90b3VyX2hhc2godG91cjogbnAubmRhcnJheSkgLT4gc3RyOgogICAgdCA9IG5wLmFzYXJyYXkodG91ciwgZHR5cGU9bnAuaW50MzIpCiAgICBueHQgPSBucC5yb2xsKHQsIC0xKQogICAgZWRnZXMgPSBucC5jb2x1bW5fc3RhY2soKG5wLm1pbmltdW0odCwgbnh0KSwgbnAubWF4aW11bSh0LCBueHQpKSkKICAgIGVkZ2VzID0gbnAuYXNjb250aWd1b3VzYXJyYXkoZWRnZXNbbnAubGV4c29ydCgoZWRnZXNbOiwgMV0sIGVkZ2VzWzosIDBdKSldKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KGVkZ2VzLnRvYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgY2FuZGlkYXRlX2hhc2gocm93czogbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dKSAtPiBzdHI6CiAgICAiIiJIYXNoIGZ1bGwgZGlyZWN0ZWQgY2FuZGlkYXRlIGNvbnRlbnRzLCBpbmNsdWRpbmcgYWxwaGEgYW5kIHJvdyBvcmRlci4iIiIKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICBkaWdlc3QudXBkYXRlKGxlbihyb3cpLnRvX2J5dGVzKDQsICJsaXR0bGUiKSkKICAgICAgICBmb3IgbmVpZ2hib3IsIGFscGhhIGluIHJvdzoKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShpbnQobmVpZ2hib3IpLnRvX2J5dGVzKDQsICJsaXR0bGUiLCBzaWduZWQ9RmFsc2UpKQogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGludChhbHBoYSkudG9fYnl0ZXMoOCwgImxpdHRsZSIsIHNpZ25lZD1UcnVlKSkKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgX2F0b21pY19qc29uKHBhdGg6IFBhdGgsIHZhbHVlOiBkaWN0KSAtPiBOb25lOgogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdGVtcG9yYXJ5ID0gcGF0aC53aXRoX25hbWUocGF0aC5uYW1lICsgIi50bXAiKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSwgaW5kZW50PTIsIGVuc3VyZV9hc2NpaT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYWxsb3dfbmFuPUZhbHNlKSArICJcbiIpCiAgICBvcy5yZXBsYWNlKHRlbXBvcmFyeSwgcGF0aCkKCgpkZWYgX3ZhbGlkX2Nvc3QoVCwgeHk6IG5wLm5kYXJyYXksIHRvdXI6IG5wLm5kYXJyYXkpIC0+IGludDoKICAgIHRvdXIgPSBucC5hc2FycmF5KHRvdXIsIGR0eXBlPW5wLmludDMyKQogICAgaWYgbm90IFQuaXNfdmFsaWRfdG91cih0b3VyLCBsZW4oeHkpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIEhhbWlsdG9uIHRvdXIiKQogICAgbGlicmFyeV9jb3N0ID0gaW50KFQudG91cl9sZW5ndGgoeHksIHRvdXIsICJFVUNfMkQiKSkKICAgIGEgPSBucC5hc2FycmF5KHh5W3RvdXJdLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgYiA9IG5wLmFzYXJyYXkoeHlbbnAucm9sbCh0b3VyLCAtMSldLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgaW5kZXBlbmRlbnQgPSBpbnQobnAuZmxvb3IobnAuc3FydChucC5zdW0oKGEgLSBiKSAqKiAyLCBheGlzPTEpKSArIDAuNSkKICAgICAgICAgICAgICAgICAgICAgIC5zdW0oZHR5cGU9bnAuaW50NjQpKQogICAgaWYgaW5kZXBlbmRlbnQgIT0gbGlicmFyeV9jb3N0OgogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYiaW5kZXBlbmRlbnQgRVVDXzJEIGNvc3RzIGRpc2FncmVlOiB7aW5kZXBlbmRlbnR9ICE9IHtsaWJyYXJ5X2Nvc3R9IikKICAgIHJldHVybiBsaWJyYXJ5X2Nvc3QKCgpkZWYgX2xvYWRfbmF0aXZlX3RvdXIoVCwgcGF0aDogUGF0aCkgLT4gbnAubmRhcnJheToKICAgIGZpcnN0ID0gcGF0aC5yZWFkX3RleHQoKS5zcGxpdChtYXhzcGxpdD0xKVswXQogICAgaWYgZmlyc3QuaXNkaWdpdCgpOgogICAgICAgIHJldHVybiBucC5hc2FycmF5KHJlYWRfdG91cihwYXRoKSwgZHR5cGU9bnAuaW50MzIpIC0gMQogICAgcmV0dXJuIG5wLmFzYXJyYXkoVC5yZWFkX3RvdXIocGF0aCksIGR0eXBlPW5wLmludDMyKQoKCmRlZiBfdmVyaWZ5X2FwcGxpZWRfYWN0aW9ucyh3b3JrX2RpcjogUGF0aCwgc2VlZHM6IFNlcXVlbmNlW2ludF0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBhY3Rpb25zOiBsaXN0W2xpc3RbdHVwbGVbaW50LCBpbnQsIGludF1dXSkgLT4gTm9uZToKICAgICIiIkNoZWNrIHRoZSBhY3R1YWwgTEtIIGNhbmRpZGF0ZSByb3dzLCBub3QganVzdCB0aGUgY2FsbGJhY2sncyBwcm9wb3NhbC4iIiIKICAgIGZvciBzZWVkLCBtYXRyaXggaW4gemlwKHNlZWRzLCBhY3Rpb25zKToKICAgICAgICBiYXNlID0gd29ya19kaXIgLyBmInNlZWRfe3NlZWR9IgogICAgICAgIGxpdmUgPSByZWFkX2NhbmRpZGF0ZXMoYmFzZSAvICJsaXZlLmNhbmRpZGF0ZXMiKQogICAgICAgIHBvc3QgPSByZWFkX2NhbmRpZGF0ZXMoYmFzZSAvICJwb3N0LmNhbmRpZGF0ZXMiKQogICAgICAgIGlmIGxlbihsaXZlKSAhPSBsZW4obWF0cml4KSBvciBsZW4ocG9zdCkgIT0gbGVuKG1hdHJpeCk6CiAgICAgICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYiY2FuZGlkYXRlIGRpbWVuc2lvbiBjaGFuZ2VkIGZvciBzZWVkIHtzZWVkfSIpCiAgICAgICAgZm9yIHNvdXJjZSwgKChuZWlnaGJvciwgc2xvdCwgYWxwaGEpLCBiZWZvcmUsIGFmdGVyKSBpbiBlbnVtZXJhdGUoCiAgICAgICAgICAgICAgICB6aXAobWF0cml4LCBsaXZlLCBwb3N0KSwgMSk6CiAgICAgICAgICAgIGV4cGVjdGVkID0gbGlzdChiZWZvcmUpCiAgICAgICAgICAgIGlmIG5laWdoYm9yOgogICAgICAgICAgICAgICAgaWYgYW55KHRvID09IG5laWdoYm9yIGZvciB0bywgXyBpbiBiZWZvcmUpOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYiYWN0aW9uIHdhcyBhbHJlYWR5IGluIHNvdXJjZSByb3cge3NvdXJjZX0iKQogICAgICAgICAgICAgICAgZXhwZWN0ZWQuaW5zZXJ0KG1pbihzbG90IC0gMSwgbGVuKGV4cGVjdGVkKSksIChuZWlnaGJvciwgYWxwaGEpKQogICAgICAgICAgICBpZiBhZnRlciAhPSBleHBlY3RlZDoKICAgICAgICAgICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYidW5leHBlY3RlZCBwb3N0LWFjdGlvbiBjYW5kaWRhdGUgcm93IHtzZWVkfS97c291cmNlfSIpCiAgICAgICAgc3RhdHMgPSBkaWN0KGxpbmUuc3BsaXQoKSBmb3IgbGluZSBpbiAoYmFzZSAvICJhcHBsaWVkLnN0YXRzIikucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpCiAgICAgICAgICAgICAgICAgICAgIGlmIGxlbihsaW5lLnNwbGl0KCkpID09IDIpCiAgICAgICAgc2VsZWN0ZWQgPSBzdW0obmVpZ2hib3IgIT0gMCBmb3IgbmVpZ2hib3IsIF8sIF8gaW4gbWF0cml4KQogICAgICAgIGlmIChpbnQoc3RhdHNbImFkZGVkIl0pICE9IHNlbGVjdGVkIG9yIGludChzdGF0c1siZXhpc3RpbmciXSkgIT0gMCBvcgogICAgICAgICAgICAgICAgaW50KHN0YXRzWyJyZWplY3RlZCJdKSAhPSAwKToKICAgICAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoZiJMS0ggZGlkIG5vdCBhcHBseSBleGFjdGx5IHtzZWxlY3RlZH0gbm92ZWwgYWN0aW9ucyBmb3Igc2VlZCB7c2VlZH0iKQoKCmRlZiBydW5fZ3VpZGVkX2FybSgKICAgIGtpbmQ6IHN0ciwKICAgIHBhdGNoZWRfYmluYXJ5OiBQYXRoLAogICAgcHJvYmxlbTogUGF0aCwKICAgIHh5OiBucC5uZGFycmF5LAogICAgd29ya19kaXI6IFBhdGgsCiAgICBzZWVkczogU2VxdWVuY2VbaW50XSwKICAgIGJ1ZGdldF9zOiBmbG9hdCwKICAgIG1vZGVsLAogICAgZGV2aWNlLAogICAgVCwKICAgIE0sCiAgICAqLAogICAgZnJhY3Rpb246IGZsb2F0ID0gMC4yNSwKICAgIGdyYXBoX2s6IGludCA9IDE2LAopIC0+IGRpY3Q6CiAgICBpZiBraW5kIG5vdCBpbiBHVUlERURfQVJNUyBvciBsZW4oc2VlZHMpICE9IDYgb3IgbGVuKHNldChzZWVkcykpICE9IDY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW52YWxpZCBndWlkZWQgYXJtIG9yIHNlZWQgcGFuZWwiKQogICAgeHkgPSBucC5hc2FycmF5KHh5KQogICAgY2FsbGJhY2tfbWV0YTogZGljdCA9IHt9CiAgICBwcm9wb3NlZF9hY3Rpb25zOiBsaXN0W2xpc3RbdHVwbGVbaW50LCBpbnQsIGludF1dXSA9IFtdCgogICAgZGVmIGNhbGxiYWNrKHNuYXBzaG90cyk6CiAgICAgICAgaWYgbGVuKHNuYXBzaG90cykgIT0gNjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW5jb21wbGV0ZSBmaXJzdC10b3VyIGJhcnJpZXIiKQogICAgICAgIGZpcnN0X3RvdXJzID0gW25wLmFzYXJyYXkocy5maXJzdF90b3VyLCBkdHlwZT1ucC5pbnQzMikgLSAxIGZvciBzIGluIHNuYXBzaG90c10KICAgICAgICBmaXJzdF9jb3N0cyA9IFtfdmFsaWRfY29zdChULCB4eSwgdG91cikgZm9yIHRvdXIgaW4gZmlyc3RfdG91cnNdCiAgICAgICAgZmlyc3RfaGFzaGVzID0gW2Nhbm9uaWNhbF90b3VyX2hhc2godG91cikgZm9yIHRvdXIgaW4gZmlyc3RfdG91cnNdCiAgICAgICAgbGl2ZV9oYXNoZXMgPSBbY2FuZGlkYXRlX2hhc2gocy5saXZlX2NhbmRpZGF0ZXMpIGZvciBzIGluIHNuYXBzaG90c10KICAgICAgICBuYXRpdmVfaGFzaGVzID0gW2NhbmRpZGF0ZV9oYXNoKHMubmF0aXZlX2NhbmRpZGF0ZXMpIGZvciBzIGluIHNuYXBzaG90c10KICAgICAgICBhY3Rpb25zLCBhdWRpdCA9IGNob29zZV9hY3Rpb25zKGtpbmQsIHNuYXBzaG90cywgeHksIG1vZGVsLCBkZXZpY2UsIFQsIE0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmcmFjdGlvbj1mcmFjdGlvbiwgZ3JhcGhfaz1ncmFwaF9rKQogICAgICAgIGlmIGxlbihhY3Rpb25zKSAhPSBsZW4oc2VlZHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwb2xpY3kgcmV0dXJuZWQgd3Jvbmcgc2VlZCBjb3VudCIpCiAgICAgICAgcHJvcG9zZWRfYWN0aW9uc1s6XSA9IGFjdGlvbnMKICAgICAgICBjYWxsYmFja19tZXRhLnVwZGF0ZSh7ImFybSI6IGtpbmQsICJmaXJzdF9jb3N0cyI6IGZpcnN0X2Nvc3RzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZmlyc3RfdG91cl9oYXNoZXMiOiBmaXJzdF9oYXNoZXMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJsaXZlX2NhbmRpZGF0ZV9oYXNoZXMiOiBsaXZlX2hhc2hlcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm5hdGl2ZV9jYW5kaWRhdGVfaGFzaGVzIjogbmF0aXZlX2hhc2hlcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInBvbGljeV9hdWRpdCI6IGF1ZGl0fSkKICAgICAgICBfYXRvbWljX2pzb24od29ya19kaXIgLyAicG9saWN5X21ldGEuanNvbiIsIGNhbGxiYWNrX21ldGEpCiAgICAgICAgcmV0dXJuIGFjdGlvbnMKCiAgICByZXN1bHQgPSBydW5fc2l4KFBhdGgocGF0Y2hlZF9iaW5hcnkpLCBQYXRoKHByb2JsZW0pLCBQYXRoKHdvcmtfZGlyKSwKICAgICAgICAgICAgICAgICAgICAgc2VlZHMsIGJ1ZGdldF9zLCBjYWxsYmFjaywKICAgICAgICAgICAgICAgICAgICAgZXh0cmFfcGFyYW1zPXsiTUFYX0NBTkRJREFURVMiOiA2fSkKICAgIGlmIG5vdCByZXN1bHRbImJhcnJpZXJfY29tcGxldGVkIl06CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJmaXJzdC10cmlhbCBiYXJyaWVyIGluY29tcGxldGUiKQogICAgaWYgYW55KHJ1blsiYmVzdF90b3VyIl0gaXMgTm9uZSBvciBydW5bImFwcGxpZWRfc3RhdHMiXSBpcyBOb25lIGZvciBydW4gaW4gcmVzdWx0WyJydW5zIl0pOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigib25lIG9yIG1vcmUgTEtIIHByb2Nlc3NlcyBkaWQgbm90IHJlc3VtZSIpCiAgICBfdmVyaWZ5X2FwcGxpZWRfYWN0aW9ucyh3b3JrX2Rpciwgc2VlZHMsIHByb3Bvc2VkX2FjdGlvbnMpCiAgICBzZWVkX2Nvc3RzID0gW10KICAgIGltcHJvdmVtZW50X2V2ZW50czogZGljdFtzdHIsIGxpc3RbZGljdF1dID0ge30KICAgIGZvciBzZWVkIGluIHNlZWRzOgogICAgICAgIGJhc2UgPSB3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iCiAgICAgICAgdG91ciA9IG5wLmFzYXJyYXkocmVhZF90b3VyKGJhc2UgLyAiYmVzdC50b3VyIiksIGR0eXBlPW5wLmludDMyKSAtIDEKICAgICAgICBmaW5hbF9jb3N0ID0gX3ZhbGlkX2Nvc3QoVCwgeHksIHRvdXIpCiAgICAgICAgc2VlZF9jb3N0cy5hcHBlbmQoZmluYWxfY29zdCkKICAgICAgICBldmVudF9maWxlID0gYmFzZSAvICJiZXN0LmV2ZW50cyIKICAgICAgICBpZiBub3QgZXZlbnRfZmlsZS5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYibWlzc2luZyBwYXRjaGVkIGJlc3QuZXZlbnRzIGZvciBzZWVkIHtzZWVkfSIpCiAgICAgICAgZXZlbnRzID0gW10KICAgICAgICBmb3IgbGluZSBpbiBldmVudF9maWxlLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKToKICAgICAgICAgICAgd29yZHMgPSBsaW5lLnNwbGl0KCkKICAgICAgICAgICAgaWYgbGVuKHdvcmRzKSAhPSAyOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIm1hbGZvcm1lZCBiZXN0IGV2ZW50OiB7ZXZlbnRfZmlsZX0iKQogICAgICAgICAgICBlcG9jaCwgY29zdCA9IGZsb2F0KHdvcmRzWzBdKSwgaW50KHdvcmRzWzFdKQogICAgICAgICAgICBpZiBub3QgZXZlbnRzIG9yIGNvc3QgPCBldmVudHNbLTFdWyJjb3N0Il06CiAgICAgICAgICAgICAgICBldmVudHMuYXBwZW5kKHsiZWxhcHNlZF9zIjogbWF4KDAuMCwgZXBvY2ggLSByZXN1bHRbInN0YXJ0X2Vwb2NoX3MiXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiY29zdCI6IGNvc3QsICJzb3VyY2UiOiAicGF0Y2hlZF9leGFjdF9ob29rIn0pCiAgICAgICAgaWYgbm90IGV2ZW50czoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiZW1wdHkgcGF0Y2hlZCBiZXN0LmV2ZW50cyBmb3Igc2VlZCB7c2VlZH0iKQogICAgICAgIGlmIGV2ZW50c1stMV1bImNvc3QiXSAhPSBmaW5hbF9jb3N0OgogICAgICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcihmInBhdGNoZWQgdG91ci9ldmVudCBjb3N0cyBkaXNhZ3JlZSBmb3Igc2VlZCB7c2VlZH0iKQogICAgICAgIGltcHJvdmVtZW50X2V2ZW50c1tzdHIoc2VlZCldID0gZXZlbnRzCiAgICByZXN1bHQudXBkYXRlKGFybT1raW5kLCBzZWVkX2Nvc3RzPXNlZWRfY29zdHMsCiAgICAgICAgICAgICAgICAgIGJlc3RfY29zdD1taW4oc2VlZF9jb3N0cyksCiAgICAgICAgICAgICAgICAgIGZpcnN0X2Nvc3RzPWNhbGxiYWNrX21ldGFbImZpcnN0X2Nvc3RzIl0sCiAgICAgICAgICAgICAgICAgIGZpcnN0X3RvdXJfaGFzaGVzPWNhbGxiYWNrX21ldGFbImZpcnN0X3RvdXJfaGFzaGVzIl0sCiAgICAgICAgICAgICAgICAgIGxpdmVfY2FuZGlkYXRlX2hhc2hlcz1jYWxsYmFja19tZXRhWyJsaXZlX2NhbmRpZGF0ZV9oYXNoZXMiXSwKICAgICAgICAgICAgICAgICAgbmF0aXZlX2NhbmRpZGF0ZV9oYXNoZXM9Y2FsbGJhY2tfbWV0YVsibmF0aXZlX2NhbmRpZGF0ZV9oYXNoZXMiXSwKICAgICAgICAgICAgICAgICAgcG9saWN5X2F1ZGl0PWNhbGxiYWNrX21ldGFbInBvbGljeV9hdWRpdCJdLAogICAgICAgICAgICAgICAgICBndWlkZWRfaW1wcm92ZW1lbnRfZXZlbnRzPWltcHJvdmVtZW50X2V2ZW50cywKICAgICAgICAgICAgICAgICAgYmluYXJ5X3NoYTI1Nj1zaGEyNTZfZmlsZShwYXRjaGVkX2JpbmFyeSkpCiAgICBfYXRvbWljX2pzb24od29ya19kaXIgLyAiZW5kcG9pbnQuanNvbiIsIHsKICAgICAgICBrZXk6IHJlc3VsdFtrZXldIGZvciBrZXkgaW4gKCJhcm0iLCAic2VlZF9jb3N0cyIsICJiZXN0X2Nvc3QiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZmlyc3RfY29zdHMiLCAiZmlyc3RfdG91cl9oYXNoZXMiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibGl2ZV9jYW5kaWRhdGVfaGFzaGVzIiwgIm5hdGl2ZV9jYW5kaWRhdGVfaGFzaGVzIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInBvbGljeV9hdWRpdCIsICJndWlkZWRfaW1wcm92ZW1lbnRfZXZlbnRzIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImJpbmFyeV9zaGEyNTYiKQogICAgfSkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgcnVuX25hdGl2ZV9hcm0oCiAgICBraW5kOiBzdHIsCiAgICBuYXRpdmVfYmluYXJ5OiBQYXRoLAogICAgcHJvYmxlbTogUGF0aCwKICAgIHh5OiBucC5uZGFycmF5LAogICAgd29ya19kaXI6IFBhdGgsCiAgICBzZWVkczogU2VxdWVuY2VbaW50XSwKICAgIGJ1ZGdldF9zOiBmbG9hdCwKICAgIFQsCikgLT4gZGljdDoKICAgIGlmIGtpbmQgbm90IGluIE5BVElWRV9BUk1TIG9yIGxlbihzZWVkcykgIT0gNiBvciBsZW4oc2V0KHNlZWRzKSkgIT0gNiBvciB3b3JrX2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIG5hdGl2ZSBhcm0sIHNlZWQgcGFuZWwsIG9yIHdvcmsgZGlyZWN0b3J5IikKICAgIGsgPSA2IGlmIGtpbmQgPT0gIm5hdGl2ZV9wb3A2IiBlbHNlIDcKICAgIHdvcmtfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIHByb2Nlc3NlczogbGlzdFtzdWJwcm9jZXNzLlBvcGVuXSA9IFtdCiAgICBzdHJlYW1zID0gW10KICAgIHN0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgc3RhcnRfZXBvY2ggPSB0aW1lLnRpbWUoKQogICAgZGVhZGxpbmUgPSBzdGFydCArIGJ1ZGdldF9zCiAgICBzZWVuX210aW1lX25zOiBkaWN0W2ludCwgaW50XSA9IHt9CiAgICBldmVudHM6IGRpY3Rbc3RyLCBsaXN0W2RpY3RdXSA9IHtzdHIoc2VlZCk6IFtdIGZvciBzZWVkIGluIHNlZWRzfQoKICAgIGRlZiBvYnNlcnZlX25hdGl2ZV90b3VycygpIC0+IE5vbmU6CiAgICAgICAgIyBOYXRpdmUgTEtIIHJld3JpdGVzIE9VVFBVVF9UT1VSX0ZJTEUgd2hlbmV2ZXIgaXQgZmluZHMgYSBuZXcgYmVzdC4KICAgICAgICAjIFRoZSBmaWxlc3lzdGVtIHBvbGwgdGltZXN0YW1wcyBhcmUgYXBwcm94aW1hdGUgYW5kIG1heSBtaXNzIHJhcGlkCiAgICAgICAgIyBvdmVyd3JpdHRlbiBpbmN1bWJlbnRzOyB0aGV5IGFyZSBrZXB0IHNlcGFyYXRlIGZyb20gZXhhY3QgaG9vayBsb2dzLgogICAgICAgIGZvciBzZWVkIGluIHNlZWRzOgogICAgICAgICAgICBwYXRoID0gd29ya19kaXIgLyBmInNlZWRfe3NlZWR9IiAvICJiZXN0LnRvdXIiCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHN0YW1wID0gcGF0aC5zdGF0KCkKICAgICAgICAgICAgZXhjZXB0IEZpbGVOb3RGb3VuZEVycm9yOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgc3RhbXAuc3RfbXRpbWVfbnMgPT0gc2Vlbl9tdGltZV9ucy5nZXQoc2VlZCk6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBjb3N0ID0gX3ZhbGlkX2Nvc3QoVCwgeHksIF9sb2FkX25hdGl2ZV90b3VyKFQsIHBhdGgpKQogICAgICAgICAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IsIEFzc2VydGlvbkVycm9yLCBJbmRleEVycm9yKToKICAgICAgICAgICAgICAgICMgTEtIIHdyaXRlcyB0aGlzIGZpbGUgaW4gcGxhY2UuIFJldHJ5IGF0IHRoZSBuZXh0IHBvbGwgaWYgYQogICAgICAgICAgICAgICAgIyBjb25jdXJyZW50IHJlYWQgc2F3IGFuIGluY29tcGxldGUgVFNQTElCIHRvdXIuCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBzZWVuX210aW1lX25zW3NlZWRdID0gc3RhbXAuc3RfbXRpbWVfbnMKICAgICAgICAgICAgcHJpb3IgPSBldmVudHNbc3RyKHNlZWQpXQogICAgICAgICAgICBpZiBub3QgcHJpb3Igb3IgY29zdCA8IHByaW9yWy0xXVsiY29zdCJdOgogICAgICAgICAgICAgICAgcHJpb3IuYXBwZW5kKHsiZWxhcHNlZF9zIjogbWF4KDAuMCwgbWluKGZsb2F0KGJ1ZGdldF9zKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YW1wLnN0X210aW1lIC0gc3RhcnRfZXBvY2gpKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImNvc3QiOiBjb3N0LCAic291cmNlIjogIm91dHB1dF90b3VyX210aW1lX3BvbGwifSkKICAgIHRyeToKICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICAgICAgYmFzZS5ta2RpcigpCiAgICAgICAgICAgIHBhciA9IGJhc2UgLyAicnVuLnBhciIKICAgICAgICAgICAgcGFyYW1zID0geyJQUk9CTEVNX0ZJTEUiOiBzdHIocHJvYmxlbS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICAgICAgICAgIkNBTkRJREFURV9TRVRfVFlQRSI6ICJQT1BNVVNJQyIsICJNQVhfQ0FORElEQVRFUyI6IGssCiAgICAgICAgICAgICAgICAgICAgICAiU1VCR1JBRElFTlQiOiAiTk8iLCAiU0VFRCI6IHNlZWQsICJSVU5TIjogMSwKICAgICAgICAgICAgICAgICAgICAgICJNQVhfVFJJQUxTIjogMTAwXzAwMF8wMDAsCiAgICAgICAgICAgICAgICAgICAgICAiVElNRV9MSU1JVCI6IG1heCgxMC4wLCAxMC4wICogYnVkZ2V0X3MpLAogICAgICAgICAgICAgICAgICAgICAgIlRSQUNFX0xFVkVMIjogMCwgIlNUT1BfQVRfT1BUSU1VTSI6ICJOTyIsCiAgICAgICAgICAgICAgICAgICAgICAiT1VUUFVUX1RPVVJfRklMRSI6IHN0cihiYXNlIC8gImJlc3QudG91ciIpfQogICAgICAgICAgICBULndyaXRlX3BhcihwYXIsIHBhcmFtcykKICAgICAgICAgICAgc3Rkb3V0ID0gKGJhc2UgLyAic3Rkb3V0LnR4dCIpLm9wZW4oInciKQogICAgICAgICAgICBzdGRlcnIgPSAoYmFzZSAvICJzdGRlcnIudHh0Iikub3BlbigidyIpCiAgICAgICAgICAgIHN0cmVhbXMuZXh0ZW5kKChzdGRvdXQsIHN0ZGVycikpCiAgICAgICAgICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbikKICAgICAgICAgICAgZW52LnBvcCgiQkVUQVRPVVJfSE9PS19ESVIiLCBOb25lKQogICAgICAgICAgICBwcm9jZXNzZXMuYXBwZW5kKHN1YnByb2Nlc3MuUG9wZW4oW3N0cihQYXRoKG5hdGl2ZV9iaW5hcnkpLnJlc29sdmUoKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdHIocGFyLnJlc29sdmUoKSldLCBjd2Q9YmFzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3Rkb3V0PXN0ZG91dCwgc3RkZXJyPXN0ZGVyciwgZW52PWVudikpCiAgICAgICAgd2hpbGUgYW55KHByb2Nlc3MucG9sbCgpIGlzIE5vbmUgZm9yIHByb2Nlc3MgaW4gcHJvY2Vzc2VzKSBhbmQgdGltZS5tb25vdG9uaWMoKSA8IGRlYWRsaW5lOgogICAgICAgICAgICBvYnNlcnZlX25hdGl2ZV90b3VycygpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoMC4wMikKICAgIGZpbmFsbHk6CiAgICAgICAgZm9yIHByb2Nlc3MgaW4gcHJvY2Vzc2VzOgogICAgICAgICAgICBpZiBwcm9jZXNzLnBvbGwoKSBpcyBOb25lOgogICAgICAgICAgICAgICAgcHJvY2Vzcy50ZXJtaW5hdGUoKQogICAgICAgIGZvciBwcm9jZXNzIGluIHByb2Nlc3NlczoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KHRpbWVvdXQ9MikKICAgICAgICAgICAgZXhjZXB0IHN1YnByb2Nlc3MuVGltZW91dEV4cGlyZWQ6CiAgICAgICAgICAgICAgICBwcm9jZXNzLmtpbGwoKQogICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KCkKICAgICAgICBmb3Igc3RyZWFtIGluIHN0cmVhbXM6CiAgICAgICAgICAgIHN0cmVhbS5jbG9zZSgpCiAgICBvYnNlcnZlX25hdGl2ZV90b3VycygpCiAgICBzZWVkX2Nvc3RzID0gW10KICAgIGZvciBzZWVkIGluIHNlZWRzOgogICAgICAgIHRvdXJfcGF0aCA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIgLyAiYmVzdC50b3VyIgogICAgICAgIGlmIG5vdCB0b3VyX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIm5hdGl2ZSBQT1B7a30geWllbGRlZCBubyB0b3VyIGZvciBzZWVkIHtzZWVkfSIpCiAgICAgICAgc2VlZF9jb3N0cy5hcHBlbmQoX3ZhbGlkX2Nvc3QoVCwgeHksIF9sb2FkX25hdGl2ZV90b3VyKFQsIHRvdXJfcGF0aCkpKQogICAgcmVzdWx0ID0geyJhcm0iOiBraW5kLCAic2VlZHMiOiBsaXN0KHNlZWRzKSwgIndhbGxfYnVkZ2V0X3MiOiBidWRnZXRfcywKICAgICAgICAgICAgICAid2FsbF9lbGFwc2VkX3MiOiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnQsCiAgICAgICAgICAgICAgInN0YXJ0X2Vwb2NoX3MiOiBzdGFydF9lcG9jaCwKICAgICAgICAgICAgICAic2VlZF9jb3N0cyI6IHNlZWRfY29zdHMsICJiZXN0X2Nvc3QiOiBtaW4oc2VlZF9jb3N0cyksCiAgICAgICAgICAgICAgIm5hdGl2ZV9pbXByb3ZlbWVudF9ldmVudHMiOiBldmVudHMsCiAgICAgICAgICAgICAgImJpbmFyeV9zaGEyNTYiOiBzaGEyNTZfZmlsZShuYXRpdmVfYmluYXJ5KX0KICAgIF9hdG9taWNfanNvbih3b3JrX2RpciAvICJlbmRwb2ludC5qc29uIiwgcmVzdWx0KQogICAgcmV0dXJuIHJlc3VsdAo=', 'v11_candidate/run_cell.py': 'IiIidjExIENvbGFiIGV4ZWN1dGlvbiBjZWxsLiBDb21wbGV0ZWQgYXJtcyBhcmUgaW1tdXRhYmxlIGFuZCByZXN1bWFibGUuIiIiCgoKZGVmIGFybV9yZWNvcmRfcGF0aChpbnN0LCBhcm0pOgogICAgcmV0dXJuIE9VVFBVVF9ST09UIC8gImluc3RhbmNlcyIgLyBpbnN0WyJuYW1lIl0gLyBhcm0gLyAicmVzdWx0Lmpzb24iCgoKZGVmIHZhbGlkYXRlX3NhdmVkKGluc3QsIGFybSwgcmVjb3JkKToKICAgIGFzc2VydCByZWNvcmRbImNvbXBsZXRlIl0gYW5kIHJlY29yZFsiYXJtIl0gPT0gYXJtIGFuZCByZWNvcmRbImluc3RhbmNlIl0gPT0gaW5zdFsibmFtZSJdCiAgICBhc3NlcnQgcmVjb3JkWyJ4eV9zaGEyNTYiXSA9PSBpbnN0WyJ4eV9zaGEyNTYiXSBhbmQgcmVjb3JkWyJzZWVkcyJdID09IFNFRURTCiAgICBhc3NlcnQgcmVjb3JkWyJidWRnZXRfcyJdID09IGluc3RbImJ1ZGdldF9zIl0gYW5kIHJlY29yZFsiYnVpbGQiXSA9PSBCVUlMRF9JTkZPCiAgICBhc3NlcnQgcmVjb3JkWyJjb2RlX3NoYTI1NiJdID09IENPREVfU0hBMjU2CiAgICBhc3NlcnQgcmVjb3JkWyJmcmFjdGlvbiJdID09IFNQQVJTRV9GUkFDVElPTiBhbmQgcmVjb3JkWyJncmFwaF9rIl0gPT0gR1JBUEhfSwogICAgYXNzZXJ0IGxlbihyZWNvcmRbInNlZWRfY29zdHMiXSkgPT0gbGVuKFNFRURTKQogICAgYXNzZXJ0IG1pbihyZWNvcmRbInNlZWRfY29zdHMiXSkgPT0gcmVjb3JkWyJiZXN0X2Nvc3QiXQogICAgaWYgYXJtIG5vdCBpbiAoIm5hdGl2ZV9wb3A2IiwgIm5hdGl2ZV9wb3A3Iik6CiAgICAgICAgYXNzZXJ0IGxlbihyZWNvcmRbImZpcnN0X3RvdXJfaGFzaGVzIl0pID09IGxlbihTRUVEUykKICAgICAgICBhc3NlcnQgbGVuKHJlY29yZFsibGl2ZV9jYW5kaWRhdGVfaGFzaGVzIl0pID09IGxlbihTRUVEUykKICAgICAgICBhc3NlcnQgbGVuKHJlY29yZFsibmF0aXZlX2NhbmRpZGF0ZV9oYXNoZXMiXSkgPT0gbGVuKFNFRURTKQogICAgICAgIGF1ZGl0ID0gcmVjb3JkWyJwb2xpY3lfYXVkaXQiXQogICAgICAgIGFzc2VydCBsZW4oYXVkaXRbImdhdGVfc2hhMjU2X3Blcl9zZWVkIl0pID09IGxlbihTRUVEUykKICAgICAgICBhc3NlcnQgYXVkaXRbInNlZWRfaWRzIl0gPT0gU0VFRFMKICAgICAgICBpZiBhcm0gPT0gInNoYW0iOgogICAgICAgICAgICBhc3NlcnQgYXVkaXRbInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiXSA9PSBbMF0gKiBsZW4oU0VFRFMpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYXNzZXJ0IGF1ZGl0WyJzZWxlY3RlZF9hY3Rpb25zX3Blcl9zZWVkIl0gPT0gYXVkaXRbImd1aWRlZF9hY3Rpb25zX3Blcl9zZWVkIl0KICAgICAgICBhc3NlcnQgYXVkaXRbInNlbGVjdGVkX3VuaXF1ZV91bmRpcmVjdGVkX2VkZ2VzX3Blcl9zZWVkIl0gPT0gYXVkaXRbInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiXQogICAgcmV0dXJuIHJlY29yZAoKCmRlZiBzYXZlX3Byb2dyZXNzKHN0YXRlPSJydW5uaW5nIiwgbGFzdD1Ob25lKToKICAgIGNvbXBsZXRlZCA9IHN1bShhcm1fcmVjb3JkX3BhdGgoaW5zdCwgYXJtKS5leGlzdHMoKSBmb3IgaW5zdCBpbiBJTlNUQU5DRVMgZm9yIGFybSBpbiBBUk1TKQogICAgd3JpdGVfanNvbl9hdG9taWMoT1VUUFVUX1JPT1QgLyAicHJvZ3Jlc3MuanNvbiIsCiAgICAgICAgICAgICAgICAgICAgICB7InN0YXRlIjogc3RhdGUsICJjb21wbGV0ZWRfYXJtcyI6IGNvbXBsZXRlZCwKICAgICAgICAgICAgICAgICAgICAgICAidG90YWxfYXJtcyI6IGxlbihJTlNUQU5DRVMpICogbGVuKEFSTVMpLCAibGFzdCI6IGxhc3QsCiAgICAgICAgICAgICAgICAgICAgICAgInVwZGF0ZWRfdXRjIjogdGltZS5zdHJmdGltZSgiJVktJW0tJWRUJUg6JU06JVNaIiwgdGltZS5nbXRpbWUoKSl9KQoKCm5ld19hcm1zID0gMApzYXZlX3Byb2dyZXNzKCkKZm9yIGluc3QgaW4gSU5TVEFOQ0VTOgogICAgaW5zdGFuY2VfZGlyID0gT1VUUFVUX1JPT1QgLyAiaW5zdGFuY2VzIiAvIGluc3RbIm5hbWUiXQogICAgaW5zdGFuY2VfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGNvb3JkX3BhdGggPSBpbnN0YW5jZV9kaXIgLyAieHkubnB6IgogICAgaWYgbm90IGNvb3JkX3BhdGguZXhpc3RzKCk6CiAgICAgICAgdGVtcCA9IGluc3RhbmNlX2RpciAvICJ4eS50bXAubnB6IgogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQodGVtcCwgeHk9aW5zdFsieHkiXSkKICAgICAgICBvcy5yZXBsYWNlKHRlbXAsIGNvb3JkX3BhdGgpCiAgICB3aXRoIG5wLmxvYWQoY29vcmRfcGF0aCwgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBzYXZlZF94eToKICAgICAgICBhc3NlcnQgaGFzaGxpYi5zaGEyNTYobnAuYXNjb250aWd1b3VzYXJyYXkoc2F2ZWRfeHlbInh5Il0pLnRvYnl0ZXMoKSkuaGV4ZGlnZXN0KCkgPT0gaW5zdFsieHlfc2hhMjU2Il0KICAgIGxvY2FsX3Byb2JsZW0gPSBXT1JLX1JPT1QgLyAicHJvYmxlbXMiIC8gKGluc3RbIm5hbWUiXSArICIudHNwIikKICAgIGxvY2FsX3Byb2JsZW0ucGFyZW50Lm1rZGlyKGV4aXN0X29rPVRydWUpCiAgICBULndyaXRlX3RzcChsb2NhbF9wcm9ibGVtLCBpbnN0WyJ4eSJdKQogICAgZm9yIGFybSBpbiBhcm1fb3JkZXIoaW5zdFsibmFtZSJdKToKICAgICAgICByZWNvcmRfcGF0aCA9IGFybV9yZWNvcmRfcGF0aChpbnN0LCBhcm0pCiAgICAgICAgaWYgcmVjb3JkX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHZhbGlkYXRlX3NhdmVkKGluc3QsIGFybSwganNvbi5sb2FkcyhyZWNvcmRfcGF0aC5yZWFkX3RleHQoKSkpCiAgICAgICAgICAgIHByaW50KCJBVExBOiIsIGluc3RbIm5hbWUiXSwgYXJtLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIE1BWF9ORVdfQVJNUyBpcyBub3QgTm9uZSBhbmQgbmV3X2FybXMgPj0gTUFYX05FV19BUk1TOgogICAgICAgICAgICBicmVhawogICAgICAgIGF0dGVtcHRfaWQgPSB1dWlkLnV1aWQ0KCkuaGV4CiAgICAgICAgbG9jYWxfcnVuID0gV09SS19ST09UIC8gInJ1bnMiIC8gaW5zdFsibmFtZSJdIC8gYXJtIC8gYXR0ZW1wdF9pZAogICAgICAgIGxvY2FsX3J1bi5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHRyeToKICAgICAgICAgICAgcHJpbnQoIkJBxZ5MQToiLCBpbnN0WyJuYW1lIl0sIGFybSwgImLDvHTDp2UiLCBpbnN0WyJidWRnZXRfcyJdLCAic24iLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBpZiBhcm0uc3RhcnRzd2l0aCgibmF0aXZlXyIpOgogICAgICAgICAgICAgICAgcmF3ID0gcnVuX25hdGl2ZV9hcm0oYXJtLCBOQVRJVkVfTEtILCBsb2NhbF9wcm9ibGVtLCBpbnN0WyJ4eSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbG9jYWxfcnVuLCBTRUVEUywgaW5zdFsiYnVkZ2V0X3MiXSwgVCkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJhdyA9IHJ1bl9ndWlkZWRfYXJtKGFybSwgUEFUQ0hFRF9MS0gsIGxvY2FsX3Byb2JsZW0sIGluc3RbInh5Il0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBsb2NhbF9ydW4sIFNFRURTLCBpbnN0WyJidWRnZXRfcyJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgTU9ERUwsIERFVklDRSwgVCwgTSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZyYWN0aW9uPVNQQVJTRV9GUkFDVElPTiwgZ3JhcGhfaz1HUkFQSF9LKQogICAgICAgICAgICBhdWRpdF9yZWwgPSBmImluc3RhbmNlcy97aW5zdFsnbmFtZSddfS97YXJtfS97YXR0ZW1wdF9pZH0iCiAgICAgICAgICAgIGF1ZGl0X2Rlc3QgPSBPVVRQVVRfUk9PVCAvIGF1ZGl0X3JlbAogICAgICAgICAgICBzaHV0aWwuY29weXRyZWUobG9jYWxfcnVuLCBhdWRpdF9kZXN0KQogICAgICAgICAgICByZWNvcmQgPSB7InNjaGVtYSI6IDEsICJjb21wbGV0ZSI6IFRydWUsICJpbnN0YW5jZSI6IGluc3RbIm5hbWUiXSwKICAgICAgICAgICAgICAgICAgICAgICJncm91cCI6IGluc3RbImdyb3VwIl0sICJuIjogaW5zdFsibiJdLAogICAgICAgICAgICAgICAgICAgICAgInh5X3NoYTI1NiI6IGluc3RbInh5X3NoYTI1NiJdLCAiYXJtIjogYXJtLAogICAgICAgICAgICAgICAgICAgICAgInNlZWRzIjogU0VFRFMsICJidWRnZXRfcyI6IGluc3RbImJ1ZGdldF9zIl0sCiAgICAgICAgICAgICAgICAgICAgICAic2VlZF9jb3N0cyI6IHJhd1sic2VlZF9jb3N0cyJdLCAiYmVzdF9jb3N0IjogcmF3WyJiZXN0X2Nvc3QiXSwKICAgICAgICAgICAgICAgICAgICAgICJ3YWxsX2VsYXBzZWRfcyI6IHJhd1sid2FsbF9lbGFwc2VkX3MiXSwKICAgICAgICAgICAgICAgICAgICAgICJzdGFydF9lcG9jaF9zIjogcmF3WyJzdGFydF9lcG9jaF9zIl0sCiAgICAgICAgICAgICAgICAgICAgICAiZGVhZGxpbmVfZXBvY2hfcyI6IHJhd1siZGVhZGxpbmVfZXBvY2hfcyJdIGlmICJkZWFkbGluZV9lcG9jaF9zIiBpbiByYXcgZWxzZSByYXdbInN0YXJ0X2Vwb2NoX3MiXSArIGluc3RbImJ1ZGdldF9zIl0sCiAgICAgICAgICAgICAgICAgICAgICAiZmlyc3RfY29zdHMiOiByYXcuZ2V0KCJmaXJzdF9jb3N0cyIpLAogICAgICAgICAgICAgICAgICAgICAgImZpcnN0X3RvdXJfaGFzaGVzIjogcmF3LmdldCgiZmlyc3RfdG91cl9oYXNoZXMiKSwKICAgICAgICAgICAgICAgICAgICAgICJsaXZlX2NhbmRpZGF0ZV9oYXNoZXMiOiByYXcuZ2V0KCJsaXZlX2NhbmRpZGF0ZV9oYXNoZXMiKSwKICAgICAgICAgICAgICAgICAgICAgICJuYXRpdmVfY2FuZGlkYXRlX2hhc2hlcyI6IHJhdy5nZXQoIm5hdGl2ZV9jYW5kaWRhdGVfaGFzaGVzIiksCiAgICAgICAgICAgICAgICAgICAgICAicG9saWN5X2F1ZGl0IjogcmF3LmdldCgicG9saWN5X2F1ZGl0IiksCiAgICAgICAgICAgICAgICAgICAgICAiZ3VpZGVkX2ltcHJvdmVtZW50X2V2ZW50cyI6IHJhdy5nZXQoImd1aWRlZF9pbXByb3ZlbWVudF9ldmVudHMiKSwKICAgICAgICAgICAgICAgICAgICAgICJuYXRpdmVfaW1wcm92ZW1lbnRfZXZlbnRzIjogcmF3LmdldCgibmF0aXZlX2ltcHJvdmVtZW50X2V2ZW50cyIpLAogICAgICAgICAgICAgICAgICAgICAgImZyYWN0aW9uIjogU1BBUlNFX0ZSQUNUSU9OLCAiZ3JhcGhfayI6IEdSQVBIX0ssCiAgICAgICAgICAgICAgICAgICAgICAiYnVpbGQiOiBCVUlMRF9JTkZPLCAiY29kZV9zaGEyNTYiOiBDT0RFX1NIQTI1NiwKICAgICAgICAgICAgICAgICAgICAgICJoYXJkd2FyZSI6IEhBUkRXQVJFLCAiYXVkaXRfcmVscGF0aCI6IGF1ZGl0X3JlbCwKICAgICAgICAgICAgICAgICAgICAgICJhdHRlbXB0X2lkIjogYXR0ZW1wdF9pZCwKICAgICAgICAgICAgICAgICAgICAgICJjb21wbGV0ZWRfdXRjIjogdGltZS5zdHJmdGltZSgiJVktJW0tJWRUJUg6JU06JVNaIiwgdGltZS5nbXRpbWUoKSl9CiAgICAgICAgICAgIHZhbGlkYXRlX3NhdmVkKGluc3QsIGFybSwgcmVjb3JkKQogICAgICAgICAgICB3cml0ZV9qc29uX2F0b21pYyhyZWNvcmRfcGF0aCwgcmVjb3JkKQogICAgICAgICAgICBuZXdfYXJtcyArPSAxCiAgICAgICAgICAgIHNhdmVfcHJvZ3Jlc3MobGFzdD1pbnN0WyJuYW1lIl0gKyAiLyIgKyBhcm0pCiAgICAgICAgICAgIHByaW50KCJCxLBUVMSwOiIsIGluc3RbIm5hbWUiXSwgYXJtLCAiZW4gaXlpIiwgcmF3WyJiZXN0X2Nvc3QiXSwgZmx1c2g9VHJ1ZSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgd3JpdGVfanNvbl9hdG9taWMoT1VUUFVUX1JPT1QgLyAiZXJyb3JzIiAvCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIChpbnN0WyJuYW1lIl0gKyAiXyIgKyBhcm0gKyAiXyIgKyBhdHRlbXB0X2lkICsgIi5qc29uIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHsiaW5zdGFuY2UiOiBpbnN0WyJuYW1lIl0sICJhcm0iOiBhcm0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZXJyb3IiOiByZXByKGV4YyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAidXRjIjogdGltZS5zdHJmdGltZSgiJVktJW0tJWRUJUg6JU06JVNaIiwgdGltZS5nbXRpbWUoKSl9KQogICAgICAgICAgICBzYXZlX3Byb2dyZXNzKHN0YXRlPSJlcnJvciIsIGxhc3Q9aW5zdFsibmFtZSJdICsgIi8iICsgYXJtKQogICAgICAgICAgICByYWlzZQogICAgaWYgTUFYX05FV19BUk1TIGlzIG5vdCBOb25lIGFuZCBuZXdfYXJtcyA+PSBNQVhfTkVXX0FSTVM6CiAgICAgICAgYnJlYWsKCmNvbXBsZXRlZF9jb3VudCA9IHN1bShhcm1fcmVjb3JkX3BhdGgoaW5zdCwgYXJtKS5leGlzdHMoKSBmb3IgaW5zdCBpbiBJTlNUQU5DRVMgZm9yIGFybSBpbiBBUk1TKQpzYXZlX3Byb2dyZXNzKHN0YXRlPSJjb21wbGV0ZSIgaWYgY29tcGxldGVkX2NvdW50ID09IGxlbihJTlNUQU5DRVMpICogbGVuKEFSTVMpIGVsc2UgInBhcnRpYWwiKQpwcmludChmIlRhbWFtbGFuYW4ga29sOiB7Y29tcGxldGVkX2NvdW50fS97bGVuKElOU1RBTkNFUykgKiBsZW4oQVJNUyl9LiAiCiAgICAgIGYiQnUgb3R1cnVtZGFraSB5ZW5pIGtvbDoge25ld19hcm1zfSIpCg==', 'v11_candidate/analysis_cell.py': 'IiIidjExIENvbGFiIHN1bW1hcnkgY2VsbDogbWVjaGFuaXNtIG1lYXN1cmVtZW50cyBhbmQgZ3VhcmRlZCBjb21wYXJpc29ucy4iIiIKCgpkZWYgcmVhZF9rZXlfdmFsdWVfbWV0cmljcyhwYXRoKToKICAgIHZhbHVlcyA9IHt9CiAgICBmb3IgbGluZSBpbiBQYXRoKHBhdGgpLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKToKICAgICAgICB3b3JkcyA9IGxpbmUuc3BsaXQoKQogICAgICAgIGlmIGxlbih3b3JkcykgIT0gMiBvciB3b3Jkc1swXSBpbiB2YWx1ZXM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJCb3p1ayB0ZWxlbWV0cmkgc2F0xLFyxLE6IHtwYXRofToge2xpbmUhcn0iKQogICAgICAgIHZhbHVlc1t3b3Jkc1swXV0gPSBmbG9hdCh3b3Jkc1sxXSkgaWYgIi4iIGluIHdvcmRzWzFdIGVsc2UgaW50KHdvcmRzWzFdKQogICAgcmV0dXJuIHZhbHVlcwoKCmRlZiBjb21wYXJpc29uX3N1bW1hcnkoZnJhbWUsIHRyZWF0bWVudCwgY29udHJvbCwgYmFzaXMpOgogICAgaWYgZnJhbWUuZW1wdHk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJlbCA9IDEwMC4wICogKGZyYW1lW3RyZWF0bWVudF0udG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkgLQogICAgICAgICAgICAgICAgICAgZnJhbWVbY29udHJvbF0udG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkpIC8gZnJhbWVbY29udHJvbF0udG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkKICAgIHJldHVybiB7InRyZWF0bWVudCI6IHRyZWF0bWVudCwgImNvbnRyb2wiOiBjb250cm9sLCAiYmFzaXMiOiBiYXNpcywKICAgICAgICAgICAgIm5faW5zdGFuY2VzIjogbGVuKHJlbCksICJ3aW5zIjogaW50KChyZWwgPCAwKS5zdW0oKSksCiAgICAgICAgICAgICJsb3NzZXMiOiBpbnQoKHJlbCA+IDApLnN1bSgpKSwgInRpZXMiOiBpbnQoKHJlbCA9PSAwKS5zdW0oKSksCiAgICAgICAgICAgICJtZWFuX3JlbGF0aXZlX3BjdCI6IGZsb2F0KHJlbC5tZWFuKCkpLAogICAgICAgICAgICAibWVkaWFuX3JlbGF0aXZlX3BjdCI6IGZsb2F0KG5wLm1lZGlhbihyZWwpKX0KCgpwYXRjaGVkX2FybXMgPSBbInNoYW0iLCAicmFuZG9tX2FkZCIsICJmcmVxdWVuY3lfYWRkIiwgImJldGFfYWRkIl0KZ3VpZGVkX2FybXMgPSBbInJhbmRvbV9hZGQiLCAiZnJlcXVlbmN5X2FkZCIsICJiZXRhX2FkZCJdCnJvd3MsIG1lY2hhbmlzbV9yb3dzLCB0YXJnZXRfcm93cywgZnJlcXVlbmN5X3Jvd3MgPSBbXSwgW10sIFtdLCBbXQpzdW1tYXJ5X2RpciA9IE9VVFBVVF9ST09UIC8gInN1bW1hcnkiCnN1bW1hcnlfZGlyLm1rZGlyKGV4aXN0X29rPVRydWUpCgpmb3IgaW5zdCBpbiBJTlNUQU5DRVM6CiAgICByZWNvcmRzID0ge30KICAgIGZvciBhcm0gaW4gQVJNUzoKICAgICAgICBwYXRoID0gYXJtX3JlY29yZF9wYXRoKGluc3QsIGFybSkKICAgICAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgICAgICByZWNvcmRzW2FybV0gPSB2YWxpZGF0ZV9zYXZlZChpbnN0LCBhcm0sIGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoKSkpCiAgICBpZiBsZW4ocmVjb3JkcykgIT0gbGVuKEFSTVMpOgogICAgICAgIGNvbnRpbnVlCgogICAgcm93ID0geyJpbnN0YW5jZSI6IGluc3RbIm5hbWUiXSwgImdyb3VwIjogaW5zdFsiZ3JvdXAiXSwKICAgICAgICAgICAibiI6IGluc3RbIm4iXSwgImJ1ZGdldF9zIjogaW5zdFsiYnVkZ2V0X3MiXSwKICAgICAgICAgICAibWl4ZWRfaGFyZHdhcmUiOiBsZW4oe2pzb24uZHVtcHMocmVjb3Jkc1thcm1dWyJoYXJkd2FyZSJdLCBzb3J0X2tleXM9VHJ1ZSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgYXJtIGluIEFSTVN9KSA+IDF9CiAgICBmb3IgYXJtIGluIEFSTVM6CiAgICAgICAgcm93W2FybV0gPSByZWNvcmRzW2FybV1bImJlc3RfY29zdCJdCiAgICByb3dbImZpcnN0X3RvdXJzX21hdGNoIl0gPSBhbGwoCiAgICAgICAgbGVuKHtyZWNvcmRzW2FybV1bImZpcnN0X3RvdXJfaGFzaGVzIl1baV0gZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9KSA9PSAxCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKFNFRURTKSkpCiAgICByb3dbInJhd19maXJzdF90b3Vyc19tYXRjaCJdID0gYWxsKAogICAgICAgIGxlbih7cmVjb3Jkc1thcm1dWyJwb2xpY3lfYXVkaXQiXVsiZmlyc3RfdG91cl9yYXdfc2hhMjU2Il1baV0KICAgICAgICAgICAgIGZvciBhcm0gaW4gcGF0Y2hlZF9hcm1zfSkgPT0gMSBmb3IgaSBpbiByYW5nZShsZW4oU0VFRFMpKSkKICAgIHJvd1sibmF0aXZlX2NhbmRpZGF0ZXNfbWF0Y2giXSA9IGFsbCgKICAgICAgICBsZW4oe3JlY29yZHNbYXJtXVsibmF0aXZlX2NhbmRpZGF0ZV9oYXNoZXMiXVtpXSBmb3IgYXJtIGluIHBhdGNoZWRfYXJtc30pID09IDEKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oU0VFRFMpKSkKICAgIHJvd1sibGl2ZV9jYW5kaWRhdGVzX21hdGNoIl0gPSBhbGwoCiAgICAgICAgbGVuKHtyZWNvcmRzW2FybV1bImxpdmVfY2FuZGlkYXRlX2hhc2hlcyJdW2ldIGZvciBhcm0gaW4gcGF0Y2hlZF9hcm1zfSkgPT0gMQogICAgICAgIGZvciBpIGluIHJhbmdlKGxlbihTRUVEUykpKQogICAgcm93WyJnYXRlc19tYXRjaCJdID0gYWxsKAogICAgICAgIGxlbih7cmVjb3Jkc1thcm1dWyJwb2xpY3lfYXVkaXQiXVsiZ2F0ZV9zaGEyNTZfcGVyX3NlZWQiXVtpXQogICAgICAgICAgICAgZm9yIGFybSBpbiBwYXRjaGVkX2FybXN9KSA9PSAxIGZvciBpIGluIHJhbmdlKGxlbihTRUVEUykpKQogICAgcm93WyJtb2RlbF9wcmVkaWN0aW9uc19tYXRjaCJdID0gbGVuKHsKICAgICAgICByZWNvcmRzW2FybV1bInBvbGljeV9hdWRpdCJdWyJtb2RlbF9wcm9iYWJpbGl0eV9zaGEyNTYiXQogICAgICAgIGZvciBhcm0gaW4gcGF0Y2hlZF9hcm1zfSkgPT0gMQogICAgcm93WyJwbGFubmVkX2FjdGlvbnNfbWF0Y2giXSA9IGxlbih7CiAgICAgICAganNvbi5kdW1wcyhyZWNvcmRzW2FybV1bInBvbGljeV9hdWRpdCJdWyJhbGxfZ3VpZGVkX2FjdGlvbl9zaGEyNTZfcGVyX3NlZWQiXSwKICAgICAgICAgICAgICAgICAgIHNvcnRfa2V5cz1UcnVlKSBmb3IgYXJtIGluIHBhdGNoZWRfYXJtc30pID09IDEKICAgIHJvd1siY2FwYWNpdGllc19tYXRjaCJdID0gYWxsKAogICAgICAgIGxlbih7cmVjb3Jkc1thcm1dWyJwb2xpY3lfYXVkaXQiXVsic2VsZWN0ZWRfYWN0aW9uc19wZXJfc2VlZCJdW2ldCiAgICAgICAgICAgICBmb3IgYXJtIGluIGd1aWRlZF9hcm1zfSkgPT0gMSBmb3IgaSBpbiByYW5nZShsZW4oU0VFRFMpKSkKICAgIHJvd1sidW5kaXJlY3RlZF9jYXBhY2l0aWVzX21hdGNoIl0gPSBhbGwoCiAgICAgICAgbGVuKHtyZWNvcmRzW2FybV1bInBvbGljeV9hdWRpdCJdWyJzZWxlY3RlZF91bmlxdWVfdW5kaXJlY3RlZF9lZGdlc19wZXJfc2VlZCJdW2ldCiAgICAgICAgICAgICBmb3IgYXJtIGluIGd1aWRlZF9hcm1zfSkgPT0gMSBmb3IgaSBpbiByYW5nZShsZW4oU0VFRFMpKSkKICAgIHJvd1sibWF0Y2hlZF9wb2xpY3lfc3RhdGUiXSA9IGFsbChyb3dba2V5XSBmb3Iga2V5IGluCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKCJmaXJzdF90b3Vyc19tYXRjaCIsICJyYXdfZmlyc3RfdG91cnNfbWF0Y2giLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibmF0aXZlX2NhbmRpZGF0ZXNfbWF0Y2giLCAibGl2ZV9jYW5kaWRhdGVzX21hdGNoIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1vZGVsX3ByZWRpY3Rpb25zX21hdGNoIiwgInBsYW5uZWRfYWN0aW9uc19tYXRjaCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJnYXRlc19tYXRjaCIsICJjYXBhY2l0aWVzX21hdGNoIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInVuZGlyZWN0ZWRfY2FwYWNpdGllc19tYXRjaCIpKQogICAgcm93cy5hcHBlbmQocm93KQoKICAgIGZvciBhcm0gaW4gcGF0Y2hlZF9hcm1zOgogICAgICAgIHJlY29yZCA9IHJlY29yZHNbYXJtXQogICAgICAgIGZvciBpbmRleCwgc2VlZCBpbiBlbnVtZXJhdGUoU0VFRFMpOgogICAgICAgICAgICBiYXNlID0gT1VUUFVUX1JPT1QgLyByZWNvcmRbImF1ZGl0X3JlbHBhdGgiXSAvIGYic2VlZF97c2VlZH0iCiAgICAgICAgICAgIG1ldHJpY3MgPSByZWFkX2tleV92YWx1ZV9tZXRyaWNzKGJhc2UgLyAidGVsZW1ldHJ5LnR4dCIpCiAgICAgICAgICAgIHNlbGVjdGVkID0gaW50KHJlY29yZFsicG9saWN5X2F1ZGl0Il1bInNlbGVjdGVkX2FjdGlvbnNfcGVyX3NlZWQiXVtpbmRleF0pCiAgICAgICAgICAgIGlmIGludChtZXRyaWNzWyJhY3Rpb25fY291bnQiXSkgIT0gc2VsZWN0ZWQ6CiAgICAgICAgICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcihmIlNlw6dpbGVuL3V5Z3VsYW5hbiBrZW5hciBzYXnEsXPEsSBmYXJrbMSxOiB7aW5zdFsnbmFtZSddfS97YXJtfS97c2VlZH0iKQogICAgICAgICAgICBtZWNoYW5pc21fcm93cy5hcHBlbmQoeyJpbnN0YW5jZSI6IGluc3RbIm5hbWUiXSwgImFybSI6IGFybSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAic2VlZCI6IHNlZWQsICJzZWxlY3RlZF9hY3Rpb25zIjogc2VsZWN0ZWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRlbGVtZXRyeV9hZ2VfYXRfZW5kX3MiOiBtYXgoMC4wLCBtaW4oCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZsb2F0KHJlY29yZFsiZGVhZGxpbmVfZXBvY2hfcyJdKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZmxvYXQocmVjb3JkWyJzdGFydF9lcG9jaF9zIl0pICsgZmxvYXQocmVjb3JkWyJ3YWxsX2VsYXBzZWRfcyJdKSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgLSBmbG9hdChtZXRyaWNzWyJ0ZWxlbWV0cnlfc25hcHNob3RfZXBvY2hfcyJdKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKiptZXRyaWNzfSkKCiAgICBmcmVxdWVuY3lfYXVkaXQgPSByZWNvcmRzWyJiZXRhX2FkZCJdWyJwb2xpY3lfYXVkaXQiXQogICAgZm9yIGFybSBpbiBndWlkZWRfYXJtczoKICAgICAgICBmb3IgaW5kZXgsIHNlZWQgaW4gZW51bWVyYXRlKFNFRURTKToKICAgICAgICAgICAgdmFsdWVzID0gZnJlcXVlbmN5X2F1ZGl0WyJhbGxfZ3VpZGVkX3NlbGVjdGVkX2Vuc2VtYmxlX2ZyZXF1ZW5jeV9wZXJfc2VlZCJdW2FybV1baW5kZXhdCiAgICAgICAgICAgIGZyZXF1ZW5jeV9yb3dzLmFwcGVuZCh7Imluc3RhbmNlIjogaW5zdFsibmFtZSJdLCAiYXJtIjogYXJtLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzZWVkIjogc2VlZCwgInNlbGVjdGVkX2FjdGlvbnMiOiBsZW4odmFsdWVzKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiemVyb19mcmVxdWVuY3lfYWN0aW9ucyI6IHN1bSh2YWx1ZSA9PSAwIGZvciB2YWx1ZSBpbiB2YWx1ZXMpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJtZWFuX2Vuc2VtYmxlX2ZyZXF1ZW5jeSI6IGZsb2F0KG5wLm1lYW4odmFsdWVzKSkgaWYgdmFsdWVzIGVsc2UgbnAubmFufSkKCiAgICB0YXJnZXQgPSBUQVJHRVRfQ09TVFNbaW5zdFsibmFtZSJdXQogICAgZm9yIGFybSBpbiBBUk1TOgogICAgICAgIHJlY29yZCA9IHJlY29yZHNbYXJtXQogICAgICAgIGV2ZW50X21hcCA9IChyZWNvcmRbIm5hdGl2ZV9pbXByb3ZlbWVudF9ldmVudHMiXSBpZiBhcm0uc3RhcnRzd2l0aCgibmF0aXZlXyIpCiAgICAgICAgICAgICAgICAgICAgIGVsc2UgcmVjb3JkWyJndWlkZWRfaW1wcm92ZW1lbnRfZXZlbnRzIl0pCiAgICAgICAgb2JzZXJ2ZWQgPSBbZmxvYXQoZXZlbnRbImVsYXBzZWRfcyJdKQogICAgICAgICAgICAgICAgICAgIGZvciBzZWVkIGluIFNFRURTIGZvciBldmVudCBpbiBldmVudF9tYXBbc3RyKHNlZWQpXQogICAgICAgICAgICAgICAgICAgIGlmIGludChldmVudFsiY29zdCJdKSA8PSB0YXJnZXRdCiAgICAgICAgdGFyZ2V0X3Jvd3MuYXBwZW5kKHsiaW5zdGFuY2UiOiBpbnN0WyJuYW1lIl0sICJhcm0iOiBhcm0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAidGFyZ2V0X2Nvc3RfZnJvbV92OV9uYXRpdmVfcG9wNiI6IHRhcmdldCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJyZWFjaGVkIjogYm9vbChvYnNlcnZlZCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZWFybGllc3RfZWxhcHNlZF9zIjogbWluKG9ic2VydmVkKSBpZiBvYnNlcnZlZCBlbHNlIG5wLm5hbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0cmFjZV9raW5kIjogImFwcHJveGltYXRlX25hdGl2ZV9maWxlX3BvbGwiIGlmIGFybS5zdGFydHN3aXRoKCJuYXRpdmVfIikKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVsc2UgImV4YWN0X3BhdGNoZWRfYmVzdF9ldmVudCJ9KQoKdGFibGUgPSBwZC5EYXRhRnJhbWUocm93cykKbWVjaGFuaXNtX3RhYmxlID0gcGQuRGF0YUZyYW1lKG1lY2hhbmlzbV9yb3dzKQp0YXJnZXRfdGFibGUgPSBwZC5EYXRhRnJhbWUodGFyZ2V0X3Jvd3MpCmZyZXF1ZW5jeV90YWJsZSA9IHBkLkRhdGFGcmFtZShmcmVxdWVuY3lfcm93cykKdGFibGUudG9fY3N2KHN1bW1hcnlfZGlyIC8gImNvbXBsZXRlX2luc3RhbmNlcy5jc3YiLCBpbmRleD1GYWxzZSkKbWVjaGFuaXNtX3RhYmxlLnRvX2NzdihzdW1tYXJ5X2RpciAvICJjYW5kaWRhdGVfbWVjaGFuaXNtLmNzdiIsIGluZGV4PUZhbHNlKQp0YXJnZXRfdGFibGUudG9fY3N2KHN1bW1hcnlfZGlyIC8gInRpbWVfdG9fdGFyZ2V0X2V4cGxvcmF0b3J5LmNzdiIsIGluZGV4PUZhbHNlKQpmcmVxdWVuY3lfdGFibGUudG9fY3N2KHN1bW1hcnlfZGlyIC8gInNlbGVjdGVkX2VkZ2VfZnJlcXVlbmNpZXMuY3N2IiwgaW5kZXg9RmFsc2UpCgpzYW1lX2hhcmR3YXJlID0gdGFibGUubG9jW350YWJsZVsibWl4ZWRfaGFyZHdhcmUiXV0uY29weSgpIGlmIG5vdCB0YWJsZS5lbXB0eSBlbHNlIHRhYmxlCm1hdGNoZWQgPSBzYW1lX2hhcmR3YXJlLmxvY1tzYW1lX2hhcmR3YXJlWyJtYXRjaGVkX3BvbGljeV9zdGF0ZSJdXS5jb3B5KCkgaWYgbm90IHNhbWVfaGFyZHdhcmUuZW1wdHkgZWxzZSBzYW1lX2hhcmR3YXJlCmVuZHBvaW50X3Jvd3MgPSBbXQpmb3IgY29udHJvbCBpbiAoIm5hdGl2ZV9wb3A2IiwgIm5hdGl2ZV9wb3A3IiwgInNoYW0iLCAicmFuZG9tX2FkZCIsICJmcmVxdWVuY3lfYWRkIik6CiAgICByZXN1bHQgPSBjb21wYXJpc29uX3N1bW1hcnkoc2FtZV9oYXJkd2FyZSwgImJldGFfYWRkIiwgY29udHJvbCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZGVzY3JpcHRpdmUgc2FtZS1pbnN0YW5jZSBlbmRwb2ludCIpCiAgICBpZiByZXN1bHQgaXMgbm90IE5vbmU6CiAgICAgICAgZW5kcG9pbnRfcm93cy5hcHBlbmQocmVzdWx0KQptYXRjaGVkX3Jvd3MgPSBbXQpmb3IgY29udHJvbCBpbiAoInNoYW0iLCAicmFuZG9tX2FkZCIsICJmcmVxdWVuY3lfYWRkIik6CiAgICByZXN1bHQgPSBjb21wYXJpc29uX3N1bW1hcnkobWF0Y2hlZCwgImJldGFfYWRkIiwgY29udHJvbCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiYWxsIHNpeCByYXcgZmlyc3QgdG91cnMvY2FuZGlkYXRlIHN0YXRlcy9wcmVkaWN0aW9ucy9hY3Rpb25zL2dhdGVzL2NhcGFjaXRpZXMgbWF0Y2giKQogICAgaWYgcmVzdWx0IGlzIG5vdCBOb25lOgogICAgICAgIG1hdGNoZWRfcm93cy5hcHBlbmQocmVzdWx0KQpjb21wYXJpc29uX2NvbHVtbnMgPSBbInRyZWF0bWVudCIsICJjb250cm9sIiwgImJhc2lzIiwgIm5faW5zdGFuY2VzIiwgIndpbnMiLAogICAgICAgICAgICAgICAgICAgICAgImxvc3NlcyIsICJ0aWVzIiwgIm1lYW5fcmVsYXRpdmVfcGN0IiwgIm1lZGlhbl9yZWxhdGl2ZV9wY3QiXQplbmRwb2ludF90YWJsZSA9IHBkLkRhdGFGcmFtZShlbmRwb2ludF9yb3dzLCBjb2x1bW5zPWNvbXBhcmlzb25fY29sdW1ucykKbWF0Y2hlZF90YWJsZSA9IHBkLkRhdGFGcmFtZShtYXRjaGVkX3Jvd3MsIGNvbHVtbnM9Y29tcGFyaXNvbl9jb2x1bW5zKQplbmRwb2ludF90YWJsZS50b19jc3Yoc3VtbWFyeV9kaXIgLyAiZW5kcG9pbnRfY29tcGFyaXNvbnMuY3N2IiwgaW5kZXg9RmFsc2UpCm1hdGNoZWRfdGFibGUudG9fY3N2KHN1bW1hcnlfZGlyIC8gIm1hdGNoZWRfcG9saWN5X2NvbXBhcmlzb25zLmNzdiIsIGluZGV4PUZhbHNlKQoKaWYgbWVjaGFuaXNtX3RhYmxlLmVtcHR5OgogICAgbWVjaGFuaXNtX2FnZ3JlZ2F0ZSA9IHBkLkRhdGFGcmFtZSgpCmVsc2U6CiAgICBtZWNoYW5pc21fYWdncmVnYXRlID0gbWVjaGFuaXNtX3RhYmxlLmdyb3VwYnkoImFybSIsIGFzX2luZGV4PUZhbHNlKVsKICAgICAgICBbInNlbGVjdGVkX2FjdGlvbnMiLCAiYWN0aW9uX2NhbmRpZGF0ZV9lbnVtZXJhdGlvbnMiLAogICAgICAgICAiZGlzdGluY3RfYWN0aW9uc19lbnVtZXJhdGVkIiwgImFjdGlvbl9hY2NlcHRlZF9lZGdlX2VudHJpZXMiLAogICAgICAgICAiZGlzdGluY3RfYWN0aW9uc19lbnRlcmluZ19hY2NlcHRlZF90b3VycyIsCiAgICAgICAgICJ0b3BfbGV2ZWxfYWN0aW9uX2ZpcnN0X2dhdGVfcGFzc2VzIiwgInRvcF9sZXZlbF9hY3Rpb25fcG9zaXRpdmVfZzEiLAogICAgICAgICAiYWN0aW9uX2luY3VtYmVudF9lbnRyaWVzIiwgImRpc3RpbmN0X2FjdGlvbnNfZXZlcl9pbl9iZXR0ZXJfdG91ciJdXS5zdW0oKQptZWNoYW5pc21fYWdncmVnYXRlLnRvX2NzdihzdW1tYXJ5X2RpciAvICJjYW5kaWRhdGVfbWVjaGFuaXNtX3RvdGFscy5jc3YiLCBpbmRleD1GYWxzZSkKCndyaXRlX2pzb25fYXRvbWljKHN1bW1hcnlfZGlyIC8gInBpbG90X25vdGUuanNvbiIsIHsKICAgICJhbmFseXNpc19yZXZpc2lvbiI6ICJ2MTEtc3RhdGUtZ3VhcmRlZC1tZWNoYW5pc20tMSIsCiAgICAiY29tcGxldGVkX2luc3RhbmNlcyI6IGxlbih0YWJsZSksICJwbGFubmVkX2luc3RhbmNlcyI6IGxlbihJTlNUQU5DRVMpLAogICAgInNhbWVfaGFyZHdhcmVfaW5zdGFuY2VzIjogbGVuKHNhbWVfaGFyZHdhcmUpLAogICAgIm1hdGNoZWRfcG9saWN5X2luc3RhbmNlcyI6IGxlbihtYXRjaGVkKSwKICAgICJpbnRlcnByZXRhdGlvbiI6ICJSZXVzZWQtY2FzZSBtZWNoYW5pc20gcGlsb3QuIEVuZHBvaW50IGNvbXBhcmlzb25zIGFnYWluc3QgbmF0aXZlIFBPUDYvNyBhcmUgZGVzY3JpcHRpdmUuIFBhdGNoZWQgcG9saWN5IGNvbXBhcmlzb25zIHJlcXVpcmUgZXhhY3Qgc2l4LXNlZWQgc3RhdGUsIGdhdGUgYW5kIGNhcGFjaXR5IG1hdGNoaW5nLiBOYXRpdmUgdGltZS10by10YXJnZXQgdHJhY2VzIGFyZSBhcHByb3hpbWF0ZSBmaWxlIHBvbGxzOyBwYXRjaGVkIGJlc3QuZXZlbnRzIGFyZSBleGFjdC4gTm8gbWFudXNjcmlwdCBzdXBlcmlvcml0eSBjbGFpbS4iCn0pCgpwcmludCgiVGFtYW1sYW5hbiDDtnJuZWs6IiwgbGVuKHRhYmxlKSwgIi8iLCBsZW4oSU5TVEFOQ0VTKSwKICAgICAgIjsgdGFtIGXFn2xlxZ9tacWfIHBvbGl0aWthIMO2cm5lxJ9pOiIsIGxlbihtYXRjaGVkKSkKZGlzcGxheSh0YWJsZSkKZGlzcGxheShlbmRwb2ludF90YWJsZSkKZGlzcGxheShtYXRjaGVkX3RhYmxlKQpkaXNwbGF5KG1lY2hhbmlzbV9hZ2dyZWdhdGUpCmRpc3BsYXkoZnJlcXVlbmN5X3RhYmxlKQpkaXNwbGF5KHRhcmdldF90YWJsZSkK', 'v11_candidate/build_colab_notebook.py': 'IiIiQnVpbGQgdGhlIHNlbGYtY29udGFpbmVkLCByZXN1bWFibGUgdjExIFBPUDYtcmVzaWR1YWwgbWVjaGFuaXNtIG5vdGVib29rLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGJhc2U2NAppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW1wb3J0bGliLnV0aWwKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKSEVSRSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKRVhQRVJJTUVOVFMgPSBIRVJFLnBhcmVudApSRVBPID0gRVhQRVJJTUVOVFMucGFyZW50Ck9VVFBVVCA9IFJFUE8gLyAibm90ZWJvb2tzIiAvICIwNV9CZXRhVG91cl92MTFfUE9QNl9jYW5kaWRhdGVfbWVjaGFuaXNtLmlweW5iIgpWOV9CVUlMREVSID0gRVhQRVJJTUVOVFMgLyAidjlfbGl2ZSIgLyAiYnVpbGRfY29sYWJfbm90ZWJvb2sucHkiCnNwZWMgPSBpbXBvcnRsaWIudXRpbC5zcGVjX2Zyb21fZmlsZV9sb2NhdGlvbigiYmV0YXRvdXJfdjlfYnVpbGRlciIsIFY5X0JVSUxERVIpCmlmIHNwZWMgaXMgTm9uZSBvciBzcGVjLmxvYWRlciBpcyBOb25lOgogICAgcmFpc2UgUnVudGltZUVycm9yKCJ2OSBub3RlYm9vayBidWlsZGVyIG1pc3NpbmciKQp2OSA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoc3BlYykKc3BlYy5sb2FkZXIuZXhlY19tb2R1bGUodjkpCgoKZGVmIHJlcGxhY2Vfb25jZShzb3VyY2U6IHN0ciwgb2xkOiBzdHIsIG5ldzogc3RyKSAtPiBzdHI6CiAgICBpZiBzb3VyY2UuY291bnQob2xkKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHBlY3RlZCBleGFjdGx5IG9uZSBub3RlYm9vayBhbmNob3I6IHtvbGRbOjcwXSFyfSIpCiAgICByZXR1cm4gc291cmNlLnJlcGxhY2Uob2xkLCBuZXcsIDEpCgoKZGVmIHNldF9jZWxsKG5vdGVib29rOiBkaWN0LCBpbmRleDogaW50LCBzb3VyY2U6IHN0cikgLT4gTm9uZToKICAgIGNlbGwgPSBub3RlYm9va1siY2VsbHMiXVtpbmRleF0KICAgIGlmIGNlbGxbImNlbGxfdHlwZSJdID09ICJjb2RlIjoKICAgICAgICBjb21waWxlKHNvdXJjZSwgZiI8djExIGNlbGwge2luZGV4fT4iLCAiZXhlYyIpCiAgICBjZWxsWyJzb3VyY2UiXSA9IHNvdXJjZS5zcGxpdGxpbmVzKGtlZXBlbmRzPVRydWUpCiAgICBjZWxsWyJpZCJdID0gZiJiZXRhdG91cl92MTFfe2luZGV4OjAyZH1fe2hhc2hsaWIuc2hhMjU2KHNvdXJjZS5lbmNvZGUoKSkuaGV4ZGlnZXN0KClbOjEwXX0iCgoKZGVmIG1ha2Vfbm90ZWJvb2soKSAtPiBkaWN0OgogICAgbm90ZWJvb2sgPSB2OS5tYWtlX25vdGVib29rKCkKICAgIG9sZCA9IGxhbWJkYSBpbmRleDogIiIuam9pbihub3RlYm9va1siY2VsbHMiXVtpbmRleF1bInNvdXJjZSJdKQoKICAgIHNldF9jZWxsKG5vdGVib29rLCAwLCAiIiIjIEJldGFUb3VyIHYxMSDigJQgUE9QNiBhZGF5IGtlbmFyIG1la2FuaXptYSBwaWxvdHUKCkJ1IGRlbmV5ICoqZXNraSB2OC92OSBnZWxpxZ90aXJtZSDDtnJuZWtsZXJpbmluIGTDtnJ0IDEwayDDtnJuZcSfaW5pKioga3VsbGFuxLFyOyBiYcSfxLFtc8SxeiBkb8SfcnVsYW1hIGRlxJ9pbGRpci4gUE9QTVVTSUMtNi9MS0gnbmluIGF5bsSxIGlsayBkZW5lbWVzaW5kZW4gc29ucmEgQmV0YVRvdXIsIHR1ciBzxLFrbMSxxJ/EsSB2ZXlhIHJhc3RnZWxlIGtvbnRyb2wsIGF5bsSxIGtheW5hayBkw7zEn8O8bWxlcmUgdmUgYXluxLEgc2F5xLFkYSB5ZW5pIHnDtm5zw7x6IGFkYXkga2VuYXJhIG3DvGRhaGFsZSBlZGVyLiBTaGFtIGF5bsSxIG1vZGVsIMOnxLFrYXLEsW3EsW7EsSB5YXBhciB2ZSBhcmFtYXnEsSBkZcSfacWfdGlybWV6LiBOYXRpdmUgUE9QNiB2ZSBQT1A3IHByYXRpayBrxLF5YXNsYXJkxLFyLiBCw7x0w7xuIGtvbCBiw7x0w6dlbGVyaW5lIGt1cnVsdW0sIG1vZGVsIMOnxLFrYXLEsW3EsSwgYmVrbGVtZSB2ZSBMS0ggYXJhbWFzxLEgZGFoaWxkaXIuCgoqKkFtYcOnOioqIE1vZGVsaW4gZWtsZWRpxJ9pIGtlbmFybGFyxLFuIExLSCBhZGF5IHRhcmFtYXPEsW5hIGdpcmlwIGdpcm1lZGnEn2luaSB2ZSBkYWhhIGl5aSBpbmN1bWJlbnQgdHVybGFyZGEgZ8O2csO8bm1lIHPEsWtsxLHEn8SxbsSxIMO2bMOnbWVrOyBtb2RlbGluIGfDvMOnbMO8IFBPUDYvUE9QNyB2ZSBlxZ9pdCBrYXBhc2l0ZWxpIGtvbnRyb2wgc2XDp2ljaWxlcmluZGVuIGF5csSxxZ/EsXAgYXlyxLHFn21hZMSxxJ/EsW7EsSBnw7ZybWVrLiDEsGxrIHR1ciB2ZSBhZGF5IGR1cnVtbGFyxLEgZcWfbGXFn21leWVuIGtvbGxhcsSxbiBlxZ9saSBwb2xpdGlrYSBrYXLFn8SxbGHFn3TEsXJtYXPEsSB5YXDEsWxtYXouIFR1ciBtYWxpeWV0aW5pbiB5YW7EsW5kYSBoZWRlZiBrYWxpdGV5ZSB1bGHFn21hIHPDvHJlc2kgZGUga2F5ZGVkaWxpci4gQnUgcGlsb3Qgb2x1bWx1IG9sc2EgYmlsZSBtYWthbGUsIGRvbm11xZ8gcG9saXRpa2EgYmHEn8SxbXPEsXogw7ZybmVrbGVyZGUgZG/En3J1bGFuYW5hIGRlayBkZcSfacWfdGlyaWxtZXllY2VrLgoKKirDh2FsxLHFn3TEsXJtYToqKiBDb2xhYidkYSB0ZXJjaWhlbiBBMTAwIHZleWEgTDQgR1BVIHZlIEhpZ2ggUkFNIHNlw6dpbi4gQWx0xLEgZcWfemFtYW5sxLEgTEtIIGnDp2luIDEyIHZDUFUgdGVyY2loIGVkaWxpci4gSMO8Y3JlbGVyaSBzxLFyYXlsYSDDp2FsxLHFn3TEsXLEsW4uIFRhbWFtbGFuYW4ga29sbGFyIGBNeURyaXZlL0JldGFUb3VyL3YxMV9wb3A2X2NhbmRpZGF0ZV9tZWNoYW5pc21gIGnDp2luZGUgZG/En3J1bGFuYXJhayBhdGxhbsSxcjsga2VzaWxlbiBrb2wgeWVuaWRlbiDDp2FsxLHFn8Sxci4gU29udcOnIFpJUCdpIHNvbnVuZGEgdGFyYXnEsWPEsXlhIG90b21hdGlrIGluZXIsIHRhbSBkZW5ldGltIFpJUCdpIERyaXZlJ2RhIGthbMSxci4gRMO2cnQgw7ZybmVrIMOXIGFsdMSxIGtvbCDDlyAxMjAgc2FuaXllID0gKio0OCBkYWtpa2EqKiBzYWx0IMOnw7Z6w7xjw7wgc8SxbsSxcsSxOyBrdXJ1bHVtIHZlIGFyxZ9pdmxlbWUgZWtsZW5pci4KIiIiKQoKICAgIGNvbmZpZyA9IG9sZCgxKQogICAgY29uZmlnID0gcmVwbGFjZV9vbmNlKGNvbmZpZywgJ0VYUEVSSU1FTlRfTkFNRSA9ICJ2OV9saXZlX3BpbG90IicsCiAgICAgICAgICAgICAgICAgICAgICAgICAgJ0VYUEVSSU1FTlRfTkFNRSA9ICJ2MTFfcG9wNl9jYW5kaWRhdGVfbWVjaGFuaXNtIicpCiAgICBjb25maWcgPSByZXBsYWNlX29uY2UoY29uZmlnLAogICAgICAgICdBUk1TID0gWyJuYXRpdmVfcG9wNSIsICJuYXRpdmVfcG9wNiIsICJzaGFtIiwgImJldGFfcHJvbW90ZSIsICJiZXRhX3NwYXJzZSIsICJmcmVxdWVuY3lfc2FtZV9nYXRlIl0nLAogICAgICAgICdBUk1TID0gWyJuYXRpdmVfcG9wNiIsICJuYXRpdmVfcG9wNyIsICJzaGFtIiwgInJhbmRvbV9hZGQiLCAiZnJlcXVlbmN5X2FkZCIsICJiZXRhX2FkZCJdJykKICAgIGNvbmZpZyA9IHJlcGxhY2Vfb25jZShjb25maWcsCiAgICAgICAgJyAgICBbInVuaTIwayIsIDIwMDAwLCAyLCAidW5pZm9ybSIsIDQzMDksIDMwMC4wXSxcbiAgICBbImNsdTIwayIsIDIwMDAwLCAyLCAiY2x1c3RlcmVkIiwgNDQxMCwgMzAwLjBdLFxuJywgJycpCiAgICBjb25maWcgPSByZXBsYWNlX29uY2UoY29uZmlnLCAnU1BBUlNFX0ZSQUNUSU9OID0gMC4yNScsCiAgICAgICAgICAgICAgICAgICAgICAgICAgJ1NQQVJTRV9GUkFDVElPTiA9IDAuMjVcbkdSQVBIX0sgPSAxNicpCiAgICBjb25maWcgPSByZXBsYWNlX29uY2UoY29uZmlnLCAnYXNzZXJ0IGxlbihTRUVEUykgPT0gNiBhbmQgbGVuKEVYUEVDVEVEX1hZX1NIQTI1NikgPT0gOCcsCiAgICAgICAgJ0VYUEVDVEVEX1hZX1NIQTI1NiA9IHtuYW1lOiBkaWdlc3QgZm9yIG5hbWUsIGRpZ2VzdCBpbiBFWFBFQ1RFRF9YWV9TSEEyNTYuaXRlbXMoKVxuJwogICAgICAgICcgICAgICAgICAgICAgICAgICAgICAgaWYgIjEwayIgaW4gbmFtZX1cbicKICAgICAgICAnVEFSR0VUX0NPU1RTID0geyJhdWdtZW50X2NsdTEwa18wMDAiOiAzMzc2MDE5OSxcbicKICAgICAgICAnICAgICAgICAgICAgICAgICJhdWdtZW50X2NsdTEwa18wMDEiOiAxNjIwMTE4NixcbicKICAgICAgICAnICAgICAgICAgICAgICAgICJhdWdtZW50X3VuaTEwa18wMDAiOiA3MjA4Nzc4MSxcbicKICAgICAgICAnICAgICAgICAgICAgICAgICJhdWdtZW50X3VuaTEwa18wMDEiOiA3MTY0ODU3MH1cbicKICAgICAgICAnYXNzZXJ0IGxlbihTRUVEUykgPT0gNiBhbmQgbGVuKEVYUEVDVEVEX1hZX1NIQTI1NikgPT0gNCBhbmQgbGVuKFRBUkdFVF9DT1NUUykgPT0gNCcpCiAgICBzZXRfY2VsbChub3RlYm9vaywgMSwgY29uZmlnKQoKICAgIHNldHVwID0gb2xkKDIpCiAgICBzZXR1cCA9IHJlcGxhY2Vfb25jZShzZXR1cCwgJ0NvbGFiIEdQVSBydW50aW1lIHNlw6dpbiAoQTEwMCB2ZXlhIEw0KS4nLAogICAgICAgICAgICAgICAgICAgICAgICAgICdDb2xhYiBHUFUgcnVudGltZSBzZcOnaW4gKHRlcmNpaGVuIEExMDAgdmV5YSBMNCkuJykKICAgIHNldHVwID0gcmVwbGFjZV9vbmNlKHNldHVwLAogICAgICAgICdhc3NlcnQgIkExMDAiIGluIEdQVV9OQU1FIG9yICJMNCIgaW4gR1BVX05BTUUsIGYiQmVrbGVuZW4gR1BVIEExMDAvTDQ7IGfDtnLDvG5lbjoge0dQVV9OQU1FfSInLAogICAgICAgICdpZiBub3QgYW55KG5hbWUgaW4gR1BVX05BTUUgZm9yIG5hbWUgaW4gKCJBMTAwIiwgIkw0IikpOlxuJwogICAgICAgICcgICAgcHJpbnQoIlVZQVJJOiBHUFUiLCBHUFVfTkFNRSwgIuKAlCBzb251w6dsYXLEsSBmYXJrbMSxIHByb2ZpbGxlcmxlIGXFn2xlbWV5aW4uIiknKQogICAgc2V0dXAgPSByZXBsYWNlX29uY2Uoc2V0dXAsCiAgICAgICAgJ2Fzc2VydCBDUFVfQ09VTlQgPj0gMTIsIGYiRW4gYXogMTIgZ8O2csO8bsO8ciB2Q1BVIGdlcmVrbGk7IGfDtnLDvG5lbjoge0NQVV9DT1VOVH0iJywKICAgICAgICAnaWYgQ1BVX0NPVU5UIDwgNjpcbicKICAgICAgICAnICAgIHByaW50KCJVWUFSSTogQWx0xLEgTEtIIHPDvHJlY2kgacOnaW4gZW4gYXogNiB2Q1BVIMO2bmVyaWxpcjsgZ8O2csO8bmVuOiIsIENQVV9DT1VOVCknKQogICAgc2V0dXAgPSByZXBsYWNlX29uY2Uoc2V0dXAsICdiZXRhdG91cl92OV9saXZlX3dvcmsnLCAnYmV0YXRvdXJfdjExX2NhbmRpZGF0ZV93b3JrJykKICAgIHNldHVwID0gcmVwbGFjZV9vbmNlKHNldHVwLAogICAgICAgICcicHl0aG9uIjogcGxhdGZvcm0ucHl0aG9uX3ZlcnNpb24oKSwgInRvcmNoIjogdG9yY2guX192ZXJzaW9uX199JywKICAgICAgICAnInB5dGhvbiI6IHBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksICJ0b3JjaCI6IHRvcmNoLl9fdmVyc2lvbl9fLFxuJwogICAgICAgICcgICAgICAgICAgICAibnVtcHkiOiBucC5fX3ZlcnNpb25fXywgInNjaXB5Ijogc2NpcHkuX192ZXJzaW9uX18sXG4nCiAgICAgICAgJyAgICAgICAgICAgICJwYW5kYXMiOiBwZC5fX3ZlcnNpb25fXywgImN1ZGEiOiB0b3JjaC52ZXJzaW9uLmN1ZGF9JykKICAgIHNldF9jZWxsKG5vdGVib29rLCAyLCBzZXR1cCkKCiAgICBmaWxlcyA9IHsKICAgICAgICAibGl2ZWhvb2svY29vcmRpbmF0b3IucHkiOiBFWFBFUklNRU5UUyAvICJsaXZlaG9vayIgLyAiY29vcmRpbmF0b3IucHkiLAogICAgICAgICJ2MTFfY2FuZGlkYXRlL0JldGFUb3VyU2VhcmNoSG9vay5oIjogSEVSRSAvICJCZXRhVG91clNlYXJjaEhvb2suaCIsCiAgICAgICAgInYxMV9jYW5kaWRhdGUvQmV0YVRvdXJWaXNpdC5oIjogSEVSRSAvICJCZXRhVG91clZpc2l0LmgiLAogICAgICAgICJ2MTFfY2FuZGlkYXRlL2J1aWxkX3NlYXJjaF9ob29rLnB5IjogSEVSRSAvICJidWlsZF9zZWFyY2hfaG9vay5weSIsCiAgICAgICAgInYxMV9jYW5kaWRhdGUvcG9saWN5LnB5IjogSEVSRSAvICJwb2xpY3kucHkiLAogICAgICAgICJ2MTFfY2FuZGlkYXRlL3BpbG90LnB5IjogSEVSRSAvICJwaWxvdC5weSIsCiAgICAgICAgInYxMV9jYW5kaWRhdGUvcnVuX2NlbGwucHkiOiBIRVJFIC8gInJ1bl9jZWxsLnB5IiwKICAgICAgICAidjExX2NhbmRpZGF0ZS9hbmFseXNpc19jZWxsLnB5IjogSEVSRSAvICJhbmFseXNpc19jZWxsLnB5IiwKICAgICAgICAidjExX2NhbmRpZGF0ZS9idWlsZF9jb2xhYl9ub3RlYm9vay5weSI6IEhFUkUgLyAiYnVpbGRfY29sYWJfbm90ZWJvb2sucHkiLAogICAgfQogICAgYnVuZGxlID0ge25hbWU6IGJhc2U2NC5iNjRlbmNvZGUocGF0aC5yZWFkX2J5dGVzKCkpLmRlY29kZSgiYXNjaWkiKQogICAgICAgICAgICAgIGZvciBuYW1lLCBwYXRoIGluIGZpbGVzLml0ZW1zKCl9CiAgICBzZXRfY2VsbChub3RlYm9vaywgMywgIiIiQlVORExFRF9GSUxFUyA9ICIiIiArIHJlcHIoYnVuZGxlKSArICIiIgpDT0RFX1JPT1QgPSBXT1JLX1JPT1QgLyAiY29kZSIKZm9yIHJlbCwgZW5jb2RlZCBpbiBCVU5ETEVEX0ZJTEVTLml0ZW1zKCk6CiAgICBjb250ZW50ID0gYmFzZTY0LmI2NGRlY29kZShlbmNvZGVkKQogICAgZm9yIGRlc3QgaW4gKENPREVfUk9PVCAvIHJlbCwgT1VUUFVUX1JPT1QgLyAiY29kZSIgLyByZWwpOgogICAgICAgIGRlc3QucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBpZiBkZXN0LmV4aXN0cygpOgogICAgICAgICAgICBhc3NlcnQgZGVzdC5yZWFkX2J5dGVzKCkgPT0gY29udGVudCwgZiJEZW5leSBrb2R1IGRlxJ9pxZ9tacWfOiB7cmVsfSIKICAgICAgICBlbHNlOgogICAgICAgICAgICBkZXN0LndyaXRlX2J5dGVzKGNvbnRlbnQpCkNPREVfU0hBMjU2ID0ge3JlbDogaGFzaGxpYi5zaGEyNTYoYmFzZTY0LmI2NGRlY29kZShlbmNvZGVkKSkuaGV4ZGlnZXN0KCkKICAgICAgICAgICAgICAgZm9yIHJlbCwgZW5jb2RlZCBpbiBCVU5ETEVEX0ZJTEVTLml0ZW1zKCl9CnN5cy5wYXRoWzowXSA9IFtzdHIoQ09ERV9ST09UIC8gImxpdmVob29rIiksIHN0cihDT0RFX1JPT1QgLyAidjExX2NhbmRpZGF0ZSIpXQppbXBvcnQgYnVpbGRfc2VhcmNoX2hvb2sKZnJvbSBjb29yZGluYXRvciBpbXBvcnQgcmVhZF90b3VyCmZyb20gcGlsb3QgaW1wb3J0IHJ1bl9ndWlkZWRfYXJtLCBydW5fbmF0aXZlX2FybSwgc2hhMjU2X2ZpbGUKcHJpbnQoIkRvbmR1cnVsbXXFnyBkZW5leSBrb2R1OiIsIENPREVfU0hBMjU2KQoiIiIpCgogICAgYnVpbGQgPSBvbGQoNCkKICAgIGJ1aWxkID0gcmVwbGFjZV9vbmNlKGJ1aWxkLCAnUEFUQ0hFRF9ESVIgPSBMS0hfUk9PVCAvICJMS0gtMy4wLjE0LUJldGFUb3VyIicsCiAgICAgICAgICAgICAgICAgICAgICAgICAnUEFUQ0hFRF9ESVIgPSBMS0hfUk9PVCAvICJMS0gtMy4wLjE0LUJldGFUb3VyLXYxMSInKQogICAgYnVpbGQgPSByZXBsYWNlX29uY2UoYnVpbGQsICdidWlsZF9saXZlaG9vay5idWlsZChTT1VSQ0VfRElSLCBQQVRDSEVEX0RJUiwgam9icz1taW4oNiwgQ1BVX0NPVU5UKSknLAogICAgICAgICAgICAgICAgICAgICAgICAgJ2J1aWxkX3NlYXJjaF9ob29rLmJ1aWxkKFNPVVJDRV9ESVIsIFBBVENIRURfRElSLCBqb2JzPW1pbig2LCBDUFVfQ09VTlQpKScpCiAgICBidWlsZCA9IHJlcGxhY2Vfb25jZShidWlsZCwKICAgICAgICAnaWYgbm90IFNPVVJDRV9JTkZPX1BBVEguZXhpc3RzKCk6JywKICAgICAgICAnZm9yIGZpbGVuYW1lLCBleHBlY3RlZCBpbiBidWlsZF9zZWFyY2hfaG9vay5VUFNUUkVBTV9TSEEyNTYuaXRlbXMoKTpcbicKICAgICAgICAnICAgIGFzc2VydCBoYXNobGliLnNoYTI1NigoU09VUkNFX0RJUiAvICJTUkMiIC8gZmlsZW5hbWUpLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkgPT0gZXhwZWN0ZWQsIGZpbGVuYW1lXG4nCiAgICAgICAgJ2lmIG5vdCBTT1VSQ0VfSU5GT19QQVRILmV4aXN0cygpOicpCiAgICBidWlsZCA9IHJlcGxhY2Vfb25jZShidWlsZCwKICAgICAgICAnQlVJTERfSU5GTyA9IHsibmF0aXZlX3NoYTI1NiI6IHNoYTI1Nl9maWxlKE5BVElWRV9MS0gpLCcsCiAgICAgICAgJ3BhdGNoX21hbmlmZXN0ID0ganNvbi5sb2FkcygoUEFUQ0hFRF9ESVIgLyAiYmV0YXRvdXJfYnVpbGQuanNvbiIpLnJlYWRfdGV4dCgpKVxuJwogICAgICAgICdhc3NlcnQgcGF0Y2hfbWFuaWZlc3RbInVwc3RyZWFtX3NoYTI1NiJdID09IGJ1aWxkX3NlYXJjaF9ob29rLlVQU1RSRUFNX1NIQTI1NlxuJwogICAgICAgICdhc3NlcnQgcGF0Y2hfbWFuaWZlc3RbInNlYXJjaF9ob29rX3NoYTI1NiJdID09IENPREVfU0hBMjU2WyJ2MTFfY2FuZGlkYXRlL0JldGFUb3VyU2VhcmNoSG9vay5oIl1cbicKICAgICAgICAnYXNzZXJ0IHBhdGNoX21hbmlmZXN0WyJ2aXNpdF9oZWFkZXJfc2hhMjU2Il0gPT0gQ09ERV9TSEEyNTZbInYxMV9jYW5kaWRhdGUvQmV0YVRvdXJWaXNpdC5oIl1cbicKICAgICAgICAnYXNzZXJ0IHBhdGNoX21hbmlmZXN0WyJiaW5hcnlfc2hhMjU2Il0gPT0gc2hhMjU2X2ZpbGUoUEFUQ0hFRF9MS0gpXG4nCiAgICAgICAgJ0JVSUxEX0lORk8gPSB7Im5hdGl2ZV9zaGEyNTYiOiBzaGEyNTZfZmlsZShOQVRJVkVfTEtIKSwnKQogICAgc2V0X2NlbGwobm90ZWJvb2ssIDQsIGJ1aWxkKQoKICAgIHNldF9jZWxsKG5vdGVib29rLCA1LCAiIiJkZWYgd3JpdGVfanNvbl9hdG9taWMocGF0aCwgb2JqKToKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0bXAgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnRtcCIpCiAgICB0bXAud3JpdGVfdGV4dChqc29uLmR1bXBzKG9iaiwgaW5kZW50PTIsIGVuc3VyZV9hc2NpaT1GYWxzZSwgYWxsb3dfbmFuPUZhbHNlKSkKICAgIG9zLnJlcGxhY2UodG1wLCBwYXRoKQoKTUFOSUZFU1QgPSB7InNjaGVtYSI6IDEsICJwcm9maWxlIjogImZvdXJfcmV1c2VkXzEwa19kZXZlbG9wbWVudF9pbnN0YW5jZXMiLAogICAgICAgICAgICAiZXhwZXJpbWVudF9uYW1lIjogRVhQRVJJTUVOVF9OQU1FLCAiYXJtcyI6IEFSTVMsICJzZWVkcyI6IFNFRURTLAogICAgICAgICAgICAiaW5zdGFuY2Vfc3BlY3MiOiBJTlNUQU5DRV9TUEVDUywgInh5X2hhc2hlcyI6IEVYUEVDVEVEX1hZX1NIQTI1NiwKICAgICAgICAgICAgInRpbWVfdG9fdGFyZ2V0X2Nvc3RzX2Zyb21fdjlfbmF0aXZlX3BvcDYiOiBUQVJHRVRfQ09TVFMsCiAgICAgICAgICAgICJzcGFyc2VfZnJhY3Rpb24iOiBTUEFSU0VfRlJBQ1RJT04sICJncmFwaF9rIjogR1JBUEhfSywKICAgICAgICAgICAgInJlcG9fY29tbWl0IjogUkVQT19DT01NSVQsICJjaGVja3BvaW50X3NoYTI1NiI6IENIRUNLUE9JTlRfU0hBMjU2LAogICAgICAgICAgICAiaGFyZHdhcmUiOiBIQVJEV0FSRSwgImNvZGVfc2hhMjU2IjogQ09ERV9TSEEyNTYsICJidWlsZCI6IEJVSUxEX0lORk8sCiAgICAgICAgICAgICJhbmFseXNpc19wbGFuIjogIk1lY2hhbmlzbSBwaWxvdCBvbiByZXVzZWQgY2FzZXM7IG5vIHBhcGVyIHBlcmZvcm1hbmNlIGNsYWltIGJlZm9yZSBmcm96ZW4gaW5kZXBlbmRlbnQgaG9sZG91dC4ifQptYW5pZmVzdF9wYXRoID0gT1VUUFVUX1JPT1QgLyAibWFuaWZlc3QuanNvbiIKaWYgbWFuaWZlc3RfcGF0aC5leGlzdHMoKToKICAgIHNhdmVkID0ganNvbi5sb2FkcyhtYW5pZmVzdF9wYXRoLnJlYWRfdGV4dCgpKQogICAgZm9yIGtleSBpbiBNQU5JRkVTVDoKICAgICAgICBpZiBrZXkgIT0gImhhcmR3YXJlIjoKICAgICAgICAgICAgYXNzZXJ0IHNhdmVkW2tleV0gPT0gTUFOSUZFU1Rba2V5XSwgZiJEcml2ZSdkYWtpIHBpbG90dW4ge2tleX0gYXlhcsSxIGRlxJ9pxZ90aTsgeWVuaSBkZW5leSBhZMSxIGt1bGxhbsSxbi4iCiAgICBmb3Iga2V5IGluICgiZ3B1IiwgInZjcHVzIik6CiAgICAgICAgYXNzZXJ0IHNhdmVkWyJoYXJkd2FyZSJdW2tleV0gPT0gSEFSRFdBUkVba2V5XSwgZiJEb25hbsSxbSB7a2V5fSBkZcSfacWfdGk7IGXFn2xpIHBpbG90dSBheW7EsSB0w7xyZGUgw6dhbMSxxZ90xLFyxLFuLiIKZWxzZToKICAgIHdyaXRlX2pzb25fYXRvbWljKG1hbmlmZXN0X3BhdGgsIE1BTklGRVNUKQoKIyBHUFUgd2FybXVwIGFuZCBleGFjdCBmaXJzdC10cmlhbC9uYXRpdmUgZXF1aXZhbGVuY2UgY2hlY2suCndhcm1feHkgPSBULnRvX2ludF9jb29yZHMoVC5nZW5faW5zdGFuY2UoMTAwLCAidW5pZm9ybSIsIG5wLnJhbmRvbS5kZWZhdWx0X3JuZyg5MDkwKSkpCndhcm1fc3VjYyA9IG5wLnN0YWNrKFtucC5yb2xsKG5wLmFyYW5nZSgxMDApLCAtMSldICogNiwgYXhpcz0xKS5hc3R5cGUobnAuaW50MzIpCk0ucHJlZGljdF90b3VyZm9sZChNT0RFTCwgd2FybV94eSwgR1JBUEhfSywgREVWSUNFLAogICAgICAgICAgICAgICAgICAgZW5zX3N1Y2M9d2FybV9zdWNjLCBlbnNfY29zdD1ucC5vbmVzKDYpKQp3YXJtX3Byb2JsZW0gPSBXT1JLX1JPT1QgLyAid2FybXVwLnRzcCIKVC53cml0ZV90c3Aod2FybV9wcm9ibGVtLCB3YXJtX3h5KQp3YXJtX2RpciA9IFdPUktfUk9PVCAvICJ3YXJtdXAiIC8gdXVpZC51dWlkNCgpLmhleAp3YXJtX2Rpci5wYXJlbnQubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKd2FybSA9IHJ1bl9ndWlkZWRfYXJtKCJzaGFtIiwgUEFUQ0hFRF9MS0gsIHdhcm1fcHJvYmxlbSwgd2FybV94eSwgd2FybV9kaXIsCiAgICAgICAgICAgICAgICAgICAgICBTRUVEUywgMTUuMCwgTU9ERUwsIERFVklDRSwgVCwgTSwKICAgICAgICAgICAgICAgICAgICAgIGZyYWN0aW9uPVNQQVJTRV9GUkFDVElPTiwgZ3JhcGhfaz1HUkFQSF9LKQpkZWYgZWRnZV9zZXQodG91cik6CiAgICB0b3VyID0gbnAuYXNhcnJheSh0b3VyLCBkdHlwZT1ucC5pbnQzMikKICAgIHJldHVybiBzZXQoemlwKG5wLm1pbmltdW0odG91ciwgbnAucm9sbCh0b3VyLCAtMSkpLAogICAgICAgICAgICAgICAgICAgbnAubWF4aW11bSh0b3VyLCBucC5yb2xsKHRvdXIsIC0xKSkpKQpmb3Igc2VlZCBpbiBTRUVEUzoKICAgIG5hdGl2ZSA9IFQucnVuX2xraChzdHIoTkFUSVZFX0xLSCksIHN0cih3YXJtX3Byb2JsZW0pLAogICAgICAgICAgICAgICAgICAgICAgIHsiQ0FORElEQVRFX1NFVF9UWVBFIjogIlBPUE1VU0lDIiwgIk1BWF9DQU5ESURBVEVTIjogNiwKICAgICAgICAgICAgICAgICAgICAgICAgIlNVQkdSQURJRU5UIjogIk5PIiwgIlNFRUQiOiBzZWVkLCAiTUFYX1RSSUFMUyI6IDEsCiAgICAgICAgICAgICAgICAgICAgICAgICJUSU1FX0xJTUlUIjogMzAsICJTVE9QX0FUX09QVElNVU0iOiAiTk8ifSwKICAgICAgICAgICAgICAgICAgICAgICBzdHIoV09SS19ST09UIC8gIndhcm11cCIgLyBmIm5hdGl2ZV97c2VlZH1fe3V1aWQudXVpZDQoKS5oZXh9IiksIHRpbWVvdXQ9MzApCiAgICBmaXJzdCA9IG5wLmFzYXJyYXkocmVhZF90b3VyKHdhcm1fZGlyIC8gZiJzZWVkX3tzZWVkfSIgLyAiZmlyc3QudG91ciIpKSAtIDEKICAgIGFzc2VydCBlZGdlX3NldChuYXRpdmVbInRvdXIiXSkgPT0gZWRnZV9zZXQoZmlyc3QpLCBmIsSwbGsgdHVyIHV5dcWfbXV5b3I6IHtzZWVkfSIKcHJpbnQoIkdQVSwgUE9QNiB5YW1hc8SxIHZlIGlsayB0dXIgZcWfaXRsacSfaSBkb8SfcnVsYW5kxLEuIikKIiIiKQoKICAgIGluc3RhbmNlcyA9IG9sZCg2KQogICAgaW5zdGFuY2VzID0gcmVwbGFjZV9vbmNlKGluc3RhbmNlcywgJ2Fzc2VydCBsZW4oSU5TVEFOQ0VTKSA9PSA4JywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnYXNzZXJ0IGxlbihJTlNUQU5DRVMpID09IDQnKQogICAgc2V0X2NlbGwobm90ZWJvb2ssIDYsIGluc3RhbmNlcykKICAgIHNldF9jZWxsKG5vdGVib29rLCA3LCAoSEVSRSAvICJydW5fY2VsbC5weSIpLnJlYWRfdGV4dCgpKQogICAgc2V0X2NlbGwobm90ZWJvb2ssIDgsIChIRVJFIC8gImFuYWx5c2lzX2NlbGwucHkiKS5yZWFkX3RleHQoKSkKCiAgICBhcmNoaXZlID0gb2xkKDkpCiAgICBhcmNoaXZlID0gcmVwbGFjZV9vbmNlKGFyY2hpdmUsCiAgICAgICAgJ3sieHkubnB6IiwgIm5hdGl2ZS5jYW5kaWRhdGVzIiwgImxpdmUuY2FuZGlkYXRlcyIsICJwb3N0LmNhbmRpZGF0ZXMiLCAiZmlyc3QudG91ciIsICJiZXN0LnRvdXIiLCAic3Rkb3V0LnR4dCIsICJzdGRlcnIudHh0IiwgInBvbGljeV9hcnJheXMubnB6In0nLAogICAgICAgICd7Inh5Lm5weiIsICJuYXRpdmUuY2FuZGlkYXRlcyIsICJsaXZlLmNhbmRpZGF0ZXMiLCAicG9zdC5jYW5kaWRhdGVzIiwgImZpcnN0LnRvdXIiLCAiYmVzdC50b3VyIiwgImFjdGlvbi50eHQiLCAic3Rkb3V0LnR4dCIsICJzdGRlcnIudHh0IiwgInBvbGljeV9hcnJheXMubnB6In0nKQogICAgYXJjaGl2ZSA9IHJlcGxhY2Vfb25jZShhcmNoaXZlLCAnY29tcGxldGUgPSBsZW4ocm93cykgPT0gOCcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICdjb21wbGV0ZSA9IGxlbihyb3dzKSA9PSBsZW4oSU5TVEFOQ0VTKScpCiAgICBhcmNoaXZlID0gcmVwbGFjZV9vbmNlKGFyY2hpdmUsCiAgICAgICAgJ3RhbWFtbGFuYW4gw7ZybmVrIHtsZW4ocm93cyl9LzguJywKICAgICAgICAndGFtYW1sYW5hbiDDtnJuZWsge2xlbihyb3dzKX0ve2xlbihJTlNUQU5DRVMpfS4nKQogICAgc2V0X2NlbGwobm90ZWJvb2ssIDksIGFyY2hpdmUpCgogICAgbm90ZWJvb2tbIm1ldGFkYXRhIl1bImNvbGFiIl1bIm5hbWUiXSA9IE9VVFBVVC5uYW1lCiAgICByZXR1cm4gbm90ZWJvb2sKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBub3RlYm9vayA9IG1ha2Vfbm90ZWJvb2soKQogICAgT1VUUFVULnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBPVVRQVVQud3JpdGVfdGV4dChqc29uLmR1bXBzKG5vdGVib29rLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0xKSArICJcbiIpCiAgICBwcmludChPVVRQVVQpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='}
CODE_ROOT = WORK_ROOT / "code"
for rel, encoded in BUNDLED_FILES.items():
    content = base64.b64decode(encoded)
    for dest in (CODE_ROOT / rel, OUTPUT_ROOT / "code" / rel):
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists():
            assert dest.read_bytes() == content, f"Deney kodu değişmiş: {rel}"
        else:
            dest.write_bytes(content)
CODE_SHA256 = {rel: hashlib.sha256(base64.b64decode(encoded)).hexdigest()
               for rel, encoded in BUNDLED_FILES.items()}
sys.path[:0] = [str(CODE_ROOT / "livehook"), str(CODE_ROOT / "v11_candidate")]
import build_search_hook
from coordinator import read_tour
from pilot import run_guided_arm, run_native_arm, sha256_file
print("Dondurulmuş deney kodu:", CODE_SHA256)


Dondurulmuş deney kodu: {'livehook/coordinator.py': 'd6064841d1ab6d7a0ae00ddd33d142934349df11c75ce76741466df9c9d43a56', 'v11_candidate/BetaTourSearchHook.h': 'ce8f759ac6a9e5f8405953b013a3cdad24f9000db62c7e0273845b82336cf935', 'v11_candidate/BetaTourVisit.h': '6c133c2875a2b71a81de93a3ee4e00301898fe0db7022a464577e15de5210b51', 'v11_candidate/build_search_hook.py': '4059268950519b176194c12b3a679837b48d62122d2f20d54b909a8b8a39d6a2', 'v11_candidate/policy.py': '9ced44dae9f1a810c0c54d8dc521fa420925d4b9c37e75bab1d2a0854bcb28ce', 'v11_candidate/pilot.py': 'd4ee28ce79a7db0044ceb681f04a73643fcd1d548ea5b6e9906ce663e83740ab', 'v11_candidate/run_cell.py': 'bf3c56842cb605f3f8a6889e4693b570c91ab5c0a87ed9dfb3764721ad524524', 'v11_candidate/analysis_cell.py': '72400a1b557fd503f82e88a35ea2b97e7d2fde10f85f8bee1e1103763b294030', 'v11_candidate/build_colab_notebook.py': '5efe0f4e5336de557f5f30b676119226056e294c39cd6b95fbef9ecc62037100'}


In [ ]:
LKH_ROOT = WORK_ROOT / "lkh"
LKH_ROOT.mkdir(exist_ok=True)
SOURCE_DIR = LKH_ROOT / "LKH-3.0.14"
SOURCE_INFO_PATH = LKH_ROOT / "source_info.json"
if not SOURCE_DIR.exists():
    sources = [
        ("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("pinned_mirror", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar", LKH_TAR_SHA256, 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout_s in sources:
        archive = LKH_ROOT / ("LKH-3.0.14.tgz" if url.endswith(".tgz") else "LKH-3.0.14.tar")
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout_s) as response, archive.open("wb") as out:
                shutil.copyfileobj(response, out)
            digest = hashlib.sha256(archive.read_bytes()).hexdigest()
            if expected_sha:
                assert digest == expected_sha, f"LKH arşiv SHA256 uyuşmuyor: {digest}"
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(LKH_ROOT, filter="data")
                except TypeError:
                    tar.extractall(LKH_ROOT)
            assert SOURCE_DIR.exists()
            actual_source_sha = hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest()
            assert actual_source_sha == LKH_FINDTOUR_SHA256, f"LKH 3.0.14 kaynağı beklenenden farklı: {actual_source_sha}"
            SOURCE_INFO_PATH.write_text(json.dumps({"label": label, "url": url, "archive_sha256": digest}, indent=2))
            break
        except Exception as exc:
            last_error = exc
            if SOURCE_DIR.exists():
                shutil.rmtree(SOURCE_DIR)
    else:
        raise RuntimeError(f"LKH indirilemedi: {last_error}")
assert hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest() == LKH_FINDTOUR_SHA256
for filename, expected in build_search_hook.UPSTREAM_SHA256.items():
    assert hashlib.sha256((SOURCE_DIR / "SRC" / filename).read_bytes()).hexdigest() == expected, filename
if not SOURCE_INFO_PATH.exists():
    cached = next((p for p in (LKH_ROOT / "LKH-3.0.14.tar", LKH_ROOT / "LKH-3.0.14.tgz") if p.exists()), None)
    assert cached is not None, "LKH kaynak kayıt dosyası eksik; kaynak arşiv bulunamadı"
    SOURCE_INFO_PATH.write_text(json.dumps({"label": "recovered_cache", "url": "local_cache",
                                            "archive_sha256": hashlib.sha256(cached.read_bytes()).hexdigest()}, indent=2))
NATIVE_LKH = SOURCE_DIR / "LKH"
if not NATIVE_LKH.exists():
    with (LKH_ROOT / "native_build.log").open("w") as build_log:
        subprocess.run(["make", "-j", str(min(6, CPU_COUNT))], cwd=SOURCE_DIR, check=True,
                       stdout=build_log, stderr=subprocess.STDOUT)
PATCHED_DIR = LKH_ROOT / "LKH-3.0.14-BetaTour-v11"
PATCHED_LKH = PATCHED_DIR / "LKH"
if not PATCHED_LKH.exists():
    if PATCHED_DIR.exists():
        shutil.rmtree(PATCHED_DIR)
    build_search_hook.build(SOURCE_DIR, PATCHED_DIR, jobs=min(6, CPU_COUNT))
patch_manifest = json.loads((PATCHED_DIR / "betatour_build.json").read_text())
assert patch_manifest["upstream_sha256"] == build_search_hook.UPSTREAM_SHA256
assert patch_manifest["search_hook_sha256"] == CODE_SHA256["v11_candidate/BetaTourSearchHook.h"]
assert patch_manifest["visit_header_sha256"] == CODE_SHA256["v11_candidate/BetaTourVisit.h"]
assert patch_manifest["binary_sha256"] == sha256_file(PATCHED_LKH)
BUILD_INFO = {"native_sha256": sha256_file(NATIVE_LKH),
              "patched_sha256": sha256_file(PATCHED_LKH),
              "upstream_findtour_sha256": LKH_FINDTOUR_SHA256}
origin_path = OUTPUT_ROOT / "code" / "lkh_source_info.json"
if not origin_path.exists():
    origin_path.write_text(SOURCE_INFO_PATH.read_text())
print("LKH 3.0.14 derlendi:", BUILD_INFO)


LKH 3.0.14 derlendi: {'native_sha256': '66ecaefba9beb21be588d2e6622691416134898ca07a3cc050ab60554b894b81', 'patched_sha256': '1a12de5180770ea2d6aa7962cd9370a09241f1ce1595355c8f8b5436a50f5cd0', 'upstream_findtour_sha256': '567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee'}


In [ ]:
def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

MANIFEST = {"schema": 1, "profile": "four_reused_10k_development_instances",
            "experiment_name": EXPERIMENT_NAME, "arms": ARMS, "seeds": SEEDS,
            "instance_specs": INSTANCE_SPECS, "xy_hashes": EXPECTED_XY_SHA256,
            "time_to_target_costs_from_v9_native_pop6": TARGET_COSTS,
            "sparse_fraction": SPARSE_FRACTION, "graph_k": GRAPH_K,
            "repo_commit": REPO_COMMIT, "checkpoint_sha256": CHECKPOINT_SHA256,
            "hardware": HARDWARE, "code_sha256": CODE_SHA256, "build": BUILD_INFO,
            "analysis_plan": "Mechanism pilot on reused cases; no paper performance claim before frozen independent holdout."}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in MANIFEST:
        if key != "hardware":
            assert saved[key] == MANIFEST[key], f"Drive'daki pilotun {key} ayarı değişti; yeni deney adı kullanın."
    for key in ("gpu", "vcpus"):
        assert saved["hardware"][key] == HARDWARE[key], f"Donanım {key} değişti; eşli pilotu aynı türde çalıştırın."
else:
    write_json_atomic(manifest_path, MANIFEST)

# GPU warmup and exact first-trial/native equivalence check.
warm_xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
warm_succ = np.stack([np.roll(np.arange(100), -1)] * 6, axis=1).astype(np.int32)
M.predict_tourfold(MODEL, warm_xy, GRAPH_K, DEVICE,
                   ens_succ=warm_succ, ens_cost=np.ones(6))
warm_problem = WORK_ROOT / "warmup.tsp"
T.write_tsp(warm_problem, warm_xy)
warm_dir = WORK_ROOT / "warmup" / uuid.uuid4().hex
warm_dir.parent.mkdir(exist_ok=True)
warm = run_guided_arm("sham", PATCHED_LKH, warm_problem, warm_xy, warm_dir,
                      SEEDS, 15.0, MODEL, DEVICE, T, M,
                      fraction=SPARSE_FRACTION, graph_k=GRAPH_K)
def edge_set(tour):
    tour = np.asarray(tour, dtype=np.int32)
    return set(zip(np.minimum(tour, np.roll(tour, -1)),
                   np.maximum(tour, np.roll(tour, -1))))
for seed in SEEDS:
    native = T.run_lkh(str(NATIVE_LKH), str(warm_problem),
                       {"CANDIDATE_SET_TYPE": "POPMUSIC", "MAX_CANDIDATES": 6,
                        "SUBGRADIENT": "NO", "SEED": seed, "MAX_TRIALS": 1,
                        "TIME_LIMIT": 30, "STOP_AT_OPTIMUM": "NO"},
                       str(WORK_ROOT / "warmup" / f"native_{seed}_{uuid.uuid4().hex}"), timeout=30)
    first = np.asarray(read_tour(warm_dir / f"seed_{seed}" / "first.tour")) - 1
    assert edge_set(native["tour"]) == edge_set(first), f"İlk tur uyuşmuyor: {seed}"
print("GPU, POP6 yaması ve ilk tur eşitliği doğrulandı.")


GPU, POP6 yaması ve ilk tur eşitliği doğrulandı.


In [ ]:
INSTANCES = []
for group, n, count, distribution, generator_seed, budget_s in INSTANCE_SPECS:
    rng = np.random.default_rng(generator_seed)
    for index in range(count):
        xy = T.to_int_coords(T.gen_instance(n, distribution, rng))
        name = f"augment_{group}_{index:03d}"
        digest = hashlib.sha256(np.ascontiguousarray(xy).tobytes()).hexdigest()
        assert digest == EXPECTED_XY_SHA256[name], f"Eski v8 koordinatları uyuşmuyor: {name}"
        INSTANCES.append({"name": name, "group": group, "n": n,
                          "budget_s": budget_s, "xy": xy, "xy_sha256": digest})
INSTANCES.sort(key=lambda item: (item["budget_s"], item["name"]))
assert len(INSTANCES) == 4
def arm_order(name):
    return sorted(ARMS, key=lambda arm: hashlib.sha256((name + ":" + arm).encode()).hexdigest())
print("Pilot:", len(INSTANCES), "örnek,", len(ARMS), "kol; kesintisiz tek kol bütçesi",
      sum(item["budget_s"] for item in INSTANCES) / 3600, "saat")


Pilot: 4 örnek, 6 kol; kesintisiz tek kol bütçesi 0.13333333333333333 saat


In [ ]:
"""v11 Colab execution cell. Completed arms are immutable and resumable."""


def arm_record_path(inst, arm):
    return OUTPUT_ROOT / "instances" / inst["name"] / arm / "result.json"


def validate_saved(inst, arm, record):
    assert record["complete"] and record["arm"] == arm and record["instance"] == inst["name"]
    assert record["xy_sha256"] == inst["xy_sha256"] and record["seeds"] == SEEDS
    assert record["budget_s"] == inst["budget_s"] and record["build"] == BUILD_INFO
    assert record["code_sha256"] == CODE_SHA256
    assert record["fraction"] == SPARSE_FRACTION and record["graph_k"] == GRAPH_K
    assert len(record["seed_costs"]) == len(SEEDS)
    assert min(record["seed_costs"]) == record["best_cost"]
    if arm not in ("native_pop6", "native_pop7"):
        assert len(record["first_tour_hashes"]) == len(SEEDS)
        assert len(record["live_candidate_hashes"]) == len(SEEDS)
        assert len(record["native_candidate_hashes"]) == len(SEEDS)
        audit = record["policy_audit"]
        assert len(audit["gate_sha256_per_seed"]) == len(SEEDS)
        assert audit["seed_ids"] == SEEDS
        if arm == "sham":
            assert audit["selected_actions_per_seed"] == [0] * len(SEEDS)
        else:
            assert audit["selected_actions_per_seed"] == audit["guided_actions_per_seed"]
        assert audit["selected_unique_undirected_edges_per_seed"] == audit["selected_actions_per_seed"]
    return record


def save_progress(state="running", last=None):
    completed = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      {"state": state, "completed_arms": completed,
                       "total_arms": len(INSTANCES) * len(ARMS), "last": last,
                       "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})


new_arms = 0
save_progress()
for inst in INSTANCES:
    instance_dir = OUTPUT_ROOT / "instances" / inst["name"]
    instance_dir.mkdir(parents=True, exist_ok=True)
    coord_path = instance_dir / "xy.npz"
    if not coord_path.exists():
        temp = instance_dir / "xy.tmp.npz"
        np.savez_compressed(temp, xy=inst["xy"])
        os.replace(temp, coord_path)
    with np.load(coord_path, allow_pickle=False) as saved_xy:
        assert hashlib.sha256(np.ascontiguousarray(saved_xy["xy"]).tobytes()).hexdigest() == inst["xy_sha256"]
    local_problem = WORK_ROOT / "problems" / (inst["name"] + ".tsp")
    local_problem.parent.mkdir(exist_ok=True)
    T.write_tsp(local_problem, inst["xy"])
    for arm in arm_order(inst["name"]):
        record_path = arm_record_path(inst, arm)
        if record_path.exists():
            validate_saved(inst, arm, json.loads(record_path.read_text()))
            print("ATLA:", inst["name"], arm, flush=True)
            continue
        if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
            break
        attempt_id = uuid.uuid4().hex
        local_run = WORK_ROOT / "runs" / inst["name"] / arm / attempt_id
        local_run.parent.mkdir(parents=True, exist_ok=True)
        try:
            print("BAŞLA:", inst["name"], arm, "bütçe", inst["budget_s"], "sn", flush=True)
            if arm.startswith("native_"):
                raw = run_native_arm(arm, NATIVE_LKH, local_problem, inst["xy"],
                                     local_run, SEEDS, inst["budget_s"], T)
            else:
                raw = run_guided_arm(arm, PATCHED_LKH, local_problem, inst["xy"],
                                     local_run, SEEDS, inst["budget_s"],
                                     MODEL, DEVICE, T, M,
                                     fraction=SPARSE_FRACTION, graph_k=GRAPH_K)
            audit_rel = f"instances/{inst['name']}/{arm}/{attempt_id}"
            audit_dest = OUTPUT_ROOT / audit_rel
            shutil.copytree(local_run, audit_dest)
            record = {"schema": 1, "complete": True, "instance": inst["name"],
                      "group": inst["group"], "n": inst["n"],
                      "xy_sha256": inst["xy_sha256"], "arm": arm,
                      "seeds": SEEDS, "budget_s": inst["budget_s"],
                      "seed_costs": raw["seed_costs"], "best_cost": raw["best_cost"],
                      "wall_elapsed_s": raw["wall_elapsed_s"],
                      "start_epoch_s": raw["start_epoch_s"],
                      "deadline_epoch_s": raw["deadline_epoch_s"] if "deadline_epoch_s" in raw else raw["start_epoch_s"] + inst["budget_s"],
                      "first_costs": raw.get("first_costs"),
                      "first_tour_hashes": raw.get("first_tour_hashes"),
                      "live_candidate_hashes": raw.get("live_candidate_hashes"),
                      "native_candidate_hashes": raw.get("native_candidate_hashes"),
                      "policy_audit": raw.get("policy_audit"),
                      "guided_improvement_events": raw.get("guided_improvement_events"),
                      "native_improvement_events": raw.get("native_improvement_events"),
                      "fraction": SPARSE_FRACTION, "graph_k": GRAPH_K,
                      "build": BUILD_INFO, "code_sha256": CODE_SHA256,
                      "hardware": HARDWARE, "audit_relpath": audit_rel,
                      "attempt_id": attempt_id,
                      "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            validate_saved(inst, arm, record)
            write_json_atomic(record_path, record)
            new_arms += 1
            save_progress(last=inst["name"] + "/" + arm)
            print("BİTTİ:", inst["name"], arm, "en iyi", raw["best_cost"], flush=True)
        except Exception as exc:
            write_json_atomic(OUTPUT_ROOT / "errors" /
                              (inst["name"] + "_" + arm + "_" + attempt_id + ".json"),
                              {"instance": inst["name"], "arm": arm,
                               "error": repr(exc),
                               "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
            save_progress(state="error", last=inst["name"] + "/" + arm)
            raise
    if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
        break

completed_count = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
save_progress(state="complete" if completed_count == len(INSTANCES) * len(ARMS) else "partial")
print(f"Tamamlanan kol: {completed_count}/{len(INSTANCES) * len(ARMS)}. "
      f"Bu oturumdaki yeni kol: {new_arms}")


BAŞLA: augment_clu10k_000 random_add bütçe 120.0 sn
BİTTİ: augment_clu10k_000 random_add en iyi 33779214
BAŞLA: augment_clu10k_000 native_pop6 bütçe 120.0 sn
BİTTİ: augment_clu10k_000 native_pop6 en iyi 33775546
BAŞLA: augment_clu10k_000 frequency_add bütçe 120.0 sn
BİTTİ: augment_clu10k_000 frequency_add en iyi 33785299
BAŞLA: augment_clu10k_000 sham bütçe 120.0 sn
BİTTİ: augment_clu10k_000 sham en iyi 33779532
BAŞLA: augment_clu10k_000 native_pop7 bütçe 120.0 sn
BİTTİ: augment_clu10k_000 native_pop7 en iyi 33796478
BAŞLA: augment_clu10k_000 beta_add bütçe 120.0 sn
BİTTİ: augment_clu10k_000 beta_add en iyi 33786929
BAŞLA: augment_clu10k_001 native_pop6 bütçe 120.0 sn
BİTTİ: augment_clu10k_001 native_pop6 en iyi 16213172
BAŞLA: augment_clu10k_001 random_add bütçe 120.0 sn
BİTTİ: augment_clu10k_001 random_add en iyi 16220736
BAŞLA: augment_clu10k_001 native_pop7 bütçe 120.0 sn
BİTTİ: augment_clu10k_001 native_pop7 en iyi 16214513
BAŞLA: augment_clu10k_001 beta_add bütçe 120.0 sn
BİTTİ: 

In [ ]:
"""v11 Colab summary cell: mechanism measurements and guarded comparisons."""


def read_key_value_metrics(path):
    values = {}
    for line in Path(path).read_text().splitlines():
        words = line.split()
        if len(words) != 2 or words[0] in values:
            raise ValueError(f"Bozuk telemetri satırı: {path}: {line!r}")
        values[words[0]] = float(words[1]) if "." in words[1] else int(words[1])
    return values


def comparison_summary(frame, treatment, control, basis):
    if frame.empty:
        return None
    rel = 100.0 * (frame[treatment].to_numpy(dtype=np.float64) -
                   frame[control].to_numpy(dtype=np.float64)) / frame[control].to_numpy(dtype=np.float64)
    return {"treatment": treatment, "control": control, "basis": basis,
            "n_instances": len(rel), "wins": int((rel < 0).sum()),
            "losses": int((rel > 0).sum()), "ties": int((rel == 0).sum()),
            "mean_relative_pct": float(rel.mean()),
            "median_relative_pct": float(np.median(rel))}


patched_arms = ["sham", "random_add", "frequency_add", "beta_add"]
guided_arms = ["random_add", "frequency_add", "beta_add"]
rows, mechanism_rows, target_rows, frequency_rows = [], [], [], []
summary_dir = OUTPUT_ROOT / "summary"
summary_dir.mkdir(exist_ok=True)

for inst in INSTANCES:
    records = {}
    for arm in ARMS:
        path = arm_record_path(inst, arm)
        if path.exists():
            records[arm] = validate_saved(inst, arm, json.loads(path.read_text()))
    if len(records) != len(ARMS):
        continue

    row = {"instance": inst["name"], "group": inst["group"],
           "n": inst["n"], "budget_s": inst["budget_s"],
           "mixed_hardware": len({json.dumps(records[arm]["hardware"], sort_keys=True)
                                   for arm in ARMS}) > 1}
    for arm in ARMS:
        row[arm] = records[arm]["best_cost"]
    row["first_tours_match"] = all(
        len({records[arm]["first_tour_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["raw_first_tours_match"] = all(
        len({records[arm]["policy_audit"]["first_tour_raw_sha256"][i]
             for arm in patched_arms}) == 1 for i in range(len(SEEDS)))
    row["native_candidates_match"] = all(
        len({records[arm]["native_candidate_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["live_candidates_match"] = all(
        len({records[arm]["live_candidate_hashes"][i] for arm in patched_arms}) == 1
        for i in range(len(SEEDS)))
    row["gates_match"] = all(
        len({records[arm]["policy_audit"]["gate_sha256_per_seed"][i]
             for arm in patched_arms}) == 1 for i in range(len(SEEDS)))
    row["model_predictions_match"] = len({
        records[arm]["policy_audit"]["model_probability_sha256"]
        for arm in patched_arms}) == 1
    row["planned_actions_match"] = len({
        json.dumps(records[arm]["policy_audit"]["all_guided_action_sha256_per_seed"],
                   sort_keys=True) for arm in patched_arms}) == 1
    row["capacities_match"] = all(
        len({records[arm]["policy_audit"]["selected_actions_per_seed"][i]
             for arm in guided_arms}) == 1 for i in range(len(SEEDS)))
    row["undirected_capacities_match"] = all(
        len({records[arm]["policy_audit"]["selected_unique_undirected_edges_per_seed"][i]
             for arm in guided_arms}) == 1 for i in range(len(SEEDS)))
    row["matched_policy_state"] = all(row[key] for key in
                                      ("first_tours_match", "raw_first_tours_match",
                                       "native_candidates_match", "live_candidates_match",
                                       "model_predictions_match", "planned_actions_match",
                                       "gates_match", "capacities_match",
                                       "undirected_capacities_match"))
    rows.append(row)

    for arm in patched_arms:
        record = records[arm]
        for index, seed in enumerate(SEEDS):
            base = OUTPUT_ROOT / record["audit_relpath"] / f"seed_{seed}"
            metrics = read_key_value_metrics(base / "telemetry.txt")
            selected = int(record["policy_audit"]["selected_actions_per_seed"][index])
            if int(metrics["action_count"]) != selected:
                raise AssertionError(f"Seçilen/uygulanan kenar sayısı farklı: {inst['name']}/{arm}/{seed}")
            mechanism_rows.append({"instance": inst["name"], "arm": arm,
                                   "seed": seed, "selected_actions": selected,
                                   "telemetry_age_at_end_s": max(0.0, min(
                                       float(record["deadline_epoch_s"]),
                                       float(record["start_epoch_s"]) + float(record["wall_elapsed_s"]))
                                       - float(metrics["telemetry_snapshot_epoch_s"])),
                                   **metrics})

    frequency_audit = records["beta_add"]["policy_audit"]
    for arm in guided_arms:
        for index, seed in enumerate(SEEDS):
            values = frequency_audit["all_guided_selected_ensemble_frequency_per_seed"][arm][index]
            frequency_rows.append({"instance": inst["name"], "arm": arm,
                                   "seed": seed, "selected_actions": len(values),
                                   "zero_frequency_actions": sum(value == 0 for value in values),
                                   "mean_ensemble_frequency": float(np.mean(values)) if values else np.nan})

    target = TARGET_COSTS[inst["name"]]
    for arm in ARMS:
        record = records[arm]
        event_map = (record["native_improvement_events"] if arm.startswith("native_")
                     else record["guided_improvement_events"])
        observed = [float(event["elapsed_s"])
                    for seed in SEEDS for event in event_map[str(seed)]
                    if int(event["cost"]) <= target]
        target_rows.append({"instance": inst["name"], "arm": arm,
                            "target_cost_from_v9_native_pop6": target,
                            "reached": bool(observed),
                            "earliest_elapsed_s": min(observed) if observed else np.nan,
                            "trace_kind": "approximate_native_file_poll" if arm.startswith("native_")
                            else "exact_patched_best_event"})

table = pd.DataFrame(rows)
mechanism_table = pd.DataFrame(mechanism_rows)
target_table = pd.DataFrame(target_rows)
frequency_table = pd.DataFrame(frequency_rows)
table.to_csv(summary_dir / "complete_instances.csv", index=False)
mechanism_table.to_csv(summary_dir / "candidate_mechanism.csv", index=False)
target_table.to_csv(summary_dir / "time_to_target_exploratory.csv", index=False)
frequency_table.to_csv(summary_dir / "selected_edge_frequencies.csv", index=False)

same_hardware = table.loc[~table["mixed_hardware"]].copy() if not table.empty else table
matched = same_hardware.loc[same_hardware["matched_policy_state"]].copy() if not same_hardware.empty else same_hardware
endpoint_rows = []
for control in ("native_pop6", "native_pop7", "sham", "random_add", "frequency_add"):
    result = comparison_summary(same_hardware, "beta_add", control,
                                "descriptive same-instance endpoint")
    if result is not None:
        endpoint_rows.append(result)
matched_rows = []
for control in ("sham", "random_add", "frequency_add"):
    result = comparison_summary(matched, "beta_add", control,
                                "all six raw first tours/candidate states/predictions/actions/gates/capacities match")
    if result is not None:
        matched_rows.append(result)
comparison_columns = ["treatment", "control", "basis", "n_instances", "wins",
                      "losses", "ties", "mean_relative_pct", "median_relative_pct"]
endpoint_table = pd.DataFrame(endpoint_rows, columns=comparison_columns)
matched_table = pd.DataFrame(matched_rows, columns=comparison_columns)
endpoint_table.to_csv(summary_dir / "endpoint_comparisons.csv", index=False)
matched_table.to_csv(summary_dir / "matched_policy_comparisons.csv", index=False)

if mechanism_table.empty:
    mechanism_aggregate = pd.DataFrame()
else:
    mechanism_aggregate = mechanism_table.groupby("arm", as_index=False)[
        ["selected_actions", "action_candidate_enumerations",
         "distinct_actions_enumerated", "action_accepted_edge_entries",
         "distinct_actions_entering_accepted_tours",
         "top_level_action_first_gate_passes", "top_level_action_positive_g1",
         "action_incumbent_entries", "distinct_actions_ever_in_better_tour"]].sum()
mechanism_aggregate.to_csv(summary_dir / "candidate_mechanism_totals.csv", index=False)

write_json_atomic(summary_dir / "pilot_note.json", {
    "analysis_revision": "v11-state-guarded-mechanism-1",
    "completed_instances": len(table), "planned_instances": len(INSTANCES),
    "same_hardware_instances": len(same_hardware),
    "matched_policy_instances": len(matched),
    "interpretation": "Reused-case mechanism pilot. Endpoint comparisons against native POP6/7 are descriptive. Patched policy comparisons require exact six-seed state, gate and capacity matching. Native time-to-target traces are approximate file polls; patched best.events are exact. No manuscript superiority claim."
})

print("Tamamlanan örnek:", len(table), "/", len(INSTANCES),
      "; tam eşleşmiş politika örneği:", len(matched))
display(table)
display(endpoint_table)
display(matched_table)
display(mechanism_aggregate)
display(frequency_table)
display(target_table)


Tamamlanan örnek: 4 / 4 ; tam eşleşmiş politika örneği: 4


,instance,group,n,budget_s,mixed_hardware,native_pop6,native_pop7,sham,random_add,frequency_add,...,first_tours_match,raw_first_tours_match,native_candidates_match,live_candidates_match,gates_match,model_predictions_match,planned_actions_match,capacities_match,undirected_capacities_match,matched_policy_state
0,augment_clu10k_000,clu10k,10000,120.0,False,33775546,33796478,33779532,33779214,33785299,...,True,True,True,True,True,True,True,True,True,True
1,augment_clu10k_001,clu10k,10000,120.0,False,16213172,16214513,16216039,16220736,16214480,...,True,True,True,True,True,True,True,True,True,True
2,augment_uni10k_000,uni10k,10000,120.0,False,72131411,72134433,72131411,72134094,72128876,...,True,True,True,True,True,True,True,True,True,True
3,augment_uni10k_001,uni10k,10000,120.0,False,71668676,71688449,71668676,71690672,71688415,...,True,True,True,True,True,True,True,True,True,True


,treatment,control,basis,n_instances,wins,losses,ties,mean_relative_pct,median_relative_pct
0,beta_add,native_pop6,descriptive same-instance endpoint,4,0,4,0,0.032523,0.032997
1,beta_add,native_pop7,descriptive same-instance endpoint,4,1,3,0,0.007021,0.014418
2,beta_add,sham,descriptive same-instance endpoint,4,0,4,0,0.025151,0.025111
3,beta_add,random_add,descriptive same-instance endpoint,4,1,3,0,0.009543,0.012220
4,beta_add,frequency_add,descriptive same-instance endpoint,4,0,4,0,0.017280,0.016264


,treatment,control,basis,n_instances,wins,losses,ties,mean_relative_pct,median_relative_pct
0,beta_add,sham,all six raw first tours/candidate states/predi...,4,0,4,0,0.025151,0.025111
1,beta_add,random_add,all six raw first tours/candidate states/predi...,4,1,3,0,0.009543,0.012220
2,beta_add,frequency_add,all six raw first tours/candidate states/predi...,4,0,4,0,0.017280,0.016264


,arm,selected_actions,action_candidate_enumerations,distinct_actions_enumerated,action_accepted_edge_entries,distinct_actions_entering_accepted_tours,top_level_action_first_gate_passes,top_level_action_positive_g1,action_incumbent_entries,distinct_actions_ever_in_better_tour
0,beta_add,60000,242788749,60000,96418,37693,793631,793631,340,324
1,frequency_add,60000,241999683,60000,82114,34722,1018620,1018620,268,261
2,random_add,60000,242042801,60000,51448,26899,630294,630294,83,82
3,sham,0,0,0,0,0,0,0,0,0


,instance,arm,seed,selected_actions,zero_frequency_actions,mean_ensemble_frequency
0,augment_clu10k_000,random_add,301,2500,2494,0.0036
1,augment_clu10k_000,random_add,302,2500,2493,0.0048
2,augment_clu10k_000,random_add,303,2500,2494,0.0036
3,augment_clu10k_000,random_add,304,2500,2495,0.0024
4,augment_clu10k_000,random_add,305,2500,2492,0.0048
...,...,...,...,...,...,...
67,augment_uni10k_001,beta_add,302,2500,2469,0.0180
68,augment_uni10k_001,beta_add,303,2500,2457,0.0300
69,augment_uni10k_001,beta_add,304,2500,2454,0.0264
70,augment_uni10k_001,beta_add,305,2500,2459,0.0264


,instance,arm,target_cost_from_v9_native_pop6,reached,earliest_elapsed_s,trace_kind
0,augment_clu10k_000,native_pop6,33760199,False,NaN,approximate_native_file_poll
1,augment_clu10k_000,native_pop7,33760199,False,NaN,approximate_native_file_poll
2,augment_clu10k_000,sham,33760199,False,NaN,exact_patched_best_event
3,augment_clu10k_000,random_add,33760199,False,NaN,exact_patched_best_event
4,augment_clu10k_000,frequency_add,33760199,False,NaN,exact_patched_best_event
5,augment_clu10k_000,beta_add,33760199,False,NaN,exact_patched_best_event
6,augment_clu10k_001,native_pop6,16201186,False,NaN,approximate_native_file_poll
7,augment_clu10k_001,native_pop7,16201186,False,NaN,approximate_native_file_poll
8,augment_clu10k_001,sham,16201186,False,NaN,exact_patched_best_event
9,augment_clu10k_001,random_add,16201186,False,NaN,exact_patched_best_event


In [ ]:
def checked_zip(name, include_audit):
    local = WORK_ROOT / name
    temp = WORK_ROOT / (name + ".tmp")
    files = []
    with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            if not path.is_file() or path.suffix == ".zip" or path.name.endswith(".tmp") or path.name.endswith(".tmp.npz"):
                continue
            rel = path.relative_to(OUTPUT_ROOT)
            if not include_audit and ("policy_audit" in rel.parts or path.name in
                                      {"xy.npz", "native.candidates", "live.candidates", "post.candidates", "first.tour", "best.tour", "action.txt", "stdout.txt", "stderr.txt", "policy_arrays.npz"}):
                continue
            archive.write(path, str(rel), compress_type=zipfile.ZIP_STORED if path.suffix == ".npz" else zipfile.ZIP_DEFLATED)
            files.append(str(rel))
    with zipfile.ZipFile(temp) as check:
        assert check.testzip() is None and len(check.namelist()) == len(files)
    os.replace(temp, local)
    target = OUTPUT_ROOT / "archives" / name
    tmp_target = target.with_name(target.name + ".tmp")
    shutil.copy2(local, tmp_target)
    os.replace(tmp_target, target)
    assert sha256_file(local) == sha256_file(target)
    return {"local": str(local), "drive": str(target), "sha256": sha256_file(local),
            "bytes": local.stat().st_size, "files": len(files)}

complete = len(rows) == len(INSTANCES)
audit_zip = checked_zip(EXPERIMENT_NAME + "_audit.zip", True) if complete else None
results_zip = checked_zip(EXPERIMENT_NAME + "_results.zip", False)
write_json_atomic(OUTPUT_ROOT / "archives" / "archive_info.json",
                  {"complete": complete, "results": results_zip, "audit": audit_zip})
print("Sonuç ZIP'i:", results_zip)
if complete:
    print("Tam denetim ZIP'i Drive'da:", audit_zip)
    from google.colab import files
    files.download(results_zip["local"])
else:
    print(f"Ara sonuçlar Drive'da; tamamlanan örnek {len(rows)}/{len(INSTANCES)}. Yeniden çalıştırınca devam eder.")


Sonuç ZIP'i: {'local': '/content/betatour_v11_candidate_work/v11_pop6_candidate_mechanism_results.zip', 'drive': '/content/drive/MyDrive/BetaTour/v11_pop6_candidate_mechanism/archives/v11_pop6_candidate_mechanism_results.zip', 'sha256': 'cd61292532cb59b62408876ee37281700787ba439be01c228b0859509bdb11ca', 'bytes': 17657885, 'files': 629}
Tam denetim ZIP'i Drive'da: {'local': '/content/betatour_v11_candidate_work/v11_pop6_candidate_mechanism_audit.zip', 'drive': '/content/drive/MyDrive/BetaTour/v11_pop6_candidate_mechanism/archives/v11_pop6_candidate_mechanism_audit.zip', 'sha256': 'da61212308331a655a4c3391c94449bf7e02a1a8e8838b5b516a459dcd49ea95', 'bytes': 102741223, 'files': 1545}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>